# RelCheck — Full 600-Image Run (Together.ai)

## What this notebook computes
- **B1**: BLIP-2 original captions (no correction)
- **B2**: Self-refinement baseline
- **B3**: Blind LLM correction baseline
- **KB ablations**: Objects only, Objects + geometry, Full KB dump
- **RelCheck v2**: Full enrichment (structured analysis + verification)
- **Multi-model**: LLaVA-1.5, Qwen2.5-VL-8B original + RelCheck-corrected

## Cell Order
| Cell | What | Output |
|------|------|--------|
| 0 | Setup + constants + helpers | — |
| 1 | Load Models (GDINO + BLIP-2) | GPU models |
| 2 | Load Images + R-Bench data | Image dict |
| 3 | BLIP-2 captioning | captions.json |
| 3b | Multi-model captioning (LLaVA-1.5 + Qwen2.5-VL-8B) | llava/scout captions |
| 4 | KB construction (GDINO + Maverick) | kb_results.json |
| 5 | Full RelCheck enrichment (BLIP-2) | enriched.json |
| 5b | Multi-model enrichment (LLaVA + Qwen2.5-VL-8B) | llava/scout enriched |
| 6 | Ablation correctors | ablation_captions.json |
| 7 | R-POPE LLM-Judge (all methods + multi-model) | Tables 1, 3, 4, 7, 8, 10 |
| 8 | CLIPScore | Table 5 |
| 9 | Pipeline stats + CSVs | Table 9 |
| 10 | Figures | Figures 4-6 |
| 11 | Qualitative examples | Figure 2 |
| 12 | R-CHAIR | Table 6 |
| 13 | InstructBLIP comparison | Table 11 |
| 14 | Architecture diagram | Figure 1 |


In [ ]:
# ============================================================
# Cell 0 — Install + Setup
# ============================================================
!pip install -q together gdown transformers pillow torch torchvision accelerate bitsandbytes>=0.46.1
!pip install -q spacy open-clip-torch
!python -m spacy download en_core_web_sm -q

import os
from google.colab import drive
drive.mount("/content/drive")

import json, base64, re, time, random, csv, math
from io import BytesIO
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import torch
from together import Together
from transformers import (
    AutoProcessor, AutoModelForZeroShotObjectDetection,
    VitPoseForPoseEstimation,
)
import spacy

# ── Settings ──────────────────────────────────────────────
TOGETHER_API_KEY = ""   # <-- paste your Together.ai key
N_IMAGES         = 20   # set to 600 for full run, None for all available
RANDOM_SEED      = 42

# Drive paths
DRIVE_IMAGES_DIR = "/content/drive/MyDrive/RelCheck_Data/images"
RBENCH_PATH      = "/content/drive/MyDrive/RelCheck_Data/rbench_data.json"
SAVE_DIR         = "/content/drive/MyDrive/RelCheck_Data/run_600"
FIGS_DIR         = f"{SAVE_DIR}/figures"

# Model IDs
# Vision model: Qwen3-VL-8B on Together.ai (same client already in use)
VLM_MODEL = "Qwen/Qwen3-VL-8B-Instruct"
LLM_MODEL = "meta-llama/Llama-3.3-70B-Instruct-Turbo"
GDINO_ID  = "IDEA-Research/grounding-dino-tiny"
VITPOSE_ID = "usyd-community/vitpose-base-simple"

# Detection
DETECTION_THRESHOLD = 0.25

# ── Init ──────────────────────────────────────────────────
os.environ["TOGETHER_API_KEY"] = TOGETHER_API_KEY
client      = Together(api_key=TOGETHER_API_KEY)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(RANDOM_SEED)

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(FIGS_DIR, exist_ok=True)

nlp = spacy.load("en_core_web_sm")

# ── Helper: safe LLM call with retry ──────────────────────
def llm_call(messages, model=LLM_MODEL, max_tokens=600, retries=3):
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=model, messages=messages,
                temperature=0.0, max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"  API error (attempt {attempt+1}): {e} — retrying in {wait}s")
                time.sleep(wait)
            else:
                return None

# ── Helper: incremental save ───────────────────────────────
def save_checkpoint(data, path):
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)

def load_checkpoint(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

print(f"Device:  {DEVICE}")
print(f"Save to: {SAVE_DIR}")
print(f"Images:  {N_IMAGES if N_IMAGES else 'all available'}")

In [ ]:
# ============================================================
# Cell 1 — Load Models (GDINO + ViTPose)
# ============================================================
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

print("Loading GroundingDINO...")
gdino_processor = AutoProcessor.from_pretrained(GDINO_ID)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_ID).to(DEVICE)
print("  GroundingDINO loaded.")

print("Loading ViTPose...")
vitpose_processor = AutoProcessor.from_pretrained(VITPOSE_ID)
vitpose_model = VitPoseForPoseEstimation.from_pretrained(VITPOSE_ID).to(DEVICE)
print("  ViTPose loaded.")

print(f"\nReady on {DEVICE}.")

In [ ]:
# ============================================================
# Cell 2 -- Load Images + R-Bench Data
# ============================================================
# Auto-downloads R-Bench if not present on Drive.

import pathlib

# ── Download + build rbench_data.json if missing ──
if not os.path.exists(RBENCH_PATH):
    print("R-Bench data not found on Drive. Downloading...")

    RBENCH_DIR = "/content/R-Bench"
    if not os.path.exists(RBENCH_DIR):
        get_ipython().system(f"git clone https://github.com/mrwu-mac/R-Bench {RBENCH_DIR}")
        print("  Cloned R-Bench repo")

    # Download annotations zip from Google Drive
    ANNOTATIONS_ZIP = f"{RBENCH_DIR}/rbench_annotations.zip"
    ANNOTATIONS_DIR = f"{RBENCH_DIR}/data"
    if not os.path.exists(ANNOTATIONS_DIR) or len(list(pathlib.Path(ANNOTATIONS_DIR).glob("*.json"))) == 0:
        DRIVE_FILE_ID = "1sqO0MWBg_HXp5cIKb-nstjNEEk5crUWH"
        get_ipython().system(f'gdown --id {DRIVE_FILE_ID} -O {ANNOTATIONS_ZIP} -q')
        import zipfile
        with zipfile.ZipFile(ANNOTATIONS_ZIP, 'r') as z:
            z.extractall(RBENCH_DIR)
        print("  Downloaded and extracted annotations")

    # Find image-level annotations
    all_jsons = sorted(pathlib.Path(RBENCH_DIR).rglob("*.json"))
    rbench_raw = None
    for f in all_jsons:
        fname = f.name.lower()
        if "image-level" in fname or "image_level" in fname:
            try:
                with open(f) as fh:
                    data = json.load(fh)
                if isinstance(data, list) and len(data) > 50:
                    rbench_raw = data
                    print(f"  Found annotations: {f.name} ({len(data)} entries)")
                    break
            except:
                pass

    # Fallback: any large JSON list with question-like keys
    if rbench_raw is None:
        for f in all_jsons:
            try:
                with open(f) as fh:
                    data = json.load(fh)
                if isinstance(data, list) and len(data) > 100:
                    sample = str(data[0]).lower()
                    if any(k in sample for k in ['question', 'text', 'label', 'answer']):
                        rbench_raw = data
                        print(f"  Found annotations: {f.name} ({len(data)} entries)")
                        break
            except:
                pass

    if rbench_raw is None:
        raise FileNotFoundError(
            "Could not find R-Bench annotations. Try manually downloading from:\n"
            "https://drive.google.com/file/d/1sqO0MWBg_HXp5cIKb-nstjNEEk5crUWH/view"
        )

    # Normalize entries
    def _get(entry, keys, default=None):
        for k in keys:
            if k in entry:
                return entry[k]
        return default

    normalized = []
    for entry in rbench_raw:
        image_filename = _get(entry, ["image", "img", "image_file"], "")
        if image_filename:
            img_id = os.path.splitext(os.path.basename(image_filename))[0]
        else:
            img_id = str(_get(entry, ["image_id", "img_id", "id"], ""))

        question = str(_get(entry, ["text", "question", "q"], ""))
        answer = str(_get(entry, ["label", "answer", "gt"], "")).strip().lower()
        if answer in ("1", "true"):
            answer = "yes"
        elif answer in ("0", "false"):
            answer = "no"

        if question and answer in ("yes", "no"):
            normalized.append({
                "image": image_filename,
                "image_id": img_id,
                "question": question,
                "answer": answer,
            })

    # Save to Drive for future runs
    os.makedirs(os.path.dirname(RBENCH_PATH), exist_ok=True)
    with open(RBENCH_PATH, "w") as f:
        json.dump(normalized, f)
    print(f"  Saved {len(normalized)} normalized entries to {RBENCH_PATH}")

# ── Load R-Bench ──
with open(RBENCH_PATH) as f:
    rbench_data = json.load(f)
print(f"R-Bench: {len(rbench_data)} questions total")

rbench_by_image = defaultdict(list)
for entry in rbench_data:
    # Support both 'image' key (filename) and 'image_id' key
    img_key = entry.get('image', entry.get('image_id', ''))
    rbench_by_image[img_key].append(entry)
print(f"R-Bench unique images: {len(rbench_by_image)}")

# Find images on Drive that have R-Bench questions
cached_images = list(Path(DRIVE_IMAGES_DIR).glob("*.jpg")) + \
                list(Path(DRIVE_IMAGES_DIR).glob("*.jpeg"))
cached_stems = {p.stem: p for p in cached_images}

matched = {}
for rb_key in rbench_by_image:
    stem = re.sub(r'\.(jpg|jpeg|png)$', '', rb_key)
    if stem in cached_stems:
        matched[stem] = {
            "path": str(cached_stems[stem]),
            "questions": rbench_by_image[rb_key],
        }

print(f"Images on Drive with R-Bench questions: {len(matched)}")

if len(matched) == 0:
    print("\n  WARNING: No images found on Drive!")
    print(f"  Expected images in: {DRIVE_IMAGES_DIR}")
    print("  You need to download nocaps images first.")
    print("  The R-Bench images are from nocaps validation set.")

# Sample N_IMAGES (or use all)
all_ids = list(matched.keys())
if N_IMAGES and len(all_ids) > N_IMAGES:
    selected_ids = random.sample(all_ids, N_IMAGES)
    print(f"Sampled {N_IMAGES} of {len(all_ids)} available images")
else:
    selected_ids = all_ids
    print(f"Using all {len(selected_ids)} available images")

# Load images into memory + question mapping
images = {}
img_to_questions = {}
for img_id in selected_ids:
    info = matched[img_id]
    images[img_id] = Image.open(info["path"]).convert("RGB")
    img_to_questions[img_id] = info["questions"]

total_q = sum(len(qs) for qs in img_to_questions.values())
print(f"\nLoaded {len(images)} images")
print(f"R-Bench questions: {total_q} ({total_q/len(images):.1f} per image)")


In [ ]:
# ============================================================
# Cell 3 — Stage 1: Caption Generation (BLIP-2)
# ============================================================
# Checkpoint: loads from Drive if already computed.

BLIP2_PROMPT = "Describe this image in detail. Include all objects, their spatial positions relative to each other, any actions or interactions taking place, and notable attributes like colors and sizes."
CAPTIONS_PATH = f"{SAVE_DIR}/captions.json"

def caption_with_blip2(image):
    inputs = blip2_processor(
        images=image, text=BLIP2_PROMPT, return_tensors="pt"
    ).to(blip2_model.device, torch.float16)
    with torch.no_grad():
        out = blip2_model.generate(**inputs, max_new_tokens=80)
    return blip2_processor.decode(out[0], skip_special_tokens=True).strip()

# ── Load or compute ──
captions = load_checkpoint(CAPTIONS_PATH) or {}

if len(captions) == len(images):
    print(f"Loaded {len(captions)} cached captions from Drive.")
else:
    todo = [img_id for img_id in images if img_id not in captions]
    print(f"Captioning {len(todo)} images (have {len(captions)} cached)...")

    for i, img_id in enumerate(todo):
        captions[img_id] = caption_with_blip2(images[img_id])
        if (i + 1) % 50 == 0 or i == 0:
            print(f"  [{i+1}/{len(todo)}] {img_id[:20]}: {captions[img_id][:70]}")
            save_checkpoint(captions, CAPTIONS_PATH)  # incremental save

    save_checkpoint(captions, CAPTIONS_PATH)
    print(f"\nDone. Captions saved to Drive.")

# Quick sanity check
lens = [len(c) for c in captions.values()]
print(f"Caption length: min={min(lens)}, mean={sum(lens)//len(lens)}, max={max(lens)} chars")

In [ ]:
# ============================================================
# Cell 3b -- Multi-Model Captioning (LLaVA-1.5 + Qwen3-VL-8B)
# ============================================================
# Generates detailed captions from multiple MLLMs for the same images.
# Each model is loaded, used, then unloaded to fit T4 GPU memory.
# Uses "describe in detail" prompt to elicit rich relational descriptions.
# Checkpoint: loads from Drive if already computed.
#
# IMPORTANT: Run this AFTER Cell 3 (BLIP-2 captioning) and BEFORE Cell 4 (KB).
# BLIP-2 must be unloaded first to free GPU memory.

import gc
import base64
from io import BytesIO

def encode_b64(image):
    """Encode PIL image to base64 JPEG string for API calls."""
    buf = BytesIO()
    image.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

LLAVA_CAPTIONS_PATH = f"{SAVE_DIR}/llava_captions.json"
# (legacy variable name kept for checkpoint compatibility)

DESCRIBE_PROMPT = "Describe this image in detail. Include all objects, their spatial positions relative to each other, any actions or interactions taking place, and notable attributes like colors and sizes."

# ── Unload BLIP-2 to free GPU ──
try:
    del blip2_model, blip2_processor
    gc.collect()
    torch.cuda.empty_cache()
    print("Unloaded BLIP-2 to free GPU memory.")
except:
    print("BLIP-2 not loaded, skipping unload.")

# ============================
# LLaVA-1.5-7B (4-bit quantized)
# ============================
llava_captions = load_checkpoint(LLAVA_CAPTIONS_PATH) or {}

if len(llava_captions) >= len(images):
    print(f"LLaVA captions: loaded {len(llava_captions)} from cache.")
else:
    print("Loading LLaVA-1.5-7B (4-bit)...")
    from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    llava_model = LlavaForConditionalGeneration.from_pretrained(
        "llava-hf/llava-1.5-7b-hf",
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    llava_processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")
    print("LLaVA loaded.")

    todo = [img_id for img_id in images if img_id not in llava_captions]
    print(f"Captioning {len(todo)} images with LLaVA...")

    for idx, img_id in enumerate(todo):
        # LLaVA expects: "USER: <image>\n{prompt}\nASSISTANT:"
        conversation = [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": DESCRIBE_PROMPT},
            ]},
        ]
        prompt_text = llava_processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = llava_processor(images=images[img_id], text=prompt_text, return_tensors="pt").to("cuda")

        with torch.no_grad():
            output = llava_model.generate(**inputs, max_new_tokens=256, do_sample=False)
        decoded = llava_processor.decode(output[0], skip_special_tokens=True)

        # Extract only the assistant response
        if "ASSISTANT:" in decoded:
            caption = decoded.split("ASSISTANT:")[-1].strip()
        else:
            caption = decoded.strip()

        llava_captions[img_id] = caption

        if (idx + 1) % 10 == 0:
            print(f"  [{idx+1}/{len(todo)}] {img_id}: {caption[:80]}...")
            save_checkpoint(llava_captions, LLAVA_CAPTIONS_PATH)

    save_checkpoint(llava_captions, LLAVA_CAPTIONS_PATH)
    print(f"LLaVA captioning done: {len(llava_captions)} images.")

    # Unload LLaVA
    del llava_model, llava_processor
    gc.collect()
    torch.cuda.empty_cache()
    print("Unloaded LLaVA.")


# ============================
# Qwen3-VL-8B (via Together.ai — serverless endpoint)
# ============================
# Second VLM captioner: Qwen3-VL-8B (8B, weaker than Maverick 17B MoE).
# Generates detailed captions via Together.ai serverless API.
# Smaller model = more hallucination-prone = better for demonstrating correction.
# We use Qwen2.5-VL-8B (smaller, more hallucination-prone than Maverick)
# as a proxy for a weaker, hallucination-prone captioner.

SCOUT_CAPTIONS_PATH = f"{SAVE_DIR}/scout_captions.json"
SCOUT_MODEL = "Qwen/Qwen3-VL-8B-Instruct"  # serverless on Together.ai

scout_captions = load_checkpoint(SCOUT_CAPTIONS_PATH) or {}

if len(scout_captions) >= len(images):
    print(f"Qwen3-VL-8B captions: loaded {len(scout_captions)} from cache.")
else:
    todo = [img_id for img_id in images if img_id not in scout_captions]
    print(f"Captioning {len(todo)} images with Qwen2.5-VL-8B (via Together.ai)...")

    for idx, img_id in enumerate(todo):
        b64 = encode_b64(images[img_id])
        raw = llm_call(
            messages=[{"role": "user", "content": [
                {"type": "text", "text": DESCRIBE_PROMPT},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            ]}],
            model=SCOUT_MODEL,
            max_tokens=256,
        )
        scout_captions[img_id] = raw.strip() if raw else ""

        if (idx + 1) % 10 == 0:
            print(f"  [{idx+1}/{len(todo)}] {img_id}: {raw[:80] if raw else 'EMPTY'}...")
            save_checkpoint(scout_captions, SCOUT_CAPTIONS_PATH)
        time.sleep(0.3)

    save_checkpoint(scout_captions, SCOUT_CAPTIONS_PATH)
    print(f"Qwen3-VL-8B captioning done: {len(scout_captions)} images.")


# ── Summary ──
print(f"\n=== Caption Summary ===")
for name, caps in [("BLIP-2", captions), ("LLaVA-1.5", llava_captions), ("Qwen2.5-VL-8B", scout_captions)]:
    lengths = [len(c) for c in caps.values() if c]
    if lengths:
        print(f"  {name}: {len(caps)} captions, avg {np.mean(lengths):.0f} chars, avg {np.mean([len(c.split('.')) for c in caps.values() if c]):.1f} sentences")


In [ ]:
# ============================================================
# Cell 4 — Stage 2: Visual Knowledge Base Construction
# ============================================================
# Three layers per image:
#   HARD  — GroundingDINO: objects + counts + bboxes (deterministic)
#   GEOM  — Bbox geometry: pairwise spatial relationships (deterministic)
#   SOFT  — Maverick VLM: actions, attributes, relationships (visual)
# Checkpoint: loads from Drive if already computed.

KB_PATH = f"{SAVE_DIR}/knowledge_bases.json"

BROAD_CATEGORIES = [
    "person", "man", "woman", "child", "boy", "girl",
    "dog", "cat", "bird", "horse", "animal",
    "car", "bicycle", "motorcycle", "bus", "truck",
    "chair", "table", "bench", "couch", "bed",
    "food", "plate", "bowl", "cup", "bottle", "glass",
    "bag", "umbrella", "phone", "book", "sign",
    "hat", "jacket", "vest", "helmet", "glasses",
    "tree", "flower", "grass", "water",
]

MAVERICK_KB_PROMPT = """Describe the RELATIONSHIPS between objects in this image.

An object detector found these objects:
{detection_list}

For each pair of detected objects that interact, describe:
- SPATIAL: Where are they relative to each other?
- ACTIONS: What is each person/animal doing?
- ATTRIBUTES: Relevant attributes tied to relationships

Rules: only describe what you can clearly see. Be brief and factual.
Format as a numbered list."""


def _clean_label(label):
    label = label.strip().lower()
    words = label.split()
    while words and words[0] in ('a', 'an', 'the'):
        words = words[1:]
    seen, clean = [], []
    for w in words:
        if w not in seen:
            seen.append(w)
            clean.append(w)
    return ' '.join(clean) if clean else label


def detect_objects(image, queries):
    text = ". ".join(queries) + "."
    inputs = gdino_processor(images=image, text=text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    results = gdino_processor.post_process_grounded_object_detection(
        outputs, inputs.input_ids,
        threshold=DETECTION_THRESHOLD, text_threshold=DETECTION_THRESHOLD,
        target_sizes=[image.size[::-1]],
    )[0]
    W, H = image.size
    lkey = "text_labels" if "text_labels" in results else "labels"
    dets = []
    for score, label, box in zip(results["scores"], results[lkey], results["boxes"]):
        x1, y1, x2, y2 = box.tolist()
        dets.append((_clean_label(label), score.item(), [x1/W, y1/H, x2/W, y2/H]))
    return dets


def dedup(dets):
    seen = {}
    for label, score, bbox in sorted(dets, key=lambda x: -x[1]):
        key = label.lower().strip()
        if key not in seen:
            seen[key] = [(score, bbox)]
        else:
            is_dup = any(
                max(0, min(bbox[2], eb[2]) - max(bbox[0], eb[0])) *
                max(0, min(bbox[3], eb[3]) - max(bbox[1], eb[1])) /
                ((bbox[2]-bbox[0])*(bbox[3]-bbox[1]) +
                 (eb[2]-eb[0])*(eb[3]-eb[1]) -
                 max(0, min(bbox[2], eb[2])-max(bbox[0], eb[0])) *
                 max(0, min(bbox[3], eb[3])-max(bbox[1], eb[1])) + 1e-8) > 0.5
                for _, eb in seen[key]
            )
            if not is_dup:
                seen[key].append((score, bbox))
    return [(label, s, b) for label, entries in seen.items() for s, b in entries]


def compute_spatial_facts(dets):
    facts = []
    counts = Counter(l for l, _, _ in dets)
    for l, c in counts.items():
        facts.append(f"There {'are' if c > 1 else 'is'} {c} '{l}'")
    labels = list(counts.keys())
    for i, l1 in enumerate(labels):
        for j, l2 in enumerate(labels):
            if i >= j:
                continue
            _, b1 = max([(s, b) for l, s, b in dets if l == l1], key=lambda x: x[0])
            _, b2 = max([(s, b) for l, s, b in dets if l == l2], key=lambda x: x[0])
            c1x, c1y = (b1[0]+b1[2])/2, (b1[1]+b1[3])/2
            c2x, c2y = (b2[0]+b2[2])/2, (b2[1]+b2[3])/2
            contained = (b1[0]>=b2[0]-0.05 and b1[2]<=b2[2]+0.05 and
                         b1[1]>=b2[1]-0.05 and b1[3]<=b2[3]+0.05)
            if contained:
                facts.append(f"'{l1}' is on/inside '{l2}'")
                continue
            dx, dy = c1x-c2x, c1y-c2y
            pos = ("to the left of" if dx < 0 else "to the right of") if abs(dx) > abs(dy) \
                  else ("above" if dy < 0 else "below")
            facts.append(f"'{l1}' is {pos} '{l2}'")
    return facts


def encode_b64(image):
    buf = BytesIO()
    image.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode("utf-8")


def extract_nouns(caption):
    doc = nlp(caption)
    nouns = set()
    for chunk in doc.noun_chunks:
        nouns.add(chunk.root.text.lower())
        nouns.add(chunk.text.lower().strip())
    return list(nouns)


# ── Load or compute KBs ──
# KB stored as serializable dict (detections as lists, not tuples)
kb_raw = load_checkpoint(KB_PATH) or {}
TIMINGS_PATH = f"{SAVE_DIR}/timings.json"
kb_timings = load_checkpoint(TIMINGS_PATH) or {}

# Reconstruct detections as tuples if loaded from JSON
knowledge_bases = {}
for img_id, kb in kb_raw.items():
    knowledge_bases[img_id] = {
        "hard_facts": kb["hard_facts"],
        "spatial_facts": kb["spatial_facts"],
        "visual_description": kb["visual_description"],
        "detections": [tuple(d) for d in kb.get("detections", [])],
    }

if len(knowledge_bases) == len(images):
    print(f"Loaded {len(knowledge_bases)} cached KBs from Drive.")
else:
    todo = [img_id for img_id in images if img_id not in knowledge_bases]
    print(f"Building KB for {len(todo)} images (have {len(knowledge_bases)} cached)...")

    for idx, img_id in enumerate(todo):
        t0 = time.time()
        img = images[img_id]
        cap = captions.get(img_id, "")
        kb = {"hard_facts": [], "spatial_facts": [], "visual_description": "", "detections": []}

        # Detection
        queries = list(set(extract_nouns(cap) + BROAD_CATEGORIES))
        raw = detect_objects(img, queries)
        dets = dedup(raw)
        kb["detections"] = dets

        counts = Counter(l for l, _, _ in dets)
        det_str = "".join(f"- {l} ({c}x)\n" for l, c in counts.most_common())
        kb["hard_facts"] = [f"{c}x {l}" for l, c in counts.most_common()]
        kb["spatial_facts"] = compute_spatial_facts(dets)

        # Maverick VLM
        b64 = encode_b64(img)
        resp_text = llm_call(
            messages=[{"role": "user", "content": [
                {"type": "text", "text": MAVERICK_KB_PROMPT.format(detection_list=det_str)},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            ]}],
            model=VLM_MODEL, max_tokens=500,
        )
        kb["visual_description"] = resp_text or ""

        knowledge_bases[img_id] = kb
        kb_timings[img_id] = {"kb_seconds": round(time.time() - t0, 2)}

        if (idx + 1) % 50 == 0 or idx == 0:
            print(f"  [{idx+1}/{len(todo)}] {img_id[:15]}: {len(dets)} objs, "
                  f"{len(kb['spatial_facts'])} spatial facts")
            # Incremental save (convert tuples to lists for JSON)
            save_kb = {k: {**v, "detections": [list(d) for d in v["detections"]]}
                       for k, v in knowledge_bases.items()}
            save_checkpoint(save_kb, KB_PATH)
        time.sleep(0.3)

    save_kb = {k: {**v, "detections": [list(d) for d in v["detections"]]}
               for k, v in knowledge_bases.items()}
    save_checkpoint(save_kb, KB_PATH)
    save_checkpoint(kb_timings, TIMINGS_PATH)
    print(f"\nDone. KBs saved to Drive.")

avg_objs = np.mean([len(Counter(l for l,_,_ in kb["detections"])) for kb in knowledge_bases.values()])
avg_spatial = np.mean([len(kb["spatial_facts"]) for kb in knowledge_bases.values()])
print(f"Avg objects/image: {avg_objs:.1f} | Avg spatial facts: {avg_spatial:.1f}")

In [ ]:
# ============================================================
# Cell 7 -- Stage 3: RelCheck Enrichment / Correction (v3)
# ============================================================
# v3 unified pipeline -- strategy auto-selected from caption length:
#
#   SHORT captions (< 30 words, e.g. BLIP-2):
#     ENRICHMENT mode -- fix errors + add missing KB facts (full rewrite)
#
#   LONG captions (>= 30 words, e.g. LLaVA-1.5, Qwen3-VL-8B):
#     CORRECTION mode -- Woodpecker-style single Maverick call:
#                        image + deterministic spatial KB -> SCoT -> corrected caption
#     Three-layer verification:
#       1) Deterministic spatial contradiction detection (bbox geometry, zero API cost)
#       2) Skeptical auditor framing + independent visual witness (kb['visual_description'])
#       3) Confidence-gated Maverick: only act on INCORRECT with HIGH/MEDIUM confidence
#     Key property: corrected caption is NEVER shorter than original.
#
# Plug-and-play: no need to specify captioner. System auto-detects.

ENRICHED_PATH = f"{SAVE_DIR}/enriched.json"
SHORT_CAPTION_THRESHOLD  = 30    # words; below -> enrichment, above -> correction

# -- Prompts ----------------------------------------------------------

# Used by enrichment mode (BLIP-2 short captions)
ANALYSIS_PROMPT = """You are a caption quality improver. You have a short image caption and a detailed Visual Knowledge Base (KB) built from the actual image.

CAPTION: "{caption}"

=== VISUAL KNOWLEDGE BASE ===
DETECTED OBJECTS (highly reliable, from object detector):
{hard_facts}

SPATIAL RELATIONSHIPS (from bounding box geometry -- deterministic):
{spatial_facts}

VISUAL DESCRIPTION (from a separate vision model observing the image):
{visual_description}

=== TASK ===
Step 1: Identify ALL problems:
  - ERRORS: Any caption claim that the KB contradicts
  - MISSING: ALL important facts from the KB that the caption omits (objects, spatial positions, actions, attributes)

Step 2: Write a COMPREHENSIVE improved caption (5-8 sentences):
  - Fix any errors using KB evidence
  - Add ALL missing relationships, spatial positions, actions, and attributes from the KB
  - MUST cover spatial relationships: describe WHERE every detected object is relative to others
    (left, right, on top of, below, in front of, behind, near, etc.)
  - MUST cover actions & interactions: describe WHAT objects/people are doing together
  - MUST cover attributes: colors, sizes, materials, states for key objects
  - MUST cover all GroundingDINO detected objects -- do not omit any detected entity
  - The caption should be detailed enough that someone could answer any spatial, action,
    or attribute question about the image from the caption alone
  - Write naturally -- not a list, but flowing descriptive sentences
  - Only include facts supported by the KB

Output valid JSON:
{{"errors": [{{"claim": "...", "correction": "..."}}],
  "missing": [{{"fact": "...", "source": "..."}}],
  "improved_caption": "The comprehensive rewritten caption."}}"""

# Used by enrichment mode verification gate
VERIFY_PROMPT = """Check if this caption is accurate based on KB evidence.

Caption: "{rewritten}"
KB objects: {objects}
KB relationships: {relationships}

FAIL only if caption:
  - Directly contradicts a KB fact
  - Has bad grammar or is incoherent
  - Contains nonsensical repetition
Do NOT fail just because the caption includes details not explicitly in KB.

Answer ONLY "PASS" or "FAIL: [reason]"."""


# Used by _extract_triples — Llama-3.3-70B extracts typed triples from any caption.
# type must be one of: SPATIAL, ACTION, ATTRIBUTE
# SPATIAL  → direction/position (left, right, above, below, in front of, behind, on, under)
# ACTION   → dynamic interaction (riding, holding, eating, carrying, pushing, playing, using)
# ATTRIBUTE→ descriptive state (wearing, covered in, painted, decorated, made of)
TRIPLE_EXTRACT_PROMPT = """Extract all relational claims from this image caption as structured triples.

Caption: "{caption}"

For each claim about how two entities relate, output a JSON object with:
  "subject"  : the entity performing or described first
  "relation" : the relationship word or short phrase (keep it brief, 1-4 words)
  "object"   : the entity being related to
  "type"     : one of SPATIAL | ACTION | ATTRIBUTE
    SPATIAL   = positional/directional (left of, right of, above, below, on, under,
                in front of, behind, next to, inside, near)
    ACTION    = dynamic interaction (riding, holding, eating, carrying, walking beside,
                playing with, pushing, throwing, sitting on, standing on)
    ATTRIBUTE = descriptive state (wearing, covered in, holding [as possession],
                decorated with, attached to, painted on)

Rules:
- Only include claims that describe a relationship BETWEEN two distinct entities.
- Skip generic single-entity descriptions ("a dog is brown").
- Use the shortest natural phrasing for relation (e.g. "on" not "is sitting on top of").
- subject and object should be short noun phrases (1-3 words).

Output ONLY a valid JSON array. No explanation. No markdown.
Example: [{{"subject": "dog", "relation": "left of", "object": "cat", "type": "SPATIAL"}},
          {{"subject": "man", "relation": "riding", "object": "horse", "type": "ACTION"}}]

If no relational claims exist, output: []"""


# Used by _apply_triple_correction — Llama edits exactly one relation in the caption.
# Intentionally does NOT require the exact wrong_phrase to appear verbatim, because
# the triple extractor may normalise the verb (e.g. extracts "riding" but caption
# says "astride"). Llama finds the contextually correct phrase and replaces it.
TRIPLE_CORRECT_PROMPT = """Edit exactly one relationship word or phrase in this caption.

Caption: "{caption}"

Task: The caption incorrectly describes how {subj} and {obj} relate to each other.
Find the word or phrase that expresses this relationship between {subj} and {obj}.
It may not be exactly "{wrong_phrase}" — the caption might use a synonym or different phrasing.
Replace it with: "{correct_phrase}"

Rules:
- Change ONLY the relationship word/phrase between {subj} and {obj}.
- Do NOT change any other words, word order, punctuation, or sentence structure.
- Do NOT add, remove, or reorder any sentences.
- The corrected caption must be the same length (±10%) as the original.
- Output the FULL corrected caption only. No explanation, no prefix, no quotes."""




# Batched correction prompt — Llama fixes ALL confirmed hallucinations in one call.
# Advantages over per-triple surgical edit:
#   1. Handles full sentence removal (e.g. "The treadmill is located in the center")
#   2. No dangling text artifacts from character-level replacement
#   3. Context-aware rewrite — can restructure sentences around false claims
BATCH_CORRECT_PROMPT = """You are correcting an image caption that contains confirmed factual errors.

Original caption:
"{caption}"

CONFIRMED ERRORS (fix ALL of these — do not skip any):
{error_list}

Rules:
1. Fix EVERY error listed above — address each one explicitly.
2. If an entity does not exist in the image, remove the ENTIRE sentence(s) about it.
3. If a relationship is wrong, correct only that relationship — keep surrounding text intact.
4. Do NOT add new information not present in the original caption.
5. Do NOT shorten the caption by removing correct information.
6. The output must be fluent, grammatical English — no dangling phrases.
7. Output the FULL corrected caption ONLY. No explanation, no prefix, no quotes."""

# -- Utility functions -------------------------------------------------


def vlm_call(messages, max_tokens=800, retries=3):
    """Call Qwen3-VL-8B on Together.ai with image + text."""
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model=VLM_MODEL,
                messages=messages,
                temperature=0.0,
                max_tokens=max_tokens,
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if attempt < retries - 1:
                wait = 2 ** attempt
                print(f"  VLM API error (attempt {attempt+1}): {e} — retrying in {wait}s")
                time.sleep(wait)
            else:
                print(f"  VLM API failed after {retries} attempts: {e}")
                return None

def levenshtein(s1, s2):
    """Character-level edit distance for computing edit rate."""
    if s1 == s2: return 0
    m, n = len(s1), len(s2)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[0]
        dp[0] = i
        for j in range(1, n + 1):
            temp = dp[j]
            dp[j] = prev if s1[i-1] == s2[j-1] else 1 + min(prev, dp[j], dp[j-1])
            prev = temp
    return dp[n]


def _check_entity_exists_vqa(entity, pil_image, retries=2):
    """
    Full-image VQA: ask Maverick whether an entity actually exists in the image.
    Used as fallback when GDino can't detect the entity (and therefore crop VQA
    is unavailable). Returns True (exists), False (does not exist), None (uncertain).
    """
    if pil_image is None:
        return None
    buf = BytesIO()
    pil_image.convert("RGB").save(buf, format="JPEG", quality=85)
    b64 = base64.b64encode(buf.getvalue()).decode()

    questions = [
        f"Is there a {entity} visible anywhere in this image? Answer only YES or NO.",
        f"Can you see a {entity} in this scene? Look carefully. Answer YES or NO only.",
    ]
    yes_v = no_v = 0
    for q in questions:
        r = vlm_call([{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
            {"type": "text", "text": q},
        ]}], max_tokens=5)
        if not r:
            continue
        rl = r.strip().lower()
        if "yes" in rl and "no" not in rl:
            yes_v += 1
        elif "no" in rl:
            no_v += 1
    total = yes_v + no_v
    if total == 0:
        return None
    if no_v > yes_v:
        return False   # majority says entity absent
    if yes_v > no_v:
        return True    # majority says entity present
    return None        # tie → uncertain


def normalize_entity(text):
    """Lowercase, strip articles for fuzzy matching."""
    if not text:
        return ""
    text = text.lower().strip()
    for art in ["a ", "an ", "the ", "some "]:
        if text.startswith(art):
            text = text[len(art):]
    return text.strip()


# -- Deterministic spatial contradiction detection ----------------------

# Opposite spatial relations for direct contradiction detection
SPATIAL_OPPOSITES = {
    "left":         "right",
    "right":        "left",
    "above":        "below",
    "below":        "above",
    "on top of":    "below",
    "under":        "above",
    "over":         "under",
    "in front of":  "behind",
    "behind":       "in front of",
    "to the left":  "to the right",
    "to the right": "to the left",
}

# Regex to extract spatial triples from free text
# Matches: '<subj> is [to the] left/right/above/below [of] <obj>'
_SPATIAL_TRIPLE_RE = re.compile(
    r"([a-z][a-z\s]{1,25}?)\s+(?:is\s+)?(?:to\s+the\s+)?"
    r"(left|right|above|below|on top of|under|over|in front of|behind)"
    r"(?:\s+of)?\s+([a-z][a-z\s]{1,25})",
    re.IGNORECASE,
)


# ── Action Geometry Taxonomy (Tier 1) ──────────────────────────────────
# Physical families whose prerequisites can be checked from bounding boxes
# alone (no pose estimation needed). If the geometric rule FAILS, the action
# is a confirmed hallucination. If it PASSES, we still run VQA (necessary
# condition, not sufficient).

ACTION_GEOMETRY_TAXONOMY = {
    # ── Tier 1: object-level bboxes only ──
    "mounting": {
        "verbs": {"riding", "sitting on", "standing on", "straddling",
                  "mounted on", "perched on", "atop", "on top of",
                  "perching on", "seated on", "crouching on"},
        "geometric_rule": "subject_above_object",
        "needs_keypoints": False,
    },
    "containment": {
        "verbs": {"inside", "in", "enclosed by", "covered by",
                  "contained in", "within", "trapped in", "wrapped in"},
        "geometric_rule": "subject_inside_object",
        "needs_keypoints": False,
    },
    "adjacency": {
        "verbs": {"next to", "beside", "near", "alongside", "adjacent to",
                  "close to", "leaning on", "leaning against"},
        "geometric_rule": "bboxes_close",
        "needs_keypoints": False,
    },
    # ── Tier 2: requires ViTPose keypoints ──
    "grasping": {
        "verbs": {"holding", "carrying", "picking up", "pulling", "pushing",
                  "grabbing", "gripping", "lifting", "dragging", "clutching",
                  "catching", "throwing", "tossing"},
        "geometric_rule": "wrist_near_object",
        "needs_keypoints": True,
    },
    "consuming": {
        "verbs": {"eating", "drinking", "tasting", "licking", "biting",
                  "sipping", "chewing", "feeding on"},
        "geometric_rule": "nose_near_object",
        "needs_keypoints": True,
    },
}

# Build a reverse lookup: verb → family name
_VERB_TO_FAMILY = {}
for _fam, _spec in ACTION_GEOMETRY_TAXONOMY.items():
    for _v in _spec["verbs"]:
        _VERB_TO_FAMILY[_v] = _fam


def _classify_action_family(relation_verb):
    """Map a relation verb to its physical family (or None if no rule exists)."""
    rel = relation_verb.strip().lower()
    if rel in _VERB_TO_FAMILY:
        return _VERB_TO_FAMILY[rel]
    # Fuzzy: only match multi-word verbs where the FULL known verb appears
    # as a substring in the relation. Single-word verbs like "in" are too
    # ambiguous for fuzzy matching ("pushing in" ≠ containment).
    for verb, fam in _VERB_TO_FAMILY.items():
        if len(verb.split()) >= 2 and verb in rel:
            return fam
    return None


# ── COCO keypoint indices ──
KP_NOSE        = 0
KP_LEFT_WRIST  = 9
KP_RIGHT_WRIST = 10


def _get_person_keypoints(pil_image, person_box_norm):
    """
    Run ViTPose on a detected person to get 17 COCO keypoints.

    Args:
        pil_image: PIL Image
        person_box_norm: [x1, y1, x2, y2] normalised to [0, 1]

    Returns:
        dict with 'keypoints' (17×2 pixel coords) and 'scores' (17,)
        or None if detection fails.
    """
    W, H = pil_image.size
    x1, y1, x2, y2 = person_box_norm
    # Convert to COCO format: [left, top, width, height] in pixels
    coco_box = [x1 * W, y1 * H, (x2 - x1) * W, (y2 - y1) * H]

    try:
        inputs = vitpose_processor(
            pil_image, boxes=[[coco_box]], return_tensors="pt"
        ).to(vitpose_model.device)

        with torch.no_grad():
            outputs = vitpose_model(**inputs)

        results = vitpose_processor.post_process_pose_estimation(
            outputs, boxes=[[coco_box]]
        )

        if results and results[0]:
            kp = results[0][0]
            keypoints = kp["keypoints"].cpu().numpy()   # (17, 2) in pixels
            scores    = kp["scores"].cpu().numpy()       # (17,)
            # Normalise to [0, 1]
            keypoints[:, 0] /= W
            keypoints[:, 1] /= H
            return {"keypoints": keypoints, "scores": scores}
    except Exception as e:
        print(f"    ViTPose error: {e}")
    return None


def _check_action_geometry(family, subj_box, obj_box, keypoints=None):
    """
    Test the geometric prerequisite for an action family.

    Args:
        family: one of "mounting", "containment", "adjacency",
                "grasping", "consuming"
        subj_box, obj_box: [x1, y1, x2, y2] normalised to [0, 1]
        keypoints: optional dict from _get_person_keypoints
                   (required for Tier 2 families)

    Returns:
        True  → prerequisite MET (inconclusive, still need VQA)
        False → prerequisite VIOLATED (hallucination confirmed)
        None  → cannot check (missing keypoints for Tier 2)
    """
    sx1, sy1, sx2, sy2 = subj_box  # subject
    ox1, oy1, ox2, oy2 = obj_box   # object

    s_cx, s_cy = (sx1 + sx2) / 2, (sy1 + sy2) / 2
    o_cx, o_cy = (ox1 + ox2) / 2, (oy1 + oy2) / 2
    s_h = sy2 - sy1
    o_h = oy2 - oy1
    o_w = ox2 - ox1

    # ── Tier 1: bbox-only checks ──

    if family == "mounting":
        # Subject should be above/on the object:
        #   1. Subject's bottom edge (sy2) should be near object's top region
        #      (within top 40% of object — generous to handle perspective)
        #   2. Horizontal overlap: subject and object x-ranges must intersect
        top_region = oy1 + 0.65 * o_h  # generous: rider's feet reach ~60% of horse height
        subject_bottom_in_top = sy2 <= top_region + 0.05  # small tolerance
        x_overlap = min(sx2, ox2) - max(sx1, ox1)
        has_x_overlap = x_overlap > 0.02 * max(o_w, 0.01)
        return subject_bottom_in_top and has_x_overlap

    elif family == "containment":
        # Subject bbox should be mostly inside object bbox (>50% overlap)
        inter_x = max(0, min(sx2, ox2) - max(sx1, ox1))
        inter_y = max(0, min(sy2, oy2) - max(sy1, oy1))
        inter_area = inter_x * inter_y
        subj_area = max((sx2 - sx1) * (sy2 - sy1), 1e-6)
        containment_ratio = inter_area / subj_area
        return containment_ratio > 0.50

    elif family == "adjacency":
        # Bboxes should be close — gap between nearest edges < 30% of avg size
        gap_x = max(0, max(sx1, ox1) - min(sx2, ox2))
        gap_y = max(0, max(sy1, oy1) - min(sy2, oy2))
        gap = (gap_x**2 + gap_y**2) ** 0.5
        avg_size = ((s_h + o_h) / 2 + ((sx2-sx1) + o_w) / 2) / 2
        return gap < 0.30 * max(avg_size, 0.01)

    # ── Tier 2: keypoint-based checks ──

    if keypoints is None:
        return None  # can't check without keypoints

    kp_xy = keypoints["keypoints"]   # (17, 2) normalised
    kp_sc = keypoints["scores"]      # (17,)
    KP_CONF_THRESH = 0.3             # minimum keypoint confidence

    if family == "grasping":
        # At least one wrist must be near/inside the object bbox.
        # Margin scales with object size (50% of object dimension on each side)
        # to handle perspective while avoiding matching distant wrists.
        margin_x = max(0.5 * o_w, 0.03)  # at least 3% of image
        margin_y = max(0.5 * o_h, 0.03)
        obj_expanded = [ox1 - margin_x, oy1 - margin_y,
                        ox2 + margin_x, oy2 + margin_y]
        for wrist_idx in [KP_LEFT_WRIST, KP_RIGHT_WRIST]:
            if kp_sc[wrist_idx] < KP_CONF_THRESH:
                continue
            wx, wy = kp_xy[wrist_idx]
            if (obj_expanded[0] <= wx <= obj_expanded[2] and
                obj_expanded[1] <= wy <= obj_expanded[3]):
                return True  # wrist near object — prerequisite met
        # Neither wrist is near the object
        return False

    elif family == "consuming":
        # Nose (proxy for mouth) must be near the object bbox.
        if kp_sc[KP_NOSE] < KP_CONF_THRESH:
            return None  # nose not detected — can't check
        nx, ny = kp_xy[KP_NOSE]
        # Margin scales with object size (75% — more generous because
        # mouth is below nose, and food can be held at varying angles)
        margin_x = max(0.75 * o_w, 0.04)
        margin_y = max(0.75 * o_h, 0.04)
        obj_expanded = [ox1 - margin_x, oy1 - margin_y,
                        ox2 + margin_x, oy2 + margin_y]
        if (obj_expanded[0] <= nx <= obj_expanded[2] and
            obj_expanded[1] <= ny <= obj_expanded[3]):
            return True  # nose near food/drink — prerequisite met
        return False

    return True  # unknown family — don't block


def _extract_spatial_triples_text(text):
    """
    Extract (subject, relation, object) spatial triples from free text.
    Returns list of (subj, rel, obj) tuples, all lowercase.
    """
    triples = []
    for m in _SPATIAL_TRIPLE_RE.finditer(text.lower()):
        subj = m.group(1).strip().rstrip(" ,;")
        rel  = m.group(2).strip()
        obj  = m.group(3).strip().rstrip(" ,;.")
        triples.append((subj, rel, obj))
    return triples


def _parse_spatial_facts(spatial_facts):
    """
    Parse the KB's spatial_facts list into (subj, rel, obj) tuples.
    spatial_facts lines look like: "'dog' is to the left of 'couch'"
    """
    parsed = []
    for fact in spatial_facts:
        fact_clean = fact.replace("'", "").replace('"', "")
        for m in _SPATIAL_TRIPLE_RE.finditer(fact_clean.lower()):
            subj = m.group(1).strip().rstrip(" ,;")
            rel  = m.group(2).strip()
            obj  = m.group(3).strip().rstrip(" ,;.")
            parsed.append((subj, rel, obj))
    return parsed


# Stop words and filler verbs stripped when extracting core noun from regex captures
_ENTITY_STOP = frozenset([
    "a","an","the","some","is","are","was","were","be","been","being",
    "sits","sit","standing","stand","positioned","position","placed","place",
    "seen","located","appears","appear","lying","lies","lay","resting","rest",
])

def _core_noun(text):
    """
    Extract the core noun from a regex-captured entity span.
    Strips leading articles, filler verbs, and trailing noise words.
    Returns first 1-2 meaningful words (the actual object/person name).
    e.g. 'a dog sits'           -> 'dog'
         'a dog is positioned'  -> 'dog'
         'the couch while a'    -> 'couch'
         'large red ball'       -> 'large red ball' (adjective + noun kept)
    """
    words = normalize_entity(text).split()
    # Drop leading stop words
    while words and words[0] in _ENTITY_STOP:
        words = words[1:]
    # Drop trailing stop words / noise
    while words and words[-1] in _ENTITY_STOP:
        words = words[:-1]
    # Cap at 3 words
    return " ".join(words[:3]).strip()


def _entity_matches(cap_entity, kb_entity):
    """
    Fuzzy match using core noun extraction.
    'a dog sits' matches 'dog', 'a dog is positioned', 'large dog', etc.
    """
    core_cap = _core_noun(cap_entity)
    core_kb  = _core_noun(kb_entity)
    if not core_cap or not core_kb:
        return False
    # Match if either core noun contains the other
    return (core_kb in core_cap) or (core_cap in core_kb)


def _check_spatial_contradictions(caption, spatial_facts):
    """
    Deterministically detect spatial contradictions between the caption
    and GroundingDINO bbox geometry.

    A contradiction exists when the caption states A <rel> B but the KB
    geometry says A <opposite_of_rel> B.  Uses fuzzy entity matching so
    'a dog sits' correctly matches KB entity 'dog'.

    Returns list of contradiction strings, e.g.:
      ["Caption says 'a dog sits left of couch' but geometry shows: 'dog right of couch'"]
    """
    if not spatial_facts:
        return []

    kb_triples = _parse_spatial_facts(spatial_facts)
    if not kb_triples:
        return []

    caption_triples = _extract_spatial_triples_text(caption)
    if not caption_triples:
        return []

    contradictions = []
    for (cap_subj, cap_rel, cap_obj) in caption_triples:
        cap_opposite = SPATIAL_OPPOSITES.get(cap_rel.lower())
        if not cap_opposite:
            continue

        for (kb_subj, kb_rel, kb_obj) in kb_triples:
            # Fuzzy entity match: 'a dog sits' matches 'dog'
            if not (_entity_matches(cap_subj, kb_subj) and _entity_matches(cap_obj, kb_obj)):
                continue
            # Contradiction: caption claims rel, KB says the opposite
            if kb_rel == cap_opposite:
                kb_fact_str = next(
                    (f for f in spatial_facts
                     if normalize_entity(kb_subj) in f.lower()
                     and normalize_entity(kb_obj) in f.lower()),
                    f"'{kb_subj}' {kb_rel} '{kb_obj}'"
                )
                contradictions.append(
                    f"Caption says '{cap_subj} {cap_rel} {cap_obj}' "
                    f"but geometry shows: {kb_fact_str}"
                )
                break  # one contradiction report per caption triple

    return contradictions
def _enrich_short_caption(img_id, caption, kb):
    """Full KB-guided enrichment for captions under 30 words."""
    hard    = "\n".join(f"- {f}" for f in kb["hard_facts"])    or "- None detected"
    spatial = "\n".join(f"- {f}" for f in kb["spatial_facts"]) or "- No spatial facts"
    visual  = kb["visual_description"][:800]                     or "- No visual description"

    prompt = ANALYSIS_PROMPT.format(
        caption=caption, hard_facts=hard,
        spatial_facts=spatial, visual_description=visual
    )
    improved = caption
    errors, missing = [], []

    raw = llm_call([{"role": "user", "content": prompt}], max_tokens=1000)
    if raw:
        raw = re.sub(r'^```json\s*', '', raw)
        raw = re.sub(r'```\s*$', '', raw)
        if raw.count('{') > raw.count('}'):
            raw += '"}' 
        try:
            result  = json.loads(raw)
            errors  = result.get("errors", [])
            missing = result.get("missing", [])
            cand    = result.get("improved_caption", "").strip().strip('"').strip("'")
            if cand and len(cand) >= 15:
                improved = cand
        except Exception:
            pass

    # Sentence cap
    if improved != caption:
        n_sent = len([s for s in re.split(r'[.!?]+', improved) if s.strip()])
        if n_sent > 10:  # prompt asks for 5-8 sentences; allow 2 slack
            improved = caption

    # Verification gate
    if improved != caption:
        counts  = Counter(l for l, _, _ in kb.get("detections", []))
        obj_str = ", ".join(f"{c}x {l}" for l, c in counts.most_common(10))
        rel_str = kb["visual_description"][:500]
        verdict = llm_call(
            [{"role": "user", "content": VERIFY_PROMPT.format(
                rewritten=improved, objects=obj_str, relationships=rel_str
            )}],
            max_tokens=50,
        )
        if verdict and verdict.upper().startswith("FAIL"):
            improved = caption

    edit_rate = levenshtein(caption, improved) / max(len(caption), len(improved), 1)
    return {
        "caption": caption, "corrected": improved,
        "errors": errors,   "missing": missing,
        "edit_rate": edit_rate,
        "status": "modified" if improved != caption else "unchanged",
        "mode": "enrich",
    }
    rel_norm = rel.lower()
    # Also check phrase variants: "left" matches "to the left", "left of", etc.
    rel_variants = {
        "left":  ["left"],  "right": ["right"],
        "above": ["above","on top of"],  "below": ["below","beneath","under"],
        "behind":["behind","in back of"], "in front of":["in front of","front"],
    }
    keywords = rel_variants.get(rel_norm, [rel_norm])
    for kw in keywords:
        if kw not in cap_lower:
            continue
        idx = cap_lower.find(kw)
        context = cap_lower[max(0, idx - 80): idx + 80]
        # If the relation keyword appears and EITHER entity is nearby, assume covered
        core_s = _core_noun(subj)
        core_o = _core_noun(obj)
        if core_s in context or core_o in context:
            return True
        # Also check synonyms in the context window
        for syn in _ENTITY_SYNONYMS.get(core_s, []):
            if syn in context:
                return True
        for syn in _ENTITY_SYNONYMS.get(core_o, []):
            if syn in context:
                return True
    return False



def _apply_geometry_corrections(caption, geo_contradictions):
    """
    Surgical fallback: directly replace wrong spatial phrases in the original caption
    using deterministic bbox geometry. Used when Maverick's full rewrite is rejected
    by the length guard (too compressed).

    Parses each geometry contradiction string:
      "Caption says 'X rel Y' but geometry shows: ..."
    Finds 'rel' near X/Y entities in caption and swaps it for the opposite.

    Returns (fixed_caption, n_fixed).
    """
    if not geo_contradictions:
        return caption, 0

    fixed = caption
    n_fixed = 0

    for contra in geo_contradictions:
        # Parse the wrong caption claim: "Caption says 'dog left of couch' but ..."
        m_cap = re.search(r"Caption says '(.+?)'", contra, re.IGNORECASE)
        if not m_cap:
            continue
        cap_claim = m_cap.group(1).strip()

        # Extract the spatial relation from the claim
        m_rel = _SPATIAL_TRIPLE_RE.search(cap_claim.lower())
        if not m_rel:
            continue
        wrong_rel = m_rel.group(2).strip()
        correct_rel = SPATIAL_OPPOSITES.get(wrong_rel)
        if not correct_rel:
            continue

        # Find the exact claim phrase in the caption (case-insensitive)
        claim_lower = cap_claim.lower()
        fixed_lower = fixed.lower()
        idx = fixed_lower.find(claim_lower)
        if idx < 0:
            # Phrase not found verbatim — try replacing just the relation keyword near entities
            rel_idx = fixed_lower.find(wrong_rel)
            if rel_idx < 0:
                continue
            # Check entity proximity (within 60 chars of the relation keyword)
            context = fixed_lower[max(0, rel_idx - 60): rel_idx + 60]
            subj = m_rel.group(1).strip()
            obj_ = m_rel.group(3).strip()
            core_s = _core_noun(subj)
            core_o = _core_noun(obj_)
            if not (core_s in context or core_o in context):
                continue
            # Replace just the relation keyword at rel_idx
            fixed = fixed[:rel_idx] + correct_rel + fixed[rel_idx + len(wrong_rel):]
            n_fixed += 1
            continue

        # Replace the relation keyword within the matched phrase
        phrase = fixed[idx: idx + len(cap_claim)]
        corrected_phrase = re.sub(
            r'\b' + re.escape(wrong_rel) + r'\b',
            correct_rel, phrase, flags=re.IGNORECASE, count=1,
        )
        if corrected_phrase != phrase:
            fixed = fixed[:idx] + corrected_phrase + fixed[idx + len(cap_claim):]
            n_fixed += 1

    return fixed, n_fixed



def _caption_name_for(kb_entity, cap_lower):
    """
    Return the synonym of kb_entity that actually appears in cap_lower.
    Falls back to kb_entity's core noun if nothing matches.
    Used by the addendum builder to produce "the man is left of the car"
    instead of "the person is left of the car" when caption says "man".
    """
    core = _core_noun(kb_entity)
    if not core:
        return kb_entity
    if core in cap_lower:
        return core
    for syn in _ENTITY_SYNONYMS.get(core, []):
        if syn in cap_lower:
            return syn
    return core  # fallback: use KB label


# COCO / GroundingDINO class synonyms — maps detector labels to caption synonyms.
# Used by _relation_already_expressed and _build_spatial_addendum to match
# caption entity names (e.g. "man") against KB labels (e.g. "person").
_ENTITY_SYNONYMS = {
    "person":       ["person","man","woman","child","boy","girl","individual","human","people"],
    "car":          ["car","vehicle","automobile","sedan","suv","truck"],
    "couch":        ["couch","sofa","settee"],
    "chair":        ["chair","seat","stool"],
    "dog":          ["dog","puppy","canine"],
    "cat":          ["cat","kitten","feline"],
    "horse":        ["horse","pony"],
    "bicycle":      ["bicycle","bike","cycle"],
    "motorcycle":   ["motorcycle","motorbike"],
    "truck":        ["truck","lorry","van"],
    "bus":          ["bus","coach"],
    "boat":         ["boat","ship","vessel"],
    "airplane":     ["airplane","plane","aircraft","jet"],
    "bird":         ["bird","pigeon","seagull","sparrow"],
    "cow":          ["cow","cattle","bull"],
    "sheep":        ["sheep","lamb"],
    "bench":        ["bench","seat"],
    "dining table": ["table","dining table","desk"],
    "tv":           ["tv","television","monitor","screen","display"],
    "laptop":       ["laptop","computer","notebook"],
    "cell phone":   ["phone","cell phone","smartphone","mobile"],
    "backpack":     ["backpack","bag","rucksack"],
    "handbag":      ["handbag","bag","purse"],
    "sports ball":  ["ball"],
    "cup":          ["cup","mug","glass"],
    "bottle":       ["bottle"],
    "bowl":         ["bowl"],
    "book":         ["book","magazine"],
    "vase":         ["vase"],
    "clock":        ["clock","watch"],
    "potted plant": ["plant","flower","pot"],
    "fire hydrant": ["fire hydrant","hydrant"],
    "traffic light":["traffic light","stoplight"],
    "umbrella":     ["umbrella"],
    "frisbee":      ["frisbee"],
    "skateboard":   ["skateboard","board"],
    "surfboard":    ["surfboard","board"],
    "skis":         ["skis","ski"],
    "kite":         ["kite"],
    "baseball bat": ["bat","baseball bat"],
    "tennis racket":["racket","tennis racket"],
    "suitcase":     ["suitcase","luggage","bag"],
    "tie":          ["tie","necktie"],
    "keyboard":     ["keyboard"],
    "mouse":        ["mouse"],
    "remote":       ["remote","controller"],
    "pizza":        ["pizza"],
    "cake":         ["cake"],
    "sandwich":     ["sandwich"],
    "refrigerator": ["refrigerator","fridge"],
    "oven":         ["oven","stove"],
    "sink":         ["sink"],
    "toilet":       ["toilet"],
    "bed":          ["bed"],
}


def _relation_already_expressed(subj, rel, obj, cap_lower):
    """
    Heuristic: return True if the caption already states subj <rel> obj
    OR the equivalent reversed relation (obj <opposite> subj).
    Uses a loose keyword proximity check.
    """
    rel_norm = rel.lower()
    # Also check phrase variants: "left" matches "to the left", "left of", etc.
    rel_variants = {
        "left":  ["left"],  "right": ["right"],
        "above": ["above","on top of"],  "below": ["below","beneath","under"],
        "behind":["behind","in back of"], "in front of":["in front of","front"],
    }
    keywords = rel_variants.get(rel_norm, [rel_norm])
    for kw in keywords:
        if kw not in cap_lower:
            continue
        idx = cap_lower.find(kw)
        context = cap_lower[max(0, idx - 80): idx + 80]
        # If the relation keyword appears and EITHER entity is nearby, assume covered
        core_s = _core_noun(subj)
        core_o = _core_noun(obj)
        if core_s in context or core_o in context:
            return True
        # Also check synonyms in the context window
        for syn in _ENTITY_SYNONYMS.get(core_s, []):
            if syn in context:
                return True
        for syn in _ENTITY_SYNONYMS.get(core_o, []):
            if syn in context:
                return True
    return False


def _build_spatial_addendum(corrected_caption, kb, max_facts=5):
    """
    Append missing KB spatial facts to the (corrected) caption.

    v2 strategy (relaxed):
      - REMOVE the requirement that both entities already appear in the caption.
        GroundingDINO bbox facts are deterministic ground truth; it's always
        safe to state them even if the captioner used different words or omitted
        an object. R-POPE asks about entities by their canonical name, so adding
        "the person is to the left of the car" helps the LLM-judge even if
        LLaVA called them "the man" and "the vehicle".
      - KEEP the deduplication check: if the relation is already expressed
        (with synonym-aware entity matching), skip to avoid repetition.

    Returns (new_caption_str, n_added) tuple.
    """
    spatial_facts = kb.get("spatial_facts", [])
    if not spatial_facts:
        return corrected_caption, 0

    cap_lower = corrected_caption.lower()
    missing = []

    for fact in spatial_facts:
        triples = _parse_spatial_facts([fact])
        if not triples:
            continue
        subj, rel, obj = triples[0]
        if not subj or not obj:
            continue

        # Skip if this spatial relation is already expressed in caption
        if _relation_already_expressed(subj, rel, obj, cap_lower):
            continue

        cap_subj = _caption_name_for(subj, cap_lower)
        cap_obj  = _caption_name_for(obj,  cap_lower)
        missing.append((subj, rel, obj, fact, cap_subj, cap_obj))
        if len(missing) >= max_facts:
            break

    if not missing:
        return corrected_caption, 0

    # Build addendum sentence
    fact_phrases = [f"the {cs} is {r} the {co}" for s, r, o, _, cs, co in missing]
    if len(fact_phrases) == 1:
        addendum = f"Spatially, {fact_phrases[0]}."
    elif len(fact_phrases) == 2:
        addendum = f"Spatially, {fact_phrases[0]}, and {fact_phrases[1]}."
    else:
        joined = ", ".join(fact_phrases[:-1]) + f", and {fact_phrases[-1]}"
        addendum = f"Spatially, {joined}."

    new_cap = corrected_caption.rstrip() + " " + addendum
    return new_cap, len(missing)
def _extract_triples(caption):
    """Use Llama to extract (subject, relation, object, type) triples.

    type field is inferred from the relation verb if Llama omits it or uses
    wrong capitalisation — prevents silent empty returns when Llama output
    is valid but slightly off-format.
    """
    _SPATIAL_WORDS = {"left","right","above","below","behind","front",
                      "beside","next to","on top of","under","over",
                      "in front of","near","inside","outside","between"}
    _ACTION_WORDS  = {"riding","holding","carrying","eating","drinking",
                      "wearing","pushing","pulling","walking","running",
                      "sitting","standing","playing","using","throwing",
                      "catching","carrying","holding","driving","leading"}

    # Use str.replace to avoid KeyError if caption contains { or }
    prompt = TRIPLE_EXTRACT_PROMPT.replace("{caption}", caption)

    raw = llm_call(
        [{"role": "user", "content": prompt}],
        max_tokens=600,
    )
    if not raw:
        print(f"    [extract_triples] llm_call returned empty for caption: {caption[:80]!r}")
        return []

    # Strip markdown fences
    raw_stripped = re.sub(r'^```json\s*', '', raw.strip(), flags=re.MULTILINE)
    raw_stripped = re.sub(r'```\s*$', '', raw_stripped.strip(), flags=re.MULTILINE)
    raw_stripped = raw_stripped.strip()

    print(f"    [extract_triples] raw={raw_stripped[:200]!r}")

    try:
        parsed = json.loads(raw_stripped)

        # Handle Llama wrapping output: {"triples": [...]} or {"relations": [...]}
        if isinstance(parsed, dict):
            for key in ("triples", "relations", "result", "data", "output"):
                if key in parsed and isinstance(parsed[key], list):
                    parsed = parsed[key]
                    print(f"    [extract_triples] unwrapped dict key '{key}'")
                    break
            else:
                print(f"    [extract_triples] got dict but no list key: {list(parsed.keys())}")
                return []

        if not isinstance(parsed, list):
            print(f"    [extract_triples] unexpected type: {type(parsed)}")
            return []

        result = []
        for t in parsed:
            if not isinstance(t, dict):
                continue
            if not all(k in t for k in ("subject", "relation", "object")):
                print(f"    [extract_triples] skipping malformed triple: {t}")
                continue

            # Normalise type: accept any capitalisation, infer if missing
            typ = str(t.get("type", "")).upper().strip()
            if typ not in ("SPATIAL", "ACTION", "ATTRIBUTE"):
                rel_lower = t.get("relation", "").lower()
                if any(w in rel_lower for w in _SPATIAL_WORDS):
                    typ = "SPATIAL"
                elif any(w in rel_lower for w in _ACTION_WORDS):
                    typ = "ACTION"
                else:
                    typ = "ACTION"   # safe default: verified by VQA

            t["type"] = typ
            result.append(t)

        return result

    except json.JSONDecodeError as e:
        print(f"    [extract_triples] JSON parse error: {e} | raw={raw_stripped[:300]!r}")
    except Exception as e:
        print(f"    [extract_triples] unexpected error: {e}")
    return []


def _find_best_bbox(entity_name, kb):
    """
    Return the highest-confidence GroundingDINO bbox for entity_name.

    Multi-pass matching to handle descriptive entity phrases from LLaVA/Qwen
    (e.g. "young man", "red bicycle", "the woman on the left"):
      Pass 1: exact core noun   → _ENTITY_SYNONYMS lookup
      Pass 2: each word in core → _ENTITY_SYNONYMS lookup (handles adjective+noun)
      Pass 3: substring match   → any synonym appears inside entity or vice versa

    Returns [x1, y1, x2, y2] normalised, or None if not found.
    """
    core = _core_noun(entity_name)
    if not core:
        return None

    detections = kb.get("detections", [])
    if not detections:
        return None

    def _candidate_synonyms(name):
        """All synonyms for a name, plus the name itself."""
        syns = set(_ENTITY_SYNONYMS.get(name, []) + [name])
        # Also try looking up each individual word
        for w in name.split():
            syns.update(_ENTITY_SYNONYMS.get(w, [w]) + [w])
        return syns

    core_syns = _candidate_synonyms(core)

    best_score, best_box = -1, None
    for label, score, box in detections:
        label_core = _core_noun(label)
        label_syns = _candidate_synonyms(label_core)

        matched = False
        # Pass 1+2: synonym set intersection
        if core_syns & label_syns:
            matched = True
        # Pass 3: substring — "young man" contains "man" which is in person synonyms
        if not matched:
            for cs in core_syns:
                for ls in label_syns:
                    if cs in ls or ls in cs:
                        matched = True
                        break
                if matched:
                    break

        if matched and score > best_score:
            best_score, best_box = score, box

    return best_box


def _crop_to_bboxes(pil_image, box1, box2, padding=0.15):
    """
    Crop PIL image to the union of two normalised bboxes with padding.
    boxes are [x1, y1, x2, y2] in [0, 1] range.
    """
    W, H = pil_image.size
    xs = [box1[0], box1[2], box2[0], box2[2]]
    ys = [box1[1], box1[3], box2[1], box2[3]]
    x1 = max(0.0, min(xs) - padding)
    y1 = max(0.0, min(ys) - padding)
    x2 = min(1.0, max(xs) + padding)
    y2 = min(1.0, max(ys) + padding)
    left, top, right, bottom = int(x1*W), int(y1*H), int(x2*W), int(y2*H)
    # Ensure minimum crop size
    if right - left < 32 or bottom - top < 32:
        return pil_image  # fall back to full image
    return pil_image.crop((left, top, right, bottom))


def _consensus_confirms_triple(subj, rel, obj, cross_captions):
    """
    Returns True if ALL available cross-captioners mention (subj rel obj)
    in their caption, indicating the claim is almost certainly correct.
    Requires at least 2 cross-captioner captions.

    Uses proximity heuristic: the relation verb must appear within 80 chars
    of the subject OR object in the other caption.  Synonym-aware.

    If consensus confirms → skip VQA entirely (free signal, zero API cost).
    """
    if not cross_captions or len(cross_captions) < 2:
        return False

    core_s   = _core_noun(subj)
    core_o   = _core_noun(obj)
    rel_lower = rel.lower()

    confirmed = 0
    for cap_text in cross_captions.values():
        if not cap_text:
            continue
        cap_lower = cap_text.lower()
        if rel_lower not in cap_lower:
            continue
        # Walk every occurrence of the relation verb
        idx = cap_lower.find(rel_lower)
        while idx >= 0:
            context = cap_lower[max(0, idx - 80): idx + 80]
            subj_near = (core_s in context or
                         any(s in context for s in _ENTITY_SYNONYMS.get(core_s, [])))
            obj_near  = (core_o in context or
                         any(s in context for s in _ENTITY_SYNONYMS.get(core_o, [])))
            if subj_near and obj_near:
                confirmed += 1
                break
            idx = cap_lower.find(rel_lower, idx + 1)

    # ALL cross-captioners must confirm (not just majority) to skip VQA
    return confirmed >= len(cross_captions)


def _verify_action_triple(subj, rel, obj, kb, pil_image, n_questions=3):
    """
    Verify an action/attribute triple using crop-based focused VQA
    with multi-question voting (default: 3 paraphrased questions).

    1. Locate subject and object bboxes in GroundingDINO detections.
    2. Crop image to their union (+ padding) to remove background noise.
    3. Ask Maverick n_questions paraphrased YES/NO questions; vote.

    Paraphrasing reduces noise from single-call VQA (a correct claim
    has ~25% chance of being called NO on a single call; with 3 calls
    and majority vote, false-positive rate drops to ~7%).

    Intentionally neutral — no geometry hints are passed here.
    VQA and geometry are independent signals.

    Returns True  (verified correct: majority YES),
            False (hallucination: majority NO),
            None  (cannot verify — missing bbox, API failure, or tie).
    """
    subj_box = _find_best_bbox(subj, kb)
    obj_box  = _find_best_bbox(obj,  kb)

    if subj_box is None or obj_box is None:
        return None  # can't ground without bboxes

    crop = _crop_to_bboxes(pil_image, subj_box, obj_box, padding=0.15)

    buf = BytesIO()
    crop.convert("RGB").save(buf, format="JPEG", quality=85)
    crop_b64 = base64.b64encode(buf.getvalue()).decode()

    # Paraphrased question templates — same claim, different phrasing.
    # Inspired by VisMin (NeurIPS 2024) multi-question filtering approach.
    question_templates = [
        f'In this image, is the {subj} {rel} the {obj}? '
        f'Look carefully at the cropped region. Answer only YES or NO.',
        f'Does the {subj} appear to be {rel} the {obj} here? '
        f'Observe the cropped region closely. Answer YES or NO only.',
        f'Can you see the {subj} actively {rel} the {obj} in this scene? '
        f'Answer only YES or NO.',
    ]
    questions = question_templates[:n_questions]

    yes_votes = 0
    no_votes  = 0

    for q in questions:
        result = vlm_call([{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{crop_b64}"}},
            {"type": "text", "text": q},
        ]}], max_tokens=5)
        if not result:
            continue
        r = result.strip().lower()
        if "yes" in r and "no" not in r:
            yes_votes += 1
        elif "no" in r:
            no_votes  += 1
        # Ambiguous responses: neither yes nor no → abstain

    total = yes_votes + no_votes
    if total == 0:
        return None  # all calls failed or ambiguous
    if yes_votes > no_votes:
        return True   # majority YES → claim verified correct
    if no_votes > yes_votes:
        return False  # majority NO  → hallucination confirmed
    return None       # exact tie → treat as uncertain, skip correction


def _apply_triple_correction(caption, wrong_phrase, correct_phrase,
                              subj="", obj_=""):
    """
    Fix exactly one relationship word/phrase in the caption.

    wrong_phrase: the extracted relation verb (e.g. "riding")
    correct_phrase: the correct relation (e.g. "standing next to")
    subj, obj_: entity names for context (used in fast path and Llama prompt)

    Fast path: verb found verbatim → find occurrence nearest to subj/obj
               mentions to avoid replacing the wrong instance.
    Llama path: verb not verbatim → Llama finds the right phrase by context.
    """
    cap_lower = caption.lower()
    wp_lower  = wrong_phrase.lower()

    if wp_lower in cap_lower:
        # Fast path — but find the occurrence NEAREST to the subject/object
        # to avoid replacing the wrong instance in multi-entity captions.
        # e.g. "man holding child ... man holding bicycle" — fix (man, bicycle)
        # by finding the "holding" closest to "bicycle".
        occurrences = []
        start = 0
        while True:
            idx = cap_lower.find(wp_lower, start)
            if idx == -1:
                break
            occurrences.append(idx)
            start = idx + 1

        if len(occurrences) == 1:
            idx = occurrences[0]
        else:
            # Score each occurrence by proximity to subj + obj mentions
            subj_idx = cap_lower.find(_core_noun(subj)) if subj else -1
            obj_idx  = cap_lower.find(_core_noun(obj_)) if obj_ else -1
            def _proximity(i):
                d = 0
                if subj_idx >= 0:
                    d += abs(i - subj_idx)
                if obj_idx >= 0:
                    d += abs(i - obj_idx)
                return d
            idx = min(occurrences, key=_proximity)

        return caption[:idx] + correct_phrase + caption[idx + len(wrong_phrase):]

    # Llama path: relation verb not verbatim — describe by entity context.
    # Ratio guard is looser here (1.25) because replacing a short verb with
    # a multi-word phrase legitimately lengthens the caption.
    raw = llm_call([{"role": "user", "content": TRIPLE_CORRECT_PROMPT.format(
        caption=caption,
        subj=subj or wrong_phrase,
        obj=obj_ or correct_phrase,
        wrong_phrase=wrong_phrase,
        correct_phrase=correct_phrase,
    )}], max_tokens=int(len(caption.split()) * 2.5))

    if raw:
        raw = raw.strip().strip('"').strip("'")
        ratio = len(raw) / max(len(caption), 1)
        # Allow up to 25% growth (multi-word replacements) but reject compression
        if 0.85 <= ratio <= 1.25:
            return raw

    return caption  # fall back to original if correction fails


# ── Main v2 correction function ───────────────────────────────────────────

def _correct_long_caption_v2(img_id, caption, kb, pil_image, cross_captions=None):
    """
    Per-triple verification correction for detailed captions (>= 30 words).

    Architecture:
      1. Llama extracts (subject, relation, object, type) triples from caption
      2. Consensus pre-filter: if ALL cross-captioners confirm the claim, skip VQA
      3. SPATIAL triples  → verified by deterministic bbox geometry
         ACTION triples   → verified by 3-question majority-vote crop VQA (Maverick)
         ATTRIBUTE triples→ verified by 3-question majority-vote crop VQA
      4. Confirmed-incorrect triples corrected one at a time via Llama
      5. Spatial addendum appended

    Key properties:
      - Consensus filter (free, zero API cost): skip VQA for claims confirmed by
        both BLIP-2 and LLaVA/Qwen → reduces false positives on strong captioners
      - Multi-question voting (3 paraphrases, majority): false-positive rate drops
        from ~25% (single call) to ~7% per triple
    """
    if pil_image is None:
        return {
            "caption": caption, "corrected": caption,
            "errors": [], "missing": [], "edit_rate": 0,
            "status": "unchanged", "mode": "correct_v2",
        }

    # Step 1: extract triples
    triples = _extract_triples(caption)
    if not triples:
        # Fallback: addendum only if triple extraction failed
        print(f"    [{img_id}] triple extraction returned 0 triples — addendum only")
        corrected, n_addendum = _build_spatial_addendum(caption, kb)
        return {
            "caption": caption, "corrected": corrected,
            "errors": [], "missing": [], "edit_rate": levenshtein(caption, corrected) / max(len(caption), 1),
            "status": "modified" if corrected != caption else "unchanged",
            "mode": "correct_v2", "n_triples": 0, "n_addendum": n_addendum,
        }

    spatial_facts  = kb.get("spatial_facts", [])
    geo_contras    = _check_spatial_contradictions(caption, spatial_facts)
    geo_set        = {c.lower() for c in geo_contras}

    errors       = []   # confirmed hallucinations
    all_checks   = []   # full audit trail

    for t in triples:
        subj = t["subject"].strip()
        rel  = t["relation"].strip()
        obj  = t["object"].strip()
        typ  = t["type"].upper()
        claim_str = f"{subj} {rel} {obj}"

        if typ == "SPATIAL":
            # Deterministic: check geometry
            kb_triples = _parse_spatial_facts(spatial_facts)
            verdict, confidence, reason = "UNKNOWN", "LOW", "no geometry available"
            for kb_s, kb_r, kb_o in kb_triples:
                if _entity_matches(subj, kb_s) and _entity_matches(obj, kb_o):
                    if kb_r == rel:
                        verdict, confidence, reason = "CORRECT", "HIGH", "geometry confirms"
                        break
                    opp = SPATIAL_OPPOSITES.get(rel)
                    if opp and kb_r == opp:
                        verdict, confidence, reason = "INCORRECT", "HIGH", f"geometry shows {kb_r}"
                        break
            # Also check pre-computed geometry contradictions
            if any(claim_str.lower() in g or
                   (subj.lower() in g and rel.lower() in g) for g in geo_set):
                verdict, confidence, reason = "INCORRECT", "HIGH", "deterministic geometry contradiction"
            # VQA fallback for SPATIAL UNKNOWN:
            # (a) If the object isn't detected by GDino at all → existence VQA
            # (b) If both bboxes exist but no spatial match → crop VQA
            if verdict == "UNKNOWN":
                obj_box_check = _find_best_bbox(obj, kb)
                if obj_box_check is None:
                    # Object not detected — ask if it even exists in the image
                    exists = _check_entity_exists_vqa(obj, pil_image)
                    if exists is False:
                        verdict    = "INCORRECT"
                        confidence = "MEDIUM"
                        reason     = f"object '{obj}' not detected by GDino; VQA confirms absence"
                    # else: exists=True or None → leave UNKNOWN, caption likely right
                else:
                    # Both entities detectable but no geometry rule matched
                    # Try VQA on the spatial claim (2 questions, conservative)
                    vqa_result = _verify_action_triple(subj, rel, obj, kb, pil_image, n_questions=2)
                    if vqa_result is False:
                        verdict    = "INCORRECT"
                        confidence = "MEDIUM"
                        reason     = "spatial VQA fallback rejected"
                    elif vqa_result is True:
                        verdict    = "CORRECT"
                        confidence = "MEDIUM"
                        reason     = "spatial VQA fallback confirmed"

            all_checks.append({"claim": claim_str, "type": typ,
                                "verdict": verdict, "confidence": confidence, "reason": reason})
            if verdict == "INCORRECT":
                errors.append({"claim": claim_str, "subject": subj,
                               "relation": rel, "object": obj,
                               "reason": reason, "confidence": confidence,
                               "type": "SPATIAL"})

        else:  # ACTION or ATTRIBUTE
            # Layer 1: action geometry pre-filter (Tier 1 + Tier 2)
            # Tier 1 (mounting, containment, adjacency): bbox-only
            # Tier 2 (grasping, consuming): requires ViTPose keypoints
            # If geometric prerequisite is VIOLATED → confirmed hallucination,
            # skip VQA entirely (saves an API call + higher precision).
            geo_family = _classify_action_family(rel)
            geo_prereq = None
            if geo_family:
                s_box = _find_best_bbox(subj, kb)
                o_box = _find_best_bbox(obj, kb)
                if s_box and o_box:
                    # For Tier 2: get keypoints from the PERSON entity.
                    # Subject isn't always the person ("cup held by man" →
                    # subject=cup). Check which entity is a person/animal.
                    kp = None
                    family_spec = ACTION_GEOMETRY_TAXONOMY.get(geo_family, {})
                    if family_spec.get("needs_keypoints"):
                        _PERSON_WORDS = {"person", "man", "woman", "boy", "girl",
                                         "child", "kid", "baby", "player", "rider",
                                         "worker", "people", "dog", "cat", "horse"}
                        subj_lower = _core_noun(subj)
                        obj_lower  = _core_noun(obj)
                        if subj_lower in _PERSON_WORDS:
                            kp = _get_person_keypoints(pil_image, s_box)
                        elif obj_lower in _PERSON_WORDS:
                            kp = _get_person_keypoints(pil_image, o_box)
                            # Swap perspective: now "person" keypoints are
                            # relative to what was the object bbox, but we
                            # still check wrist/nose vs the OTHER entity's bbox.
                            # So swap s_box/o_box for the geometry check.
                            s_box, o_box = o_box, s_box

                    geo_prereq = _check_action_geometry(
                        geo_family, s_box, o_box, keypoints=kp
                    )
                    if geo_prereq is False:
                        tier = "Tier2-keypoint" if kp else "Tier1-bbox"
                        print(f"    [{img_id}] geometry flag ({geo_family}/{tier}): '{claim_str}' — confirming with VQA")

            # Layer 1.5: cross-captioner consensus pre-filter (zero API cost)
            # If both other captioners also describe "subj rel obj", the claim
            # is almost certainly correct — skip VQA to avoid false positives.
            if cross_captions and _consensus_confirms_triple(subj, rel, obj, cross_captions):
                verdict, confidence, reason = "CORRECT", "HIGH", "cross-captioner consensus"
                all_checks.append({"claim": claim_str, "type": typ,
                                    "verdict": verdict, "confidence": confidence, "reason": reason})
                continue  # no VQA needed

            # Layer 2: crop-based VQA with multi-question voting (Maverick)
            # Runs 3 paraphrased YES/NO questions, majority vote.
            # Geometry FAIL is evidence, not a final verdict — VQA must confirm.
            # IMPORTANT: VQA gets no geometry hints — independence required
            # so that geometry+VQA agreement is a genuinely strong signal.
            verified = _verify_action_triple(subj, rel, obj, kb, pil_image, n_questions=3)
            print(f"    [{img_id}] {typ} triple ({subj!r},{rel!r},{obj!r}) "
                  f"geo={geo_prereq} vqa={verified}")

            if verified is True:
                verdict, confidence, reason = "CORRECT", "MEDIUM", "crop VQA confirmed"
            elif verified is False:
                if geo_prereq is False:
                    # Both geometry AND VQA agree it's wrong — high confidence
                    tier = "Tier2-keypoint" if kp else "Tier1-bbox"
                    verdict    = "INCORRECT"
                    confidence = "HIGH"
                    reason     = f"geometry ({geo_family}/{tier}) + VQA both FAILED"
                else:
                    # VQA alone flagged it — medium confidence
                    verdict    = "INCORRECT"
                    confidence = "MEDIUM"
                    reason     = "crop VQA rejected"
                errors.append({"claim": claim_str, "subject": subj,
                               "relation": rel, "object": obj,
                               "reason": reason, "confidence": confidence,
                               "type": typ})
            else:
                if geo_prereq is False:
                    # Geometry said FAIL but VQA is uncertain/says OK
                    # → geometry was likely wrong (perspective, occlusion, etc.)
                    verdict    = "UNKNOWN"
                    confidence = "LOW"
                    reason     = "geometry flagged but VQA inconclusive — skipping"
                else:
                    verdict, confidence, reason = "UNKNOWN", "LOW", "could not verify (missing bbox or uncertain)"
            all_checks.append({"claim": claim_str, "type": typ,
                                "verdict": verdict, "confidence": confidence, "reason": reason})

    # Step 3: batched correction — Llama fixes ALL confirmed hallucinations in one call.
    # This avoids the surgical-edit garbling problem (dangling "it up", "pink mat it")
    # and handles full-sentence removal (e.g. "The treadmill is located in the center").
    corrected = caption
    applied   = []

    if errors:
        # Build numbered error list with evidence for each hallucination
        error_lines = []
        for i, err in enumerate(errors, 1):
            subj, rel, obj_ = err["subject"], err["relation"], err["object"]
            reason = err["reason"]
            err_type = err.get("type", "ACTION")
            # Give Llama different guidance depending on type
            if err_type == "SPATIAL" and "absence" in reason:
                guidance = f"The {obj_} is NOT present in the image — remove this claim entirely."
            elif err_type == "SPATIAL":
                correct_rel = SPATIAL_OPPOSITES.get(rel)
                guidance = f"The correct spatial relation is '{correct_rel}'" if correct_rel else f"This spatial claim is unverifiable — remove or soften it."
            else:
                guidance = f"Evidence: {reason}. Fix the relationship between {subj!r} and {obj_!r}."
            error_lines.append(f"{i}. "{subj} {rel} {obj_}" — {guidance}")

        error_list_str = "
".join(error_lines)
        prompt = BATCH_CORRECT_PROMPT.format(caption=caption, error_list=error_list_str)

        raw = llm_call(
            [{"role": "user", "content": prompt}],
            max_tokens=int(len(caption.split()) * 2.5 + 50),
        )

        if raw:
            candidate = raw.strip().strip('"').strip("'")
            orig_len  = len(caption)
            cand_len  = len(candidate)
            ratio     = cand_len / max(orig_len, 1)

            # Fluency gate:
            # 1. Length: allow removal of false sentences (down to 60%) but no gross expansion
            # 2. Must differ from original (something changed)
            # 3. Quick garble check: no doubled punctuation, no "it up" after insertion artefacts
            garble_patterns = [
                r"(\w+)\s+it\s+up",     # "pulling a sled it up"
                r"on\s+\w+\s+mat\s+it",  # "on pink mat it"
                r"\w{30,}",                   # extremely long merged word
            ]
            has_garble = any(re.search(p, candidate, re.I) for p in garble_patterns)

            if 0.60 <= ratio <= 1.30 and candidate != caption and not has_garble:
                corrected = candidate
                applied = [{"method": "batch_llm", "n_errors": len(errors),
                             "errors": [e["claim"] for e in errors]}]
                print(f"    [v2] batch correction: {len(errors)} errors fixed "
                      f"(len {orig_len}→{cand_len}, ratio={ratio:.2f})")
            else:
                print(f"    [v2] batch correction REJECTED "
                      f"(ratio={ratio:.2f}, garble={has_garble}, same={candidate==caption})")
                # Fallback: try applying only the HIGHEST confidence errors one at a time
                high_errors = [e for e in errors if e.get("confidence") == "HIGH"]
                for err in high_errors:
                    subj, rel, obj_ = err["subject"], err["relation"], err["object"]
                    err_type = err.get("type", "ACTION")
                    if err_type == "SPATIAL":
                        correct_rel = SPATIAL_OPPOSITES.get(rel, rel)
                        if correct_rel != rel:
                            prev = corrected
                            corrected = _apply_triple_correction(corrected, rel, correct_rel, subj, obj_)
                            if corrected != prev:
                                applied.append({"wrong": rel, "correct": correct_rel,
                                                 "claim": f"{subj} {rel} {obj_}",
                                                 "method": "fallback_spatial"})
                if applied:
                    print(f"    [v2] fallback: applied {len(applied)} HIGH-confidence spatial fix(es)")

    # Step 4: NO spatial addendum for long captions — they are already rich
    # and appending GroundingDINO facts risks overwriting correct information.
    # Addendum is only used for short BLIP-2 captions via _enrich_short_caption.
    n_addendum = 0

    edit_rate = levenshtein(caption, corrected) / max(len(caption), len(corrected), 1)
    return {
        "caption":     caption,
        "corrected":   corrected,
        "errors":      errors,
        "all_checks":  all_checks,
        "applied":     applied,
        "missing":     [],
        "edit_rate":   edit_rate,
        "n_triples":   len(triples),
        "n_addendum":  n_addendum,
        "status":      "modified" if corrected != caption else "unchanged",
        "mode":        "correct_v2",
    }

# -- Unified entry point --------------------------------------------------

def enrich_caption_v3(img_id, caption, kb, pil_image=None, cross_captions=None):
    """
    Plug-and-play RelCheck enrichment/correction.
    Strategy auto-selected from caption word count -- no captioner name needed.

    Args:
        img_id         : image identifier
        caption        : original caption text
        kb             : visual KB dict (hard_facts, spatial_facts,
                          visual_description, detections)
        pil_image      : PIL Image (required for correction mode)
        cross_captions : dict of {captioner_name: caption_text} from other captioners.
                         Used for consensus pre-filter: if all cross-captioners
                         confirm a claim, VQA is skipped (reduces false positives).
    """
    word_count = len(caption.split())
    if word_count < SHORT_CAPTION_THRESHOLD:
        return _enrich_short_caption(img_id, caption, kb)
    else:
        return _correct_long_caption_v2(img_id, caption, kb, pil_image, cross_captions)


# -- Run BLIP-2 enrichment ------------------------------------------------
enriched = {}  # BLIP-2 enrichment not used in this run


In [ ]:
# ============================================================
# Cell 8 -- RelCheck Correction for LLaVA-1.5 + Qwen3-VL-8B (v3)
# ============================================================
# Uses enrich_caption_v3() in CORRECTION mode (auto-selected: >30 words).
# Woodpecker-style: one Maverick call per image (image + spatial KB → corrected caption).
# SCoT: forces Maverick to build a spatial map before checking each relational claim.
#
# Requires: enrich_caption_v3() from Cell 7, images from Cell 2,
#           captions/llava_captions/scout_captions from Cells 3-3b.

LLAVA_ENRICHED_PATH = f"{SAVE_DIR}/llava_enriched.json"
SCOUT_ENRICHED_PATH = f"{SAVE_DIR}/scout_enriched.json"


def run_correction(target_caps, target_name, save_path,
                   other_caps_a, name_a, other_caps_b, name_b):
    """
    Run correction-mode RelCheck on a detailed captioner.
    Cross-captioner consensus uses the other two captioners.
    """
    result = load_checkpoint(save_path) or {}

    if len(result) == len(images):
        print(f"Loaded {len(result)} corrected {target_name} captions from cache.")
        return result

    todo = [img_id for img_id in images if img_id not in result]
    valid_caps = [c for c in target_caps.values() if c]
    avg_w = np.mean([len(c.split()) for c in valid_caps]) if valid_caps else 0
    print(f"Correcting {len(todo)} {target_name} captions "
          f"(from checkpoint: {len(result)}, avg {avg_w:.0f} words → CORRECTION mode)...")

    for idx, img_id in enumerate(todo):
        cap = target_caps.get(img_id, "")
        if not cap:
            result[img_id] = {
                "caption": "", "corrected": "", "errors": [], "missing": [],
                "edit_rate": 0, "status": "skipped", "time_s": 0, "mode": "correct",
            }
            continue

        # Build cross-captioner captions for consensus check
        cross = {}
        if other_caps_a.get(img_id): cross[name_a] = other_caps_a[img_id]
        if other_caps_b.get(img_id): cross[name_b] = other_caps_b[img_id]

        t0 = time.time()
        result[img_id] = enrich_caption_v3(
            img_id      = img_id,
            caption     = cap,
            kb          = knowledge_bases[img_id],
            pil_image   = images.get(img_id),   # PIL Image for crop VQA
            cross_captions = cross,              # consensus filter
        )
        result[img_id]["time_s"] = round(time.time() - t0, 1)

        if (idx + 1) % 5 == 0 or result[img_id]["status"] == "modified":
            r         = result[img_id]
            errs      = r.get("errors", [])
            checks    = r.get("all_checks", [])
            n_triples = r.get("n_triples", 0)
            n_geo     = sum(1 for e in errs if "geometry" in e.get("reason",""))
            n_correct = sum(1 for c in checks if c.get("verdict") == "CORRECT")
            n_unknown = sum(1 for c in checks if c.get("verdict") == "UNKNOWN")
            n_addendum= r.get("n_addendum", 0)
            print(f"  [{idx+1}/{len(todo)}] {img_id} → {r['status']} "
                  f"edit={r['edit_rate']:.3f} triples={n_triples} "
                  f"errs={len(errs)}(geo={n_geo}) correct={n_correct} "
                  f"unknown={n_unknown} addendum={n_addendum} ({r['time_s']}s)")
            save_checkpoint(result, save_path)

    save_checkpoint(result, save_path)
    return result


# ── Correct LLaVA-1.5 captions ───────────────────────────────────────────
llava_enriched = run_correction(
    target_caps  = llava_captions,
    target_name  = "LLaVA-1.5",
    save_path    = LLAVA_ENRICHED_PATH,
    other_caps_a = scout_captions, name_a = "qwen",
    other_caps_b = {},             name_b = "",
)

# ── Correct Qwen3-VL-8B captions ─────────────────────────────────────────
scout_enriched = run_correction(
    target_caps  = scout_captions,
    target_name  = "Qwen3-VL-8B",
    save_path    = SCOUT_ENRICHED_PATH,
    other_caps_a = llava_captions, name_a = "llava",
    other_caps_b = {},             name_b = "",
)

# ── Summary: compare word counts before vs after ──────────────────────────
print(f"\n{'='*75}")
print(f"MULTI-MODEL CORRECTION SUMMARY (v3)")
print(f"{'='*75}")
print(f"{'Captioner':<18} {'Mode':<10} {'Modified':>10} {'Avg Edit':>10} {'Words before→after':>20}")
print("-" * 75)

for name, enr_data, orig_caps in [
    ("LLaVA-1.5",   llava_enriched, llava_captions),
    ("Qwen3-VL-8B", scout_enriched, scout_captions),
]:
    n = len(enr_data)
    if n == 0: continue
    n_mod    = sum(1 for v in enr_data.values() if v["status"] == "modified")
    avg_edit = np.mean([v["edit_rate"] for v in enr_data.values()])
    mode     = next(iter(enr_data.values())).get("mode", "?")
    before   = np.mean([len(c.split()) for c in orig_caps.values() if c])
    after    = np.mean([len(v["corrected"].split())
                        for v in enr_data.values() if v.get("corrected")])
    n_addendum = sum(v.get("n_addendum", 0) for v in enr_data.values())
    addendum_note = f"  (+{n_addendum} spatial facts appended)" if n_addendum > 0 else ""
    print(f"{name:<18} {mode:<10} {n_mod:>5}/{n:<5} ({n_mod/n:.0%})"
          f"  {avg_edit:>8.3f}    {before:.0f} → {after:.0f} words{addendum_note}")

print()
print("Note: LLaVA/Qwen use surgical edits only (no addendum) — word count should stay close to original.")
print("BLIP-2 word count increase = enrichment adding missing facts (expected).")

# ── v3 verification diagnostics ──────────────────────────────────────────
print(f"\n{'='*75}")
print("VERIFICATION DIAGNOSTICS (LLaVA + Qwen correction mode, v3 pipeline)")
print(f"{'='*75}")
for name, enr_data in [("LLaVA-1.5", llava_enriched), ("Qwen3-VL-8B", scout_enriched)]:
    all_errors = [e for v in enr_data.values() for e in v.get("errors", [])]
    if not all_errors:
        print(f"  {name}: no errors flagged"); continue
    n_geo  = sum(1 for e in all_errors if "geometry" in e.get("reason","").lower())
    n_vqa  = sum(1 for e in all_errors if "vqa"      in e.get("reason","").lower()
                                       or "action"   in e.get("reason","").lower()
                                       or "attribute" in e.get("reason","").lower())
    n_other = len(all_errors) - n_geo - n_vqa
    n_corrected = sum(1 for v in enr_data.values() if v.get("corrected"))
    conf_dist   = Counter(e.get("confidence","?") for e in all_errors)
    print(f"  {name}:")
    print(f"    Total flagged errors : {len(all_errors)}")
    print(f"    Source breakdown     : Geometry={n_geo}, VQA={n_vqa}, Other={n_other}")
    print(f"    Images corrected     : {n_corrected}")
    print(f"    Confidence dist      : {dict(conf_dist)}")
    print()


In [ ]:
# ============================================================
# Cell 8b — Correction Inspector (Interactive HTML Viewer)
# ============================================================
# For every image where RelCheck made a correction (or flagged triples),
# shows: image + GroundingDINO bboxes, original caption, per-triple
# verification verdicts, and the corrected caption side-by-side.
# Open the saved HTML file in any browser.

INSPECT_HTML = f"{SAVE_DIR}/correction_inspector.html"

def _encode_pil(pil_img, max_w=480):
    """Encode PIL image to base64 JPEG, resized to max_w."""
    from PIL import ImageDraw
    import io, base64
    ratio = max_w / pil_img.width
    h = int(pil_img.height * ratio)
    img = pil_img.resize((max_w, h)).convert("RGB")
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=80)
    return base64.b64encode(buf.getvalue()).decode()

def _draw_kb_boxes(pil_img, kb, max_boxes=10):
    """Draw GroundingDINO bboxes on image, return base64."""
    from PIL import ImageDraw
    img = pil_img.copy().convert("RGB")
    draw = ImageDraw.Draw(img)
    W, H = img.size
    colors = ["#e63946","#2a9d8f","#e9c46a","#264653","#f4a261",
              "#9b72cf","#2196f3","#ff9800","#4caf50","#e91e63"]
    for i, (label, score, bbox) in enumerate(kb.get("detections", [])[:max_boxes]):
        x1,y1,x2,y2 = bbox[0]*W, bbox[1]*H, bbox[2]*W, bbox[3]*H
        c = colors[i % len(colors)]
        draw.rectangle([x1,y1,x2,y2], outline=c, width=3)
        txt = f"{label}"
        draw.rectangle([x1, max(0,y1-16), x1+len(txt)*7+4, y1], fill=c)
        draw.text((x1+2, max(0,y1-15)), txt, fill="white")
    from io import BytesIO
    import base64
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=80)
    return base64.b64encode(buf.getvalue()).decode()

# ── Collect inspection data ──
inspect_items = []

for captioner_name, enr_data, orig_caps in [
    ("LLaVA-1.5",   llava_enriched, llava_captions),
    ("Qwen3-VL-8B", scout_enriched, scout_captions),
]:
    for img_id, r in enr_data.items():
        checks   = r.get("all_checks", [])
        errors   = r.get("errors", [])
        if not checks:
            continue   # triple extraction returned nothing — skip
        orig_cap = orig_caps.get(img_id, "")
        corr_cap = r.get("corrected", orig_cap)
        kb       = knowledge_bases.get(img_id, {})
        pil_img  = images.get(img_id)
        if pil_img is None:
            continue
        inspect_items.append({
            "captioner":  captioner_name,
            "img_id":     img_id,
            "status":     r.get("status", "unchanged"),
            "mode":       r.get("mode", ""),
            "orig_cap":   orig_cap,
            "corr_cap":   corr_cap,
            "edit_rate":  r.get("edit_rate", 0),
            "checks":     checks,
            "errors":     errors,
            "n_triples":  r.get("n_triples", len(checks)),
            "img_b64":    _encode_pil(pil_img),
            "bbox_b64":   _draw_kb_boxes(pil_img, kb),
            "spatial_facts": kb.get("spatial_facts", [])[:6],
        })

# Sort: modified first, then by captioner
inspect_items.sort(key=lambda x: (0 if x["status"]=="modified" else 1, x["captioner"]))

print(f"Inspection items: {len(inspect_items)} images with extracted triples")
print(f"  Modified : {sum(1 for x in inspect_items if x['status']=='modified')}")
print(f"  Unchanged: {sum(1 for x in inspect_items if x['status']=='unchanged')}")

# ── Build HTML ──
CSS = """
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:-apple-system,BlinkMacSystemFont,sans-serif;background:#f0f2f5;color:#1a1a2e}
.header{background:#1a1a2e;color:white;padding:14px 24px;position:sticky;top:0;z-index:100;
  display:flex;justify-content:space-between;align-items:center}
.header h1{font-size:17px;font-weight:600}
.nav{display:flex;align-items:center;gap:10px}
.nav button{padding:8px 18px;border:none;border-radius:6px;font-size:13px;font-weight:600;cursor:pointer}
.btn-prev{background:#333;color:white}.btn-next{background:#3b82f6;color:white}
.btn-prev:disabled,.btn-next:disabled{opacity:.35;cursor:not-allowed}
.counter{color:#aaa;font-size:13px}
.jump{width:54px;padding:5px;border:1px solid #555;border-radius:4px;
  background:#222;color:white;text-align:center;font-size:13px}
.container{max-width:1200px;margin:20px auto;padding:0 16px}
.card{background:white;border-radius:12px;box-shadow:0 2px 10px rgba(0,0,0,.08);overflow:hidden;margin-bottom:0}
.card-header{padding:14px 20px;display:flex;justify-content:space-between;align-items:center;
  border-bottom:1px solid #eee}
.tag{padding:4px 10px;border-radius:20px;font-size:12px;font-weight:700;letter-spacing:.5px}
.tag-modified{background:#d1fae5;color:#065f46}
.tag-unchanged{background:#f3f4f6;color:#6b7280}
.tag-skipped{background:#fef3c7;color:#92400e}
.captioner-badge{padding:4px 10px;border-radius:20px;font-size:12px;font-weight:600;
  background:#ede9fe;color:#5b21b6}
.body{display:grid;grid-template-columns:1fr 1fr;gap:0}
.img-col{padding:16px;border-right:1px solid #eee}
.img-toggle{display:flex;gap:6px;margin-bottom:10px}
.img-toggle button{padding:5px 12px;border:1px solid #ddd;border-radius:6px;
  font-size:12px;cursor:pointer;background:white}
.img-toggle button.active{background:#1a1a2e;color:white;border-color:#1a1a2e}
.img-col img{width:100%;border-radius:8px;display:block}
.spatial-facts{margin-top:10px;font-size:12px;color:#555}
.spatial-facts h4{font-size:11px;font-weight:700;text-transform:uppercase;
  letter-spacing:.5px;color:#888;margin-bottom:4px}
.sf-item{padding:3px 0;border-bottom:1px solid #f5f5f5}
.info-col{padding:16px;display:flex;flex-direction:column;gap:14px}
.section-label{font-size:11px;font-weight:700;text-transform:uppercase;
  letter-spacing:.5px;color:#888;margin-bottom:5px}
.caption-box{padding:10px 12px;border-radius:8px;font-size:13px;line-height:1.6}
.cap-orig{background:#fef3c7;border:1px solid #fbbf24}
.cap-corr{background:#d1fae5;border:1px solid #34d399}
.cap-same{background:#f3f4f6;border:1px solid #e5e7eb;color:#6b7280}
.edit-rate{font-size:11px;color:#888;margin-top:4px}
.triple-list{display:flex;flex-direction:column;gap:6px}
.triple{padding:9px 12px;border-radius:8px;font-size:13px;
  display:flex;align-items:center;justify-content:space-between;gap:8px}
.triple-text{flex:1}
.triple-text .subj{color:#1d4ed8;font-weight:600}
.triple-text .rel{color:#6b7280;font-style:italic;margin:0 4px}
.triple-text .obj{color:#1d4ed8;font-weight:600}
.triple-text .typ{font-size:10px;color:#9ca3af;margin-left:6px}
.verdict{font-size:11px;font-weight:700;padding:3px 8px;border-radius:10px;white-space:nowrap}
.v-correct{background:#d1fae5;color:#065f46}
.v-incorrect{background:#fee2e2;color:#991b1b}
.v-unknown{background:#f3f4f6;color:#6b7280}
.confidence{font-size:10px;color:#9ca3af;margin-left:4px}
.t-correct{background:#f0fdf4;border:1px solid #bbf7d0}
.t-incorrect{background:#fff1f2;border:1px solid #fecdd3}
.t-unknown{background:#f9fafb;border:1px solid #e5e7eb}
.no-triples{color:#9ca3af;font-size:13px;font-style:italic;padding:10px 0}
.filter-bar{display:flex;gap:8px;margin:16px 0;flex-wrap:wrap}
.filter-btn{padding:6px 14px;border:1px solid #ddd;border-radius:20px;
  font-size:12px;cursor:pointer;background:white;font-weight:500}
.filter-btn.active{background:#1a1a2e;color:white;border-color:#1a1a2e}
.stats-bar{display:flex;gap:16px;font-size:12px;color:#6b7280;margin-top:4px}
"""

JS = r"""
let allData = DATA;
let filtered = [...allData];
let idx = 0;
let imgMode = 'bbox';  // 'orig' or 'bbox'

function applyFilter(f) {
  document.querySelectorAll('.filter-btn').forEach(b => b.classList.remove('active'));
  event.target.classList.add('active');
  if (f === 'all') filtered = [...allData];
  else if (f === 'modified') filtered = allData.filter(x => x.status === 'modified');
  else if (f === 'hallucinated') filtered = allData.filter(x => x.errors.length > 0);
  else if (f === 'llava') filtered = allData.filter(x => x.captioner.startsWith('LLaVA'));
  else if (f === 'qwen') filtered = allData.filter(x => x.captioner.startsWith('Qwen'));
  idx = 0;
  render();
}

function render() {
  if (!filtered.length) {
    document.getElementById('card').innerHTML = '<div style="padding:40px;text-align:center;color:#888">No items match filter.</div>';
    return;
  }
  const d = filtered[idx];
  document.getElementById('counterText').textContent = `${idx+1} / ${filtered.length}`;
  document.getElementById('jumpInput').value = idx+1;
  document.getElementById('btnPrev').disabled = idx === 0;
  document.getElementById('btnNext').disabled = idx === filtered.length - 1;

  // Header
  const tagCls = d.status === 'modified' ? 'tag-modified' : d.status === 'skipped' ? 'tag-skipped' : 'tag-unchanged';
  document.getElementById('cardStatus').className = 'tag ' + tagCls;
  document.getElementById('cardStatus').textContent = d.status.toUpperCase();
  document.getElementById('cardCaptioner').textContent = d.captioner;
  document.getElementById('cardImgId').textContent = d.img_id.slice(0, 30);

  // Image
  const imgSrc = imgMode === 'bbox' ? 'data:image/jpeg;base64,' + d.bbox_b64
                                    : 'data:image/jpeg;base64,' + d.img_b64;
  document.getElementById('mainImg').src = imgSrc;

  // Spatial facts
  const sfEl = document.getElementById('spatialFacts');
  if (d.spatial_facts.length) {
    sfEl.style.display = 'block';
    sfEl.querySelector('.sf-body').innerHTML =
      d.spatial_facts.map(f => `<div class="sf-item">${f}</div>`).join('');
  } else {
    sfEl.style.display = 'none';
  }

  // Captions
  const origEl = document.getElementById('captionOrig');
  origEl.textContent = d.orig_cap;
  const corrEl = document.getElementById('captionCorr');
  const corrSec = document.getElementById('corrSection');
  if (d.status === 'modified' && d.corr_cap !== d.orig_cap) {
    corrEl.textContent = d.corr_cap;
    corrEl.className = 'caption-box cap-corr';
    corrSec.style.display = 'block';
    document.getElementById('editRate').textContent = `Edit rate: ${(d.edit_rate*100).toFixed(1)}%`;
  } else {
    corrSec.style.display = 'none';
  }

  // Triples
  const tripleEl = document.getElementById('tripleList');
  if (!d.checks.length) {
    tripleEl.innerHTML = '<div class="no-triples">No triples extracted from this caption.</div>';
    return;
  }
  tripleEl.innerHTML = d.checks.map(c => {
    const vCls = c.verdict === 'CORRECT' ? 'v-correct t-correct'
               : c.verdict === 'INCORRECT' ? 'v-incorrect t-incorrect' : 'v-unknown t-unknown';
    const vLabel = c.verdict === 'CORRECT' ? 'CORRECT'
                 : c.verdict === 'INCORRECT' ? 'HALLUCINATED' : 'UNKNOWN';
    const vBadge = c.verdict === 'CORRECT' ? 'v-correct'
                 : c.verdict === 'INCORRECT' ? 'v-incorrect' : 'v-unknown';
    const reason = c.reason ? `<div style="font-size:11px;color:#9ca3af;margin-top:3px">${c.reason}</div>` : '';
    return `<div class="triple ${vCls}">
      <div class="triple-text">
        <span class="subj">${c.claim.split(' ')[0]||''}</span>
        <span class="rel">${c.claim.split(' ').slice(1,-1).join(' ')||''}</span>
        <span class="obj">${c.claim.split(' ').slice(-1)[0]||''}</span>
        <span class="typ">[${c.type||'?'}]</span>
        ${reason}
      </div>
      <div>
        <span class="verdict ${vBadge}">${vLabel}</span>
        <span class="confidence">${c.confidence||''}</span>
      </div>
    </div>`;
  }).join('');

  // Stats
  const n_hall = d.errors.length;
  const n_corr = d.checks.filter(c=>c.verdict==='CORRECT').length;
  const n_unk  = d.checks.filter(c=>c.verdict==='UNKNOWN').length;
  document.getElementById('tripleStats').textContent =
    `${d.checks.length} triples — ${n_corr} correct, ${n_hall} hallucinated, ${n_unk} unknown`;
}

function go(delta) {
  idx = Math.max(0, Math.min(filtered.length-1, idx+delta));
  render();
  window.scrollTo(0,0);
}
function jumpTo(v) {
  const n = parseInt(v);
  if (n>=1 && n<=filtered.length) { idx=n-1; render(); window.scrollTo(0,0); }
}
function toggleImg(mode) {
  imgMode = mode;
  document.querySelectorAll('.img-toggle button').forEach(b=>b.classList.remove('active'));
  document.getElementById('btn_'+mode).classList.add('active');
  render();
}

document.addEventListener('keydown', e => {
  if (e.target.tagName==='INPUT') return;
  if (e.key==='ArrowLeft') go(-1);
  if (e.key==='ArrowRight') go(1);
});
render();
"""

BODY = """
<div class="header">
  <h1>RelCheck Correction Inspector</h1>
  <div class="nav">
    <button class="btn-prev" id="btnPrev" onclick="go(-1)">◀ Prev</button>
    <span class="counter">
      <input class="jump" id="jumpInput" onchange="jumpTo(this.value)">
      <span id="counterText"></span>
    </span>
    <button class="btn-next" id="btnNext" onclick="go(1)">Next ▶</button>
  </div>
</div>
<div class="container">
  <div class="filter-bar">
    <button class="filter-btn active" onclick="applyFilter('all')">All</button>
    <button class="filter-btn" onclick="applyFilter('modified')">Modified only</button>
    <button class="filter-btn" onclick="applyFilter('hallucinated')">Hallucinated triples</button>
    <button class="filter-btn" onclick="applyFilter('llava')">LLaVA-1.5</button>
    <button class="filter-btn" onclick="applyFilter('qwen')">Qwen3-VL-8B</button>
  </div>
  <div class="card" id="card">
    <div class="card-header">
      <div style="display:flex;align-items:center;gap:10px">
        <span class="tag" id="cardStatus"></span>
        <span class="captioner-badge" id="cardCaptioner"></span>
        <span style="font-size:12px;color:#888" id="cardImgId"></span>
      </div>
    </div>
    <div class="body">
      <div class="img-col">
        <div class="img-toggle">
          <button id="btn_bbox" class="active" onclick="toggleImg('bbox')">+ Detections</button>
          <button id="btn_orig" onclick="toggleImg('orig')">Original</button>
        </div>
        <img id="mainImg" src="">
        <div id="spatialFacts" class="spatial-facts" style="display:none">
          <h4>KB Spatial Facts</h4>
          <div class="sf-body"></div>
        </div>
      </div>
      <div class="info-col">
        <div>
          <div class="section-label">Original Caption</div>
          <div class="caption-box cap-orig" id="captionOrig"></div>
        </div>
        <div id="corrSection">
          <div class="section-label">Corrected Caption</div>
          <div class="caption-box" id="captionCorr"></div>
          <div class="edit-rate" id="editRate"></div>
        </div>
        <div>
          <div class="section-label">Extracted Triples &amp; Verdicts</div>
          <div id="tripleStats" class="stats-bar" style="margin-bottom:8px"></div>
          <div class="triple-list" id="tripleList"></div>
        </div>
      </div>
    </div>
  </div>
</div>
"""

# Serialize data (no b64 in JSON — pass separately)
import json as _json
js_items = []
for item in inspect_items:
    js_items.append({
        "captioner":     item["captioner"],
        "img_id":        item["img_id"],
        "status":        item["status"],
        "orig_cap":      item["orig_cap"],
        "corr_cap":      item["corr_cap"],
        "edit_rate":     item["edit_rate"],
        "checks":        item["checks"],
        "errors":        item["errors"],
        "n_triples":     item["n_triples"],
        "spatial_facts": item["spatial_facts"],
        "img_b64":       item["img_b64"],
        "bbox_b64":      item["bbox_b64"],
    })

html_parts = [
    '<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">',
    '<title>RelCheck Inspector</title>',
    f'<style>{CSS}</style></head><body>',
    BODY,
    '<script>',
    f'const DATA = {_json.dumps(js_items, ensure_ascii=False)};',
    JS,
    '</script></body></html>',
]
html = '\n'.join(html_parts)
with open(INSPECT_HTML, 'w') as f:
    f.write(html)

print(f"\nInspector saved: {INSPECT_HTML}")
print(f"  {len(inspect_items)} images | {sum(1 for x in inspect_items if x['status']=='modified')} modified")
print("  Open in browser. Arrow keys to navigate. Filter buttons at top.")


In [ ]:
# ============================================================
# Cell 6 — Ablation Correctors
# ============================================================
# Runs post-hoc on saved captions + KBs — no image re-processing.
#
# TABLE 3 (KB Source Ablation) — varies what evidence we give:
#   b3     : No KB at all
#   kb_obj : Objects only (GDINO counts, no geometry, no VLM)
#   kb_geom: Objects + geometric spatial relations (no VLM)
#   full   : Full KB (= RelCheck v2, already in Cell 5)
#
# TABLE 4 (Correction Method Ablation) — varies how we correct:
#   b2      : Self-refinement (Llama refines without any visual evidence)
#   b3      : Blind correction (same as Table 3 b3, shared)
#   kb_dump : Full KB as raw unstructured text, free rewrite
#   full    : Structured analysis → targeted fix (= RelCheck v2)
#
# Checkpoint: loads from Drive if already computed.

ABLATIONS_PATH = f"{SAVE_DIR}/ablations.json"

# ── Prompt templates ──────────────────────────────────────

B2_PROMPT = """This image caption may contain inaccuracies about object relationships.
Please review it carefully and rewrite it to be more accurate and complete.
Keep the same style and length. Only fix what seems wrong.

Caption: "{caption}"

Rewritten caption (one paragraph, 3-5 sentences):"""

B3_PROMPT = """This image caption may contain relational hallucinations (incorrect claims about
how objects relate to each other). Please rewrite it to fix any errors.

Caption: "{caption}"

Corrected caption (3-5 sentences, same style):"""

KB_OBJ_PROMPT = """Improve this image caption using the list of detected objects.

CAPTION: "{caption}"

DETECTED OBJECTS:
{obj_facts}

Rewrite the caption to fix any errors based on the detected objects and add important missing objects.
Keep it to 3-5 natural sentences.

Improved caption:"""

KB_GEOM_PROMPT = """Improve this image caption using detected objects and their spatial positions.

CAPTION: "{caption}"

DETECTED OBJECTS:
{obj_facts}

SPATIAL RELATIONSHIPS (from bounding boxes — deterministic):
{spatial_facts}

Rewrite the caption to fix spatial errors and add missing spatial details.
Keep it to 3-5 natural sentences.

Improved caption:"""

KB_DUMP_PROMPT = """Improve this image caption using all available visual evidence.

CAPTION: "{caption}"

FULL VISUAL KNOWLEDGE BASE:
Objects: {obj_facts}
Spatial: {spatial_facts}
Visual description: {visual_desc}

Rewrite the caption to be as accurate and complete as possible. 3-5 natural sentences.

Improved caption:"""


# ── Length-aware variants for long-caption captioners (LLaVA, Qwen) ──────
# These preserve the original caption length instead of compressing to 3-5 sentences.
# Used ONLY for cross-ablation on LLaVA/Qwen; BLIP-2 ablations still use B3/KB_DUMP above.
B3_LONG_PROMPT = """This image caption may contain relational hallucinations (incorrect claims about
how objects relate to each other). Please fix any errors in place.

Caption: "{caption}"

Rules:
- Fix ONLY incorrect relational claims (spatial positions, actions, attributes).
- Copy every other sentence EXACTLY. Do NOT compress, summarize, or drop sentences.
- The output must be approximately the same length as the input.

Corrected caption (same length and style):"""

KB_DUMP_LONG_PROMPT = """Improve this detailed image caption using visual evidence.

CAPTION: "{caption}"

FULL VISUAL KNOWLEDGE BASE:
Objects: {obj_facts}
Spatial: {spatial_facts}
Visual description: {visual_desc}

Rules:
- Fix relational errors (spatial, action, attribute) supported by the KB.
- Copy every correct sentence EXACTLY. Do NOT compress or summarize.
- The output must be approximately the same length as the input.

Corrected caption (same length):"""


def run_ablation_corrector(prompt_fn, images_dict, captions_dict, kb_dict,
                           variant_name, cached={}):
    """Run a correction variant for all images. Returns {img_id: corrected_caption}."""
    results = dict(cached)  # start from cached
    todo = [img_id for img_id in images_dict if img_id not in results]

    if not todo:
        print(f"  {variant_name}: all {len(results)} cached.")
        return results

    print(f"  {variant_name}: correcting {len(todo)} images...")
    for idx, img_id in enumerate(todo):
        caption = captions_dict[img_id]
        kb = kb_dict[img_id]
        prompt = prompt_fn(caption, kb)

        raw = llm_call([{"role": "user", "content": prompt}], max_tokens=300)
        corrected = caption  # default: unchanged
        if raw:
            raw = raw.strip().strip('"').strip("'")
            if len(raw) >= 10:
                corrected = raw

        results[img_id] = corrected
        if (idx + 1) % 100 == 0:
            print(f"    [{idx+1}/{len(todo)}]")
        time.sleep(0.25)

    return results


# ── Load or compute ablations ──
ablations_raw = load_checkpoint(ABLATIONS_PATH) or {}

# B2: self-refinement
if "b2" not in ablations_raw:
    ablations_raw["b2"] = {}
ablations_raw["b2"] = run_ablation_corrector(
    lambda cap, kb: B2_PROMPT.format(caption=cap),
    images, captions, knowledge_bases, "B2 (self-refine)",
    cached=ablations_raw["b2"],
)
save_checkpoint(ablations_raw, ABLATIONS_PATH)

# B3: blind correction (no KB)
if "b3" not in ablations_raw:
    ablations_raw["b3"] = {}
ablations_raw["b3"] = run_ablation_corrector(
    lambda cap, kb: B3_PROMPT.format(caption=cap),
    images, captions, knowledge_bases, "B3 (blind)",
    cached=ablations_raw["b3"],
)
save_checkpoint(ablations_raw, ABLATIONS_PATH)

# KB-obj: objects only
if "kb_obj" not in ablations_raw:
    ablations_raw["kb_obj"] = {}
ablations_raw["kb_obj"] = run_ablation_corrector(
    lambda cap, kb: KB_OBJ_PROMPT.format(
        caption=cap,
        obj_facts="\n".join(f"- {f}" for f in kb["hard_facts"]) or "- None",
    ),
    images, captions, knowledge_bases, "KB-obj (objects only)",
    cached=ablations_raw["kb_obj"],
)
save_checkpoint(ablations_raw, ABLATIONS_PATH)

# KB-geom: objects + geometry
if "kb_geom" not in ablations_raw:
    ablations_raw["kb_geom"] = {}
ablations_raw["kb_geom"] = run_ablation_corrector(
    lambda cap, kb: KB_GEOM_PROMPT.format(
        caption=cap,
        obj_facts="\n".join(f"- {f}" for f in kb["hard_facts"]) or "- None",
        spatial_facts="\n".join(f"- {f}" for f in kb["spatial_facts"]) or "- None",
    ),
    images, captions, knowledge_bases, "KB-geom (objects+geometry)",
    cached=ablations_raw["kb_geom"],
)
save_checkpoint(ablations_raw, ABLATIONS_PATH)

# KB-dump: full KB, unstructured rewrite
if "kb_dump" not in ablations_raw:
    ablations_raw["kb_dump"] = {}
ablations_raw["kb_dump"] = run_ablation_corrector(
    lambda cap, kb: KB_DUMP_PROMPT.format(
        caption=cap,
        obj_facts="\n".join(f"- {f}" for f in kb["hard_facts"]) or "- None",
        spatial_facts="\n".join(f"- {f}" for f in kb["spatial_facts"]) or "- None",
        visual_desc=kb["visual_description"][:300] or "- None",
    ),
    images, captions, knowledge_bases, "KB-dump (unstructured)",
    cached=ablations_raw["kb_dump"],
)
save_checkpoint(ablations_raw, ABLATIONS_PATH)

print(f"\nAll ablation variants computed. Saved to Drive.")
for variant, data in ablations_raw.items():
    n_changed = sum(1 for img_id in data if data[img_id] != captions.get(img_id, ""))
    print(f"  {variant}: {n_changed}/{len(data)} captions changed")

# ============================================================
# Cross-Model Ablations: B3 + KB-dump on LLaVA and Qwen3-VL-8B
# ============================================================
# Tests whether RelCheck's structured approach beats blind correction
# and unstructured KB dump ACROSS DIFFERENT CAPTIONERS.
# This is critical for the paper — proves the pipeline generalizes.

CROSS_ABLATIONS_PATH = f"{SAVE_DIR}/cross_ablations.json"
cross_ablations = load_checkpoint(CROSS_ABLATIONS_PATH) or {}

# Cross-ablation cache is preserved across runs — no clearing.

# For each captioner, run B3 (blind) and KB-dump (unstructured)
captioner_configs = []

if 'llava_captions' in dir() and llava_captions:
    captioner_configs.append(("llava", llava_captions))
if 'scout_captions' in dir() and scout_captions:
    captioner_configs.append(("scout", scout_captions))  # internally 'scout', displays as Qwen3-VL-8B

for cap_name, cap_dict in captioner_configs:
    # Use length-preserving prompts for LLaVA/Qwen (80-200 word captions).
    # Standard B3/KB_DUMP prompts say "3-5 sentences" which compresses and loses info.

    # B3: blind correction (no KB evidence), length-preserving
    key_b3 = f"{cap_name}_b3"
    if key_b3 not in cross_ablations:
        cross_ablations[key_b3] = {}
    cross_ablations[key_b3] = run_ablation_corrector(
        lambda cap, kb: B3_LONG_PROMPT.format(caption=cap),
        images, cap_dict, knowledge_bases,
        f"{cap_name} B3 (blind, length-preserving)",
        cached=cross_ablations[key_b3],
    )
    save_checkpoint(cross_ablations, CROSS_ABLATIONS_PATH)

    # KB-dump: full KB, unstructured, length-preserving
    key_dump = f"{cap_name}_kb_dump"
    if key_dump not in cross_ablations:
        cross_ablations[key_dump] = {}
    cross_ablations[key_dump] = run_ablation_corrector(
        lambda cap, kb: KB_DUMP_LONG_PROMPT.format(
            caption=cap,
            obj_facts="\n".join(f"- {f}" for f in kb["hard_facts"]) or "- None",
            spatial_facts="\n".join(f"- {f}" for f in kb["spatial_facts"]) or "- None",
            visual_desc=kb["visual_description"][:300] or "- None",
        ),
        images, cap_dict, knowledge_bases,
        f"{cap_name} KB-dump (length-preserving)",
        cached=cross_ablations[key_dump],
    )
    save_checkpoint(cross_ablations, CROSS_ABLATIONS_PATH)

print(f"\nCross-model ablations saved.")
for k, v in cross_ablations.items():
    print(f"  {k}: {len(v)} captions")


In [ ]:
# ============================================================
# Cell 7 — R-POPE Evaluation (LLM-Judge, all methods)
# ============================================================
# Methods evaluated:
#   llava_orig     : Original LLaVA-1.5 captions
#   llava_relcheck : LLaVA-1.5 after RelCheck correction
#   qwen_orig      : Original Qwen3-VL-8B captions
#   qwen_relcheck  : Qwen3-VL-8B after RelCheck correction
# Checkpoint: loads from Drive if already computed.

RPOPE_PATH = f"{SAVE_DIR}/rpope_results.json"

RPOPE_PROMPT = """Based ONLY on this caption, answer the question with 'yes' or 'no'.
Do not use any external knowledge. Only use information stated in the caption.

Caption: "{caption}"
Question: {question}

Answer (yes or no):"""


def rpope_judge(caption, question):
    raw = llm_call(
        [{"role": "user", "content": RPOPE_PROMPT.format(caption=caption, question=question)}],
        max_tokens=5,
    )
    if raw:
        raw = raw.lower()
        if "yes" in raw and "no" not in raw:
            return "yes"
        if "no" in raw:
            return "no"
    return "unknown"


def classify_question(question):
    q = question.lower()
    for v in ["playing","holding","eating","sitting","standing","running","walking",
               "riding","driving","reading","carrying","cooking","swimming","flying",
               "climbing","jumping","drinking","pointing","pulling","pushing","cutting",
               "hugging","leaning","lying","reaching","touching","feeding","surfing",
               "skiing","skating","waving","looking","talking","throwing","catching"]:
        if v in q:
            return "ACTION"
    for p in [" on a "," on the "," on top "," in a "," in the "," inside "," outside ",
               " near "," next to "," behind "," above "," below "," under "," beside ",
               " between "," in front of "," across "," along "," around "]:
        if p in f" {q} ":
            return "SPATIAL"
    for kw in ["wearing","color","colour","red","blue","green","yellow","black","white",
                "brown","large","small","big","tall","short","old","young","made of"]:
        if kw in q:
            return "ATTRIBUTE"
    return "OTHER"


def compute_metrics(pairs, total=None):
    """Compute precision/recall/F1/yes_ratio from list of (pred, gt) tuples."""
    if total is None:
        total = len(pairs)
    tp = sum(1 for p, g in pairs if p == "yes" and g == "yes")
    fp = sum(1 for p, g in pairs if p == "yes" and g == "no")
    fn = sum(1 for p, g in pairs if p == "no" and g == "yes")
    tn = sum(1 for p, g in pairs if p == "no" and g == "no")
    acc = (tp + tn) / total if total else 0
    prec = tp / (tp + fp) if (tp + fp) else 0
    rec = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
    yes_r = (tp + fp) / total if total else 0
    return {"accuracy": acc, "precision": prec, "recall": rec, "f1": f1,
            "yes_ratio": yes_r, "n": total}


# ── Build caption dict per method ──
method_captions = {}

if llava_captions:
    method_captions["llava_orig"] = llava_captions
    method_captions["llava_relcheck"] = {
        img_id: r["corrected"] for img_id, r in llava_enriched.items()
        if r.get("status") != "skipped" and r.get("corrected")
    }

if scout_captions:
    method_captions["qwen_orig"] = scout_captions
    method_captions["qwen_relcheck"] = {
        img_id: r["corrected"] for img_id, r in scout_enriched.items()
        if r.get("status") != "skipped" and r.get("corrected")
    }

print(f"Methods: {list(method_captions.keys())}")
for m, d in method_captions.items():
    print(f"  {m}: {len(d)} captions")


# ── Load or compute R-POPE ──
rpope_raw = load_checkpoint(RPOPE_PATH) or {}

# Determine which methods still have uncached questions
methods_to_eval = []
for _m in method_captions:
    _need_run = False
    for _img_id in list(images.keys())[:20]:
        for _entry in img_to_questions.get(_img_id, [])[:10]:
            if f"{_m}|||{_entry['question']}" not in rpope_raw.get(_img_id, {}):
                _need_run = True
                break
        if _need_run:
            break
    if _need_run:
        methods_to_eval.append(_m)

if not methods_to_eval:
    print(f"All R-POPE results loaded from Drive.")
else:
    print(f"Running R-POPE for methods: {methods_to_eval}")
    n_imgs = len(images)

    for img_idx, img_id in enumerate(images):
        if img_id not in rpope_raw:
            rpope_raw[img_id] = {}
        questions = img_to_questions.get(img_id, [])

        for entry in questions[:10]:
            q = entry["question"]
            gt = entry["answer"]
            qtype = classify_question(q)

            for method in methods_to_eval:
                key = f"{method}|||{q}"
                if key not in rpope_raw[img_id]:
                    cap = method_captions[method].get(img_id)
                    if cap is None:
                        # fall back to llava_orig if no corrected caption
                        cap = llava_captions.get(img_id, "")
                    pred = rpope_judge(cap, q)
                    rpope_raw[img_id][key] = {"pred": pred, "gt": gt, "type": qtype, "method": method}
                    time.sleep(0.1)

        if (img_idx + 1) % 50 == 0:
            print(f"  [{img_idx+1}/{n_imgs}] evaluated")
            save_checkpoint(rpope_raw, RPOPE_PATH)

    save_checkpoint(rpope_raw, RPOPE_PATH)
    print("Done. R-POPE results saved.")


# ── Aggregate results ──
method_results = {m: [] for m in method_captions}
method_type_results = {m: defaultdict(list) for m in method_captions}
corrected_image_ids = {
    img_id for d in [llava_enriched, scout_enriched]
    for img_id, r in d.items() if r.get("status") == "modified"
}
improved_llava = []   # llava_orig wrong → llava_relcheck right
regressed_llava = []  # llava_orig right → llava_relcheck wrong
improved_qwen = []
regressed_qwen = []
per_question_rows = []

for img_id, img_data in rpope_raw.items():
    if img_id not in images:
        continue
    # group by question
    q_preds = defaultdict(dict)
    q_gts = {}
    q_types = {}
    for key, entry in img_data.items():
        if "|||" not in key:
            continue
        method, q = key.split("|||", 1)
        q_preds[q][method] = entry["pred"]
        q_gts[q] = entry["gt"]
        q_types[q] = entry.get("type", "OTHER")
        m = method
        pred, gt = entry["pred"], entry["gt"]
        if pred != "unknown" and m in method_results:
            method_results[m].append((pred, gt))
            method_type_results[m][q_types[q]].append((pred, gt))

    for q, preds in q_preds.items():
        gt = q_gts[q]
        # Track LLaVA improvements/regressions
        lo = preds.get("llava_orig", "unknown")
        lr = preds.get("llava_relcheck", "unknown")
        if lo != "unknown" and lr != "unknown":
            mod = llava_enriched.get(img_id, {}).get("status", "") == "modified"
            row = {"img_id": img_id, "question": q, "gt": gt, "type": q_types[q], "was_modified": mod}
            if lo != gt and lr == gt:
                improved_llava.append(row)
            elif lo == gt and lr != gt:
                regressed_llava.append(row)
        # Track Qwen improvements/regressions
        qo = preds.get("qwen_orig", "unknown")
        qr = preds.get("qwen_relcheck", "unknown")
        if qo != "unknown" and qr != "unknown":
            mod_q = scout_enriched.get(img_id, {}).get("status", "") == "modified"
            row_q = {"img_id": img_id, "question": q, "gt": gt, "type": q_types[q], "was_modified": mod_q}
            if qo != gt and qr == gt:
                improved_qwen.append(row_q)
            elif qo == gt and qr != gt:
                regressed_qwen.append(row_q)
        # CSV row
        per_question_rows.append({
            "img_id": img_id, "question": q, "gt": gt, "type": q_types[q],
            "llava_orig": lo, "llava_relcheck": lr,
            "qwen_orig": qo, "qwen_relcheck": qr,
        })


# ── TABLE 1: Main R-POPE LLM-Judge Results ──
method_labels = {
    "llava_orig":      "LLaVA-1.5 (orig)",
    "llava_relcheck":  "LLaVA + RelCheck",
    "qwen_orig":       "Qwen3-VL-8B (orig)",
    "qwen_relcheck":   "Qwen3-VL-8B + RelCheck",
}

print(f"\n{'='*75}")
print(f"TABLE 1: R-POPE (LLM-Judge) — {len(images)} images")
print(f"{'='*75}")
print(f"{'Method':<24} {'Accuracy':>10} {'Precision':>10} {'Recall':>9} {'F1':>8} {'Yes%':>7} {'N':>6}")
print("-" * 75)
for m in ["llava_orig", "llava_relcheck", "qwen_orig", "qwen_relcheck"]:
    if method_results.get(m):
        r = compute_metrics(method_results[m])
        label = method_labels.get(m, m)
        print(f"{label:<24} {r['accuracy']:>10.1%} {r['precision']:>10.1%} "
              f"{r['recall']:>9.1%} {r['f1']:>8.1%} {r['yes_ratio']:>7.1%} {r['n']:>6}")

# Deltas
for (orig_m, rc_m, name) in [("llava_orig","llava_relcheck","LLaVA"),
                               ("qwen_orig","qwen_relcheck","Qwen")]:
    if method_results.get(orig_m) and method_results.get(rc_m):
        r_orig = compute_metrics(method_results[orig_m])
        r_rc   = compute_metrics(method_results[rc_m])
        delta  = r_rc["accuracy"] - r_orig["accuracy"]
        print(f"  → {name} delta: {delta:+.1%}")


# ── McNemar's Test ──
for (improved, regressed, name) in [
    (improved_llava, regressed_llava, "LLaVA"),
    (improved_qwen,  regressed_qwen,  "Qwen"),
]:
    n01 = len(improved)
    n10 = len(regressed)
    if n01 + n10 > 0:
        chi2 = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
        if chi2 > 6.64:   sig_str = "p < 0.01 ***"
        elif chi2 > 3.84: sig_str = "p < 0.05 *"
        else:              sig_str = "p >= 0.05 (not significant)"
        print(f"\n{'='*60}")
        print(f"McNEMAR'S TEST: {name} orig vs {name} + RelCheck")
        print(f"{'='*60}")
        print(f"  Improved (orig wrong → RC right): {n01}")
        print(f"  Regressed (orig right → RC wrong): {n10}")
        print(f"  McNemar chi² = {chi2:.2f}  →  {sig_str}")
        print(f"  Net: {n01 - n10:+d} questions")

# Sample improvements/regressions
for (improved, regressed, name) in [
    (improved_llava, regressed_llava, "LLaVA"),
    (improved_qwen,  regressed_qwen,  "Qwen"),
]:
    if improved:
        print(f"\n  {name} TOP IMPROVED ({len(improved)} total):")
        for q in improved[:3]:
            mod_s = "[MOD]" if q["was_modified"] else "[same]"
            print(f"    [{q['type']}] {mod_s} {q['img_id'][:12]}: {q['question'][:55]}")
    if regressed:
        print(f"\n  {name} REGRESSED ({len(regressed)} total):")
        for q in regressed[:3]:
            mod_s = "[MOD]" if q["was_modified"] else "[same]"
            print(f"    [{q['type']}] {mod_s} {q['img_id'][:12]}: {q['question'][:55]}")


# ── TABLE 7: Filtered R-POPE (corrected images only) ──
print(f"\n{'='*75}")
print(f"TABLE 7: Filtered R-POPE (RelCheck-modified images only, N={len(corrected_image_ids)})")
print(f"{'='*75}")
print(f"{'Method':<24} {'Accuracy':>10} {'Delta':>10} {'N questions':>12}")
print("-" * 58)
for (orig_m, rc_m, name) in [("llava_orig","llava_relcheck","LLaVA"),
                               ("qwen_orig","qwen_relcheck","Qwen")]:
    for m in [orig_m, rc_m]:
        filt = [
            (entry["pred"], entry["gt"])
            for img_id in corrected_image_ids
            for key, entry in rpope_raw.get(img_id, {}).items()
            if "|||" in key and key.split("|||")[0] == m and entry["pred"] != "unknown"
        ]
        if filt:
            r = compute_metrics(filt)
            delta_str = ""
            if m == rc_m:
                orig_filt = [
                    (entry["pred"], entry["gt"])
                    for img_id in corrected_image_ids
                    for key, entry in rpope_raw.get(img_id, {}).items()
                    if "|||" in key and key.split("|||")[0] == orig_m and entry["pred"] != "unknown"
                ]
                if orig_filt:
                    r_orig = compute_metrics(orig_filt)
                    delta_str = f"{r['accuracy']-r_orig['accuracy']:>+10.1%}"
            label = method_labels.get(m, m)
            print(f"{label:<24} {r['accuracy']:>10.1%} {delta_str:>10} {r['n']:>12}")
    print()


# ── TABLE 8: Per-Relation-Type Breakdown ──
print(f"\n{'='*75}")
print(f"TABLE 8: Per-Relation-Type Breakdown")
print(f"{'='*75}")
for (orig_m, rc_m, name) in [("llava_orig","llava_relcheck","LLaVA"),
                               ("qwen_orig","qwen_relcheck","Qwen")]:
    if not (method_type_results.get(orig_m) and method_type_results.get(rc_m)):
        continue
    print(f"\n  {name}:")
    print(f"  {'Type':<12} {'Orig Acc':>10} {'RC Acc':>10} {'Delta':>8} {'N':>6}")
    print("  " + "-" * 46)
    for qtype in ["SPATIAL", "ACTION", "ATTRIBUTE", "OTHER"]:
        orig_t = method_type_results[orig_m][qtype]
        rc_t   = method_type_results[rc_m][qtype]
        if orig_t and rc_t:
            r_orig = compute_metrics(orig_t)
            r_rc   = compute_metrics(rc_t)
            delta  = r_rc["accuracy"] - r_orig["accuracy"]
            print(f"  {qtype:<12} {r_orig['accuracy']:>10.1%} {r_rc['accuracy']:>10.1%} {delta:>+8.1%} {len(orig_t):>6}")


# ── Relevance-Filtered R-POPE (TABLE 7b) ──
import spacy
try:
    nlp_sm = spacy.load("en_core_web_sm")
except:
    nlp_sm = None

def extract_question_nouns(question):
    q = question.lower()
    for prefix in ["is there ", "are there ", "is the ", "are the ",
                   "does the ", "do the ", "is a ", "are any ",
                   "can you see ", "is this "]:
        if q.startswith(prefix):
            q = q[len(prefix):]
            break
    if nlp_sm:
        doc = nlp_sm(q)
        nouns = [t.lemma_ for t in doc if t.pos_ in ("NOUN", "PROPN") and len(t.text) > 2]
        return nouns if nouns else [w for w in q.split() if len(w) > 3][:3]
    else:
        stop = {"the","this","that","with","from","have","been","being","does","image",
                "photo","picture","there","what","which","where","about","into"}
        return [w for w in q.split() if len(w) > 2 and w not in stop][:4]

def caption_covers_question(caption, question):
    nouns = extract_question_nouns(question)
    if not nouns:
        return True
    cap_lower = caption.lower()
    return any(noun in cap_lower for noun in nouns)

print(f"\n{'='*75}")
print(f"TABLE 7b: Relevance-Filtered R-POPE")
print(f"  (Only questions where RelCheck caption mentions relevant entities)")
print(f"{'='*75}")

for (orig_m, rc_m, name) in [("llava_orig","llava_relcheck","LLaVA"),
                               ("qwen_orig","qwen_relcheck","Qwen")]:
    rc_caps = method_captions.get(rc_m, {})
    orig_caps = method_captions.get(orig_m, {})
    if not rc_caps:
        continue

    relevant_questions = {}
    total_q = rel_q = 0
    for img_id in images:
        rc_cap = rc_caps.get(img_id, orig_caps.get(img_id, ""))
        for q_entry in img_to_questions.get(img_id, []):
            question = q_entry.get("question", q_entry.get("text", ""))
            total_q += 1
            if caption_covers_question(rc_cap, question):
                rel_q += 1
                relevant_questions.setdefault(img_id, []).append(question)

    print(f"\n  {name}: relevant {rel_q}/{total_q} ({rel_q/total_q:.1%})")
    print(f"  {'Method':<24} {'Accuracy':>10} {'Delta':>8} {'N':>6}")
    print("  " + "-" * 50)

    ref_acc = None
    for m in [orig_m, rc_m]:
        pairs = []
        for img_id, questions in relevant_questions.items():
            for question in questions:
                key = f"{m}|||{question}"
                entry = rpope_raw.get(img_id, {}).get(key, {})
                if entry and entry.get("pred") != "unknown":
                    pairs.append((entry["pred"], entry["gt"]))
        if pairs:
            r = compute_metrics(pairs)
            delta_str = ""
            if ref_acc is not None:
                delta_str = f"{r['accuracy']-ref_acc:>+8.1%}"
            else:
                ref_acc = r["accuracy"]
            label = method_labels.get(m, m)
            print(f"  {label:<24} {r['accuracy']:>10.1%} {delta_str:>8} {r['n']:>6}")


In [ ]:
# ============================================================
# Cell 8 — CLIPScore (Table 5)
# ============================================================
# Measures image-caption alignment for each method.
# Uses OpenCLIP ViT-B/32 — loads locally on Colab GPU.

import open_clip

print("Loading CLIP model...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai'
)
clip_tokenizer = open_clip.get_tokenizer('ViT-B-32')
clip_model = clip_model.to(DEVICE).eval()
print("CLIP loaded.")


@torch.no_grad()
def clip_score(image, caption):
    """Compute cosine similarity between image and caption in CLIP space."""
    img_tensor = clip_preprocess(image).unsqueeze(0).to(DEVICE)
    txt_tensor = clip_tokenizer([caption]).to(DEVICE)
    img_feat = clip_model.encode_image(img_tensor)
    txt_feat = clip_model.encode_text(txt_tensor)
    img_feat /= img_feat.norm(dim=-1, keepdim=True)
    txt_feat /= txt_feat.norm(dim=-1, keepdim=True)
    return (img_feat * txt_feat).sum().item()


CLIP_PATH = f"{SAVE_DIR}/clip_scores.json"
clip_raw = load_checkpoint(CLIP_PATH) or {}

methods_clip = ["llava_orig", "llava_relcheck", "qwen_orig", "qwen_relcheck"]
todo_clip = [img_id for img_id in images
             if not all(m in clip_raw.get(img_id, {}) for m in methods_clip
                        if m in method_captions and img_id in method_captions[m])]

if not todo_clip:
    print(f"Loaded {len(clip_raw)} cached CLIPScores.")
else:
    print(f"Computing CLIPScore for {len(todo_clip)} images...")
    for idx, img_id in enumerate(todo_clip):
        if img_id not in clip_raw:
            clip_raw[img_id] = {}
        img = images[img_id]
        for m in methods_clip:
            if m in method_captions and img_id in method_captions[m] and m not in clip_raw[img_id]:
                clip_raw[img_id][m] = clip_score(img, method_captions[m][img_id])
        if (idx + 1) % 100 == 0:
            print(f"  [{idx+1}/{len(todo_clip)}]")
            save_checkpoint(clip_raw, CLIP_PATH)
    save_checkpoint(clip_raw, CLIP_PATH)
    print("Done. CLIPScores saved.")

# ── Table 5: CLIPScore ──
print(f"\n{'='*55}")
print(f"TABLE 5: CLIPScore (image-caption alignment)")
print(f"{'='*55}")
print(f"{'Method':<26} {'Mean CLIPScore':>14} {'Delta vs orig':>14}")
print("-" * 56)

_pairs = [("llava_orig", "llava_relcheck", "LLaVA-1.5"),
          ("qwen_orig",  "qwen_relcheck",  "Qwen3-VL-8B")]
for orig_k, corr_k, label in _pairs:
    orig_scores = [clip_raw[i][orig_k] for i in clip_raw if orig_k in clip_raw.get(i,{})]
    corr_scores = [clip_raw[i][corr_k] for i in clip_raw if corr_k in clip_raw.get(i,{})]
    if orig_scores:
        orig_mean = np.mean(orig_scores)
        print(f"{label + ' (orig)':<26} {orig_mean:>14.4f} {'—':>14}")
    if corr_scores:
        corr_mean = np.mean(corr_scores)
        delta = corr_mean - orig_mean if orig_scores else 0
        print(f"{label + ' (RelCheck)':<26} {corr_mean:>14.4f} {delta:>+14.4f}")

In [ ]:
# ============================================================
# Cell 9 — Table 9: Pipeline Statistics + Save All CSVs
# ============================================================

# ── Table 9: Pipeline Stats ──
print(f"{'='*55}")
print(f"TABLE 9: Pipeline Statistics")
print(f"{'='*55}")

total_images = len(images)
total_questions = sum(len(qs) for qs in img_to_questions.values())
avg_q = total_questions / total_images

all_obj_counts = [len(Counter(l for l,_,_ in kb["detections"])) for kb in knowledge_bases.values()]
all_spatial_counts = [len(kb["spatial_facts"]) for kb in knowledge_bases.values()]

print(f"  Total images processed:     {total_images}")
print(f"  R-Bench questions:          {total_questions} ({avg_q:.1f}/image)")
print(f"  Avg objects detected/image: {np.mean(all_obj_counts):.1f}")
print(f"  Avg spatial facts/image:    {np.mean(all_spatial_counts):.1f}")

# ── Multi-Model Stats ──
for name, orig, enr_data in [("LLaVA-1.5", "llava_captions", "llava_enriched"),
                               ("Qwen2.5-VL-8B", "scout_captions", "scout_enriched")]:
    orig_dict = globals().get(orig, {})
    enr_dict = globals().get(enr_data, {})
    if orig_dict:
        n = len(orig_dict)
        avg_len = np.mean([len(c) for c in orig_dict.values() if c])
        n_mod = sum(1 for v in enr_dict.values() if v.get("status") == "modified")
        avg_edit = np.mean([v["edit_rate"] for v in enr_dict.values() if v.get("status") == "modified"]) if n_mod else 0
        print(f"\n  --- {name} ---")
        print(f"  Captions generated:  {n}")
        print(f"  Avg caption length:  {avg_len:.0f} chars")
        print(f"  Images corrected:    {n_mod}/{n} ({n_mod/n:.0%})")
        print(f"  Avg edit rate:       {avg_edit:.0%}")

# ── Save all CSVs to Drive ──
csv_dir = f"{SAVE_DIR}/csvs"
os.makedirs(csv_dir, exist_ok=True)

# Per-image results CSV (LLaVA + Qwen)
with open(f"{csv_dir}/per_image_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "img_id", "captioner", "original_caption", "relcheck_caption",
        "n_errors", "edit_rate", "status", "n_objects", "n_spatial_facts",
    ])
    writer.writeheader()
    for captioner, enr_data, orig_caps in [
        ("llava", llava_enriched, llava_captions),
        ("qwen",  scout_enriched, scout_captions),
    ]:
        for img_id in images:
            r  = enr_data.get(img_id, {})
            kb = knowledge_bases.get(img_id, {})
            writer.writerow({
                "img_id": img_id,
                "captioner": captioner,
                "original_caption": orig_caps.get(img_id, ""),
                "relcheck_caption": r.get("corrected", ""),
                "n_errors": len(r.get("errors", [])),
                "edit_rate": f"{r.get('edit_rate', 0):.3f}",
                "status": r.get("status", ""),
                "n_objects": len(Counter(l for l,_,_ in kb.get("detections", []))),
                "n_spatial_facts": len(kb.get("spatial_facts", [])),
            })

# R-POPE summary CSV
with open(f"{csv_dir}/rpope_summary.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["method","accuracy","precision","recall","f1","yes_ratio","n"])
    writer.writeheader()
    all_methods = ["llava_orig","llava_relcheck","qwen_orig","qwen_relcheck"]
    for m in all_methods:
        if method_results.get(m):
            r = compute_metrics(method_results[m], len(method_results[m]))
            writer.writerow({"method": m, **{k: f"{v:.4f}" for k, v in r.items()}})

# Per-relation-type CSV
with open(f"{csv_dir}/per_type_results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["method","type","accuracy","n"])
    writer.writeheader()
    for m in ["llava_orig","llava_relcheck","qwen_orig","qwen_relcheck"]:
        for qtype in ["SPATIAL","ACTION","ATTRIBUTE","OTHER"]:
            data = method_type_results.get(m, {}).get(qtype, [])
            if data:
                r = compute_metrics(data, len(data))
                writer.writerow({"method": m, "type": qtype,
                                  "accuracy": f"{r['accuracy']:.4f}", "n": r["n"]})

print(f"\nCSVs saved to {csv_dir}/")
print(f"  per_image_results.csv")
print(f"  rpope_summary.csv")
print(f"  per_type_results.csv")

# Per-question R-POPE CSV (all method predictions)
if per_question_rows:
    with open(f"{csv_dir}/per_question_rpope.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "img_id","question","gt","type",
            "llava_orig","llava_relcheck","qwen_orig","qwen_relcheck"])
        writer.writeheader()
        writer.writerows(per_question_rows)
    print(f"  per_question_rpope.csv ({len(per_question_rows)} rows)")

# Improved/regressed CSV
all_improved = improved_llava + improved_qwen
all_regressed = regressed_llava + regressed_qwen
if all_improved or all_regressed:
    with open(f"{csv_dir}/improved_regressed.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "direction","captioner","img_id","question","gt","type","was_modified"])
        writer.writeheader()
        for q in improved_llava:
            writer.writerow({"direction":"improved","captioner":"llava",
                "img_id":q["img_id"],"question":q["question"],"gt":q["gt"],
                "type":q["type"],"was_modified":q["was_modified"]})
        for q in improved_qwen:
            writer.writerow({"direction":"improved","captioner":"qwen",
                "img_id":q["img_id"],"question":q["question"],"gt":q["gt"],
                "type":q["type"],"was_modified":q["was_modified"]})
        for q in regressed_llava:
            writer.writerow({"direction":"regressed","captioner":"llava",
                "img_id":q["img_id"],"question":q["question"],"gt":q["gt"],
                "type":q["type"],"was_modified":q["was_modified"]})
        for q in regressed_qwen:
            writer.writerow({"direction":"regressed","captioner":"qwen",
                "img_id":q["img_id"],"question":q["question"],"gt":q["gt"],
                "type":q["type"],"was_modified":q["was_modified"]})
    print(f"  improved_regressed.csv ({len(all_improved)} improved, {len(all_regressed)} regressed)")



In [ ]:
# ============================================================
# Cell 11 — Qualitative Examples (Figure 2) + Failure Cases
# ============================================================
# Generates before/after visualizations with bounding boxes for:
#   - 5 best improved examples (for Figure 2 in report)
#   - 5 worst regressions (for failure analysis in Discussion)
# Saves images + captions to Drive.

from PIL import ImageDraw, ImageFont

QUAL_DIR = f"{SAVE_DIR}/qualitative"
os.makedirs(QUAL_DIR, exist_ok=True)

# ── Draw bounding boxes on image ──
def draw_bboxes(image, detections, max_boxes=8):
    """Draw labeled bounding boxes. Returns annotated copy."""
    img = image.copy()
    draw = ImageDraw.Draw(img)
    W, H = img.size
    colors = ["#2a9d8f", "#e76f51", "#264653", "#f4a261", "#e9c46a",
              "#9b72cf", "#2196f3", "#ff5722"]

    counts = Counter(l for l, _, _ in detections)
    for i, (label, score, bbox) in enumerate(detections[:max_boxes]):
        x1, y1, x2, y2 = bbox[0]*W, bbox[1]*H, bbox[2]*W, bbox[3]*H
        color = colors[i % len(colors)]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        text = f"{label} ({score:.0%})"
        # Simple text background
        draw.rectangle([x1, y1-16, x1+len(text)*7, y1], fill=color)
        draw.text((x1+2, y1-15), text, fill="white")
    return img


def save_qualitative(questions_list, direction, max_examples=5):
    """Save qualitative examples with bbox viz + caption comparison."""
    subdir = f"{QUAL_DIR}/{direction}"
    os.makedirs(subdir, exist_ok=True)

    # Deduplicate by image
    seen_imgs = set()
    unique = []
    for q in questions_list:
        if q["img_id"] not in seen_imgs:
            seen_imgs.add(q["img_id"])
            unique.append(q)
        if len(unique) >= max_examples:
            break

    for i, q in enumerate(unique):
        img_id = q["img_id"]
        img = images[img_id]
        kb = knowledge_bases[img_id]
        r = enriched[img_id]

        # Draw bboxes
        annotated = draw_bboxes(img, kb.get("detections", []))
        annotated.save(f"{subdir}/{i+1}_{img_id[:20]}_bboxes.jpg", quality=90)

        # Save caption comparison
        info = {
            "img_id": img_id,
            "question": q["question"],
            "gt_answer": q["gt"],
            "b1_answer": q["orig"],
            "relcheck_answer": q["enr"],
            "question_type": q["type"],
            "original_caption": r["caption"],
            "enriched_caption": r["corrected"],
            "errors_found": r.get("errors", []),
            "missing_found": r.get("missing", []),
            "kb_hard_facts": kb.get("hard_facts", []),
            "kb_spatial_facts": kb.get("spatial_facts", []),
            "kb_visual_desc": kb.get("visual_description", "")[:300],
            "edit_rate": r.get("edit_rate", 0),
        }
        with open(f"{subdir}/{i+1}_{img_id[:20]}_info.json", "w") as f:
            json.dump(info, f, indent=2, default=str)

        print(f"  {direction} #{i+1}: {img_id[:15]}")
        print(f"    Q: {q['question'][:65]}")
        print(f"    GT={q['gt']} | B1={q['orig']} | RC={q['enr']}")
        print(f"    Original: {r['caption'][:80]}...")
        print(f"    Enriched: {r['corrected'][:80]}...")
        print()


print("=" * 60)
print("QUALITATIVE EXAMPLES (for Figure 2)")
print("=" * 60)
if improved_questions:
    save_qualitative(improved_questions, "improved", 8)
else:
    print("  No improved questions to visualize.")

print()
print("=" * 60)
print("FAILURE CASES (for Discussion section)")
print("=" * 60)
if regressed_questions:
    save_qualitative(regressed_questions, "regressed", 5)
else:
    print("  No regressed questions to analyze.")

# ── Also save all-images gallery: just the modified ones with best improvement ──
# Sort images by improvement (most R-POPE questions flipped)
img_improvement = defaultdict(int)
for q in improved_questions:
    img_improvement[q["img_id"]] += 1
for q in regressed_questions:
    img_improvement[q["img_id"]] -= 1

best_images = sorted(img_improvement.items(), key=lambda x: -x[1])[:10]
print(f"\nTop 10 most-improved images:")
for img_id, delta in best_images:
    r = enriched[img_id]
    print(f"  {img_id[:20]}: net +{delta} questions | "
          f"errors={len(r.get('errors',[]))} missing={len(r.get('missing',[]))}")

print(f"\nQualitative outputs saved to {QUAL_DIR}/")


# ============================================================
# Generate Woodpecker-style HTML Comparison (Figure 2 for report)
# ============================================================
# For each example: image with bboxes | original caption | KB evidence | corrected caption
# Errors in red strikethrough, additions in green bold.

import difflib

def diff_captions(original, corrected):
    """Generate HTML showing word-level diff between original and corrected captions."""
    orig_words = original.split()
    corr_words = corrected.split()
    sm = difflib.SequenceMatcher(None, orig_words, corr_words)

    html_parts = []
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        if op == "equal":
            html_parts.append(" ".join(orig_words[i1:i2]))
        elif op == "replace":
            html_parts.append(f'<span style="color:#d32f2f;text-decoration:line-through">{" ".join(orig_words[i1:i2])}</span>')
            html_parts.append(f'<span style="color:#2e7d32;font-weight:bold">{" ".join(corr_words[j1:j2])}</span>')
        elif op == "delete":
            html_parts.append(f'<span style="color:#d32f2f;text-decoration:line-through">{" ".join(orig_words[i1:i2])}</span>')
        elif op == "insert":
            html_parts.append(f'<span style="color:#2e7d32;font-weight:bold">{" ".join(corr_words[j1:j2])}</span>')
    return " ".join(html_parts)


def generate_comparison_html(examples, title="RelCheck Qualitative Examples"):
    """Generate full HTML page with Woodpecker-style before/after comparisons."""
    html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<title>{title}</title>
<style>
body {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif; max-width: 1100px; margin: 0 auto; padding: 20px; background: #fafafa; }}
h1 {{ color: #1a237e; border-bottom: 3px solid #1a237e; padding-bottom: 8px; }}
.example {{ background: white; border-radius: 12px; box-shadow: 0 2px 8px rgba(0,0,0,0.08); margin: 24px 0; padding: 24px; }}
.example-header {{ font-size: 14px; color: #666; margin-bottom: 12px; }}
.grid {{ display: grid; grid-template-columns: 350px 1fr; gap: 20px; }}
.image-col img {{ width: 100%; border-radius: 8px; }}
.caption-box {{ padding: 12px 16px; border-radius: 8px; margin: 8px 0; font-size: 14px; line-height: 1.6; }}
.original {{ background: #fff3e0; border-left: 4px solid #e65100; }}
.corrected {{ background: #e8f5e9; border-left: 4px solid #2e7d32; }}
.evidence {{ background: #e3f2fd; border-left: 4px solid #1565c0; font-size: 13px; }}
.label {{ font-weight: 700; font-size: 12px; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 4px; }}
.label-orig {{ color: #e65100; }}
.label-corrected {{ color: #2e7d32; }}
.label-evidence {{ color: #1565c0; }}
.diff {{ background: #f5f5f5; border-radius: 8px; padding: 12px 16px; margin: 8px 0; font-size: 14px; line-height: 1.6; border-left: 4px solid #7b1fa2; }}
.label-diff {{ color: #7b1fa2; }}
.q-badge {{ display: inline-block; padding: 2px 8px; border-radius: 4px; font-size: 12px; font-weight: 600; margin: 2px; }}
.q-improved {{ background: #c8e6c9; color: #1b5e20; }}
.q-regressed {{ background: #ffcdd2; color: #b71c1c; }}
.error-item {{ color: #d32f2f; font-size: 13px; margin: 2px 0; }}
.missing-item {{ color: #1565c0; font-size: 13px; margin: 2px 0; }}
.stats {{ display: flex; gap: 16px; margin: 8px 0; }}
.stat {{ font-size: 12px; color: #555; }}
</style>
</head><body>
<h1>{title}</h1>
<p style="color:#666">Generated by RelCheck pipeline. <span style="color:#d32f2f;text-decoration:line-through">Red strikethrough</span> = removed/corrected. <span style="color:#2e7d32;font-weight:bold">Green bold</span> = added/fixed.</p>
"""

    for i, ex in enumerate(examples):
        img_id = ex["img_id"]
        r = enriched[img_id]
        kb = knowledge_bases[img_id]
        orig_cap = r["caption"]
        corr_cap = r["corrected"]

        # Encode image with bboxes as base64
        annotated = draw_bboxes(images[img_id], kb.get("detections", []))
        buf = BytesIO()
        annotated.save(buf, format="JPEG", quality=85)
        img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

        # Build evidence summary
        errors_html = ""
        if r.get("errors"):
            for err in r["errors"][:3]:
                claim = err.get("claim", "")
                correction = err.get("correction", "")
                errors_html += f'<div class="error-item">&#10060; Claim: "{claim}" &#8594; {correction}</div>'

        missing_html = ""
        if r.get("missing"):
            for m in r["missing"][:4]:
                fact = m.get("fact", "")
                source = m.get("source", "")
                missing_html += f'<div class="missing-item">&#10133; {fact} <em>({source})</em></div>'

        # KB summary
        n_objects = len(set(l for l, _, _ in kb.get("detections", [])))
        n_spatial = len(kb.get("spatial_facts", []))
        kb_desc_short = kb.get("visual_description", "")[:200]
        if len(kb.get("visual_description", "")) > 200:
            kb_desc_short += "..."

        # Word-level diff
        diff_html = diff_captions(orig_cap, corr_cap) if orig_cap != corr_cap else "<em>No changes made</em>"

        # Related questions
        q_html = ""
        for q in improved_questions:
            if q["img_id"] == img_id:
                q_html += f'<span class="q-badge q-improved">&#10004; {q["question"][:50]}...</span> '
        for q in regressed_questions:
            if q["img_id"] == img_id:
                q_html += f'<span class="q-badge q-regressed">&#10008; {q["question"][:50]}...</span> '

        html += f"""
<div class="example">
  <div class="example-header">Example {i+1} &mdash; <code>{img_id[:25]}</code>
    <span class="stats">
      <span class="stat">Objects: {n_objects}</span>
      <span class="stat">Spatial facts: {n_spatial}</span>
      <span class="stat">Edit rate: {r.get("edit_rate",0):.0%}</span>
    </span>
  </div>
  <div class="grid">
    <div class="image-col">
      <img src="data:image/jpeg;base64,{img_b64}" alt="{img_id}">
    </div>
    <div>
      <div class="caption-box original">
        <div class="label label-orig">Original Caption ({len(orig_cap.split())} words)</div>
        {orig_cap}
      </div>

      <div class="caption-box evidence">
        <div class="label label-evidence">Visual KB Evidence</div>
        {errors_html}
        {missing_html}
        <div style="margin-top:6px;font-size:12px;color:#555">VLM: {kb_desc_short}</div>
      </div>

      <div class="diff">
        <div class="label label-diff">Word-Level Diff</div>
        {diff_html}
      </div>

      <div class="caption-box corrected">
        <div class="label label-corrected">RelCheck Output ({len(corr_cap.split())} words)</div>
        {corr_cap}
      </div>

      {f'<div style="margin-top:8px">{q_html}</div>' if q_html else ''}
    </div>
  </div>
</div>"""

    html += "\n</body></html>"
    return html


# ── Generate HTML for improved examples ──
if improved_questions:
    # Deduplicate by image, take top 8
    seen = set()
    unique_improved = []
    for q in improved_questions:
        if q["img_id"] not in seen and enriched[q["img_id"]]["status"] == "modified":
            seen.add(q["img_id"])
            unique_improved.append(q)
        if len(unique_improved) >= 8:
            break

    html = generate_comparison_html(unique_improved, "RelCheck: Improved Examples (Figure 2)")
    html_path = f"{QUAL_DIR}/figure2_improved.html"
    with open(html_path, "w") as f:
        f.write(html)
    print(f"\nSaved Figure 2 HTML: {html_path}")
    print(f"  {len(unique_improved)} examples with word-level diffs")

# ── Generate HTML for failure cases ──
if regressed_questions:
    seen = set()
    unique_regressed = []
    for q in regressed_questions:
        if q["img_id"] not in seen:
            seen.add(q["img_id"])
            unique_regressed.append(q)
        if len(unique_regressed) >= 5:
            break

    html = generate_comparison_html(unique_regressed, "RelCheck: Failure Cases (Discussion)")
    html_path = f"{QUAL_DIR}/failure_cases.html"
    with open(html_path, "w") as f:
        f.write(html)
    print(f"Saved failure cases HTML: {html_path}")

# ── Generate HTML for multi-model examples (LLaVA corrections) ──
# Show LLaVA's hallucination-prone captions being corrected
if 'llava_enriched' in dir() and llava_enriched:
    llava_examples = []
    for img_id in list(images.keys())[:100]:
        lr = llava_enriched.get(img_id, {})
        if lr.get("status") == "modified" and lr.get("errors"):
            llava_examples.append({"img_id": img_id})
        if len(llava_examples) >= 5:
            break

    if llava_examples:
        # Temporarily swap enriched data for the HTML generator
        _orig_enriched = enriched
        enriched_for_llava = {}
        for ex in llava_examples:
            enriched_for_llava[ex["img_id"]] = llava_enriched[ex["img_id"]]

        # Build a modified version
        enriched = enriched_for_llava
        html = generate_comparison_html(llava_examples, "RelCheck on LLaVA-1.5: Correcting Relational Hallucinations")
        enriched = _orig_enriched  # restore

        html_path = f"{QUAL_DIR}/llava_corrections.html"
        with open(html_path, "w") as f:
            f.write(html)
        print(f"Saved LLaVA corrections HTML: {html_path}")
        print(f"  {len(llava_examples)} examples showing actual hallucination correction")

print(f"\nAll qualitative HTML files saved to {QUAL_DIR}/")


In [ ]:
# ============================================================
# Cell 10 — Paper Figures (LaTeX-ready, 300 DPI)
# ============================================================
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
})

FIGS_DIR = f"{SAVE_DIR}/figures"
os.makedirs(FIGS_DIR, exist_ok=True)

COLORS = {
    "llava_orig":      "#aec6cf",
    "llava_relcheck":  "#2a9d8f",
    "qwen_orig":       "#ffab91",
    "qwen_relcheck":   "#e76f51",
    "ib_orig":         "#ddd",
    "ib_relcheck":     "#9b72cf",
}

def method_acc(m):
    d = method_results.get(m, [])
    return compute_metrics(d, len(d))["accuracy"] if d else 0


# ── Figure 3: Main R-POPE Results ──
fig, ax = plt.subplots(figsize=(9, 4.5))
pairs = [
    ("llava_orig",     "LLaVA-1.5\n(Original)"),
    ("llava_relcheck", "LLaVA-1.5\n+ RelCheck"),
    ("qwen_orig",      "Qwen3-VL\n(Original)"),
    ("qwen_relcheck",  "Qwen3-VL\n+ RelCheck"),
]
present = [(m, lbl) for m, lbl in pairs if method_results.get(m)]
if present:
    labels = [lbl for _, lbl in present]
    accs   = [method_acc(m) for m, _ in present]
    colors = [COLORS[m] for m, _ in present]

    bars = ax.bar(labels, accs, color=colors, edgecolor="gray", width=0.55)
    ax.set_ylabel("R-POPE Accuracy (LLM-Judge)")
    ax.set_ylim(max(0, min(accs) - 0.08), min(1.0, max(accs) + 0.07))
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

    for bar, acc in zip(bars, accs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{acc:.1%}", ha='center', va='bottom', fontsize=10, fontweight='bold')

    # Highlight RelCheck bars with thicker border
    for bar, (m, _) in zip(bars, present):
        if "relcheck" in m:
            bar.set_edgecolor('#1a5c4f')
            bar.set_linewidth(2)

    # Delta annotations between each pair
    for i in range(0, len(present)-1, 2):
        orig_acc = accs[i]
        rc_acc   = accs[i+1]
        delta    = rc_acc - orig_acc
        color    = '#2e7d32' if delta >= 0 else '#d32f2f'
        mid_x    = (bars[i].get_x()+bars[i].get_width() + bars[i+1].get_x()) / 2
        y_top    = max(orig_acc, rc_acc) + 0.03
        ax.annotate(f"{delta:+.1%}", xy=(mid_x, y_top), ha='center', fontsize=10,
                    color=color, fontweight='bold')

    plt.tight_layout()
    plt.savefig(f"{FIGS_DIR}/fig3_main_results.png")
    plt.savefig(f"{FIGS_DIR}/fig3_main_results.pdf")
    plt.show()
    print("Figure 3 saved (PNG + PDF)")


# ── Figure 4: Per-Relation-Type Grouped Bar Chart ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)
qtypes = ["SPATIAL", "ACTION", "ATTRIBUTE", "OTHER"]
x = np.arange(len(qtypes))
width = 0.3

for ax_i, (orig_m, rc_m, name) in enumerate([
    ("llava_orig", "llava_relcheck", "LLaVA-1.5"),
    ("qwen_orig",  "qwen_relcheck",  "Qwen3-VL-8B"),
]):
    ax = axes[ax_i]
    if not (method_type_results.get(orig_m) and method_type_results.get(rc_m)):
        ax.set_title(f"{name} — no data")
        continue
    orig_accs = [compute_metrics(method_type_results[orig_m][t])["accuracy"]
                 if method_type_results[orig_m][t] else 0 for t in qtypes]
    rc_accs   = [compute_metrics(method_type_results[rc_m][t])["accuracy"]
                 if method_type_results[rc_m][t] else 0 for t in qtypes]

    bars1 = ax.bar(x - width/2, orig_accs, width, label=f"{name} (Original)",
                   color=COLORS[orig_m], edgecolor="gray")
    bars2 = ax.bar(x + width/2, rc_accs, width, label="+ RelCheck",
                   color=COLORS[rc_m], edgecolor="gray")
    ax.set_xlabel("Relation Type")
    ax.set_ylabel("R-POPE Accuracy")
    ax.set_xticks(x)
    ax.set_xticklabels(qtypes)
    ax.set_ylim(0, 1.0)
    ax.legend(loc='upper right', fontsize=9)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
    ax.set_title(name)

    for i, (oa, rca) in enumerate(zip(orig_accs, rc_accs)):
        if rca > 0:
            delta = rca - oa
            color = '#2e7d32' if delta >= 0 else '#d32f2f'
            ax.text(x[i] + width/2, rca + 0.015, f"{delta:+.1%}",
                    ha='center', fontsize=8, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig4_per_relation_type.png")
plt.savefig(f"{FIGS_DIR}/fig4_per_relation_type.pdf")
plt.show()
print("Figure 4 saved")


# ── Figure 7: Multi-Model Comparison (key figure) ──
fig, ax = plt.subplots(figsize=(10, 5))

captioner_pairs = [
    ("llava_orig", "llava_relcheck", "LLaVA-1.5"),
    ("qwen_orig",  "qwen_relcheck",  "Qwen3-VL-8B"),
]
# Add InstructBLIP if available
if method_results.get("ib_orig"):
    captioner_pairs.append(("ib_orig", "ib_relcheck", "InstructBLIP"))

x = np.arange(len(captioner_pairs))
width = 0.3

orig_accs = [method_acc(orig_m) for orig_m, _, _ in captioner_pairs]
rc_accs   = [method_acc(rc_m) for _, rc_m, _ in captioner_pairs]
xlabels   = [name for _, _, name in captioner_pairs]

bars1 = ax.bar(x - width/2, orig_accs, width, label="Original", color="#aec6cf", edgecolor="gray")
bars2 = ax.bar(x + width/2, rc_accs,   width, label="+ RelCheck (Ours)", color="#2a9d8f", edgecolor="gray")

ax.set_xlabel("Captioner")
ax.set_ylabel("R-POPE Accuracy (LLM-Judge)")
ax.set_xticks(x)
ax.set_xticklabels(xlabels)
ax.set_ylim(max(0, min(orig_accs + rc_accs) - 0.08), min(1.0, max(orig_accs + rc_accs) + 0.10))
ax.legend(loc='upper right')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

for bar, acc in zip(bars1, orig_accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{acc:.1%}", ha='center', va='bottom', fontsize=9, fontweight='bold')
for i, (bar, acc) in enumerate(zip(bars2, rc_accs)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{acc:.1%}", ha='center', va='bottom', fontsize=9, fontweight='bold')
    delta = acc - orig_accs[i]
    color = '#2e7d32' if delta >= 0 else '#d32f2f'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.025,
            f"{delta:+.1%}", ha='center', va='bottom', fontsize=9, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{FIGS_DIR}/fig7_multi_model_comparison.png")
plt.savefig(f"{FIGS_DIR}/fig7_multi_model_comparison.pdf")
plt.show()
print("Figure 7 saved")


# ── Figure 8: Improvement Breakdown ──
all_improved = improved_llava + improved_qwen
if all_improved:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))

    # Left: by relation type
    type_counts = Counter(q["type"] for q in all_improved)
    type_colors = {"SPATIAL": "#2196f3", "ACTION": "#ff9800",
                   "ATTRIBUTE": "#4caf50", "OTHER": "#9e9e9e"}
    labels_t = list(type_counts.keys())
    sizes_t  = list(type_counts.values())
    colors_t = [type_colors.get(t, "#ccc") for t in labels_t]
    axes[0].pie(sizes_t, labels=labels_t, colors=colors_t,
                autopct='%1.0f%%', startangle=90, pctdistance=0.75,
                wedgeprops=dict(width=0.4, edgecolor='white'))
    axes[0].set_title(f"Improved Questions by Type\n(N={len(all_improved)})")

    # Right: modified vs unmodified
    mod_counts = Counter("Caption modified" if q["was_modified"] else "Caption unchanged"
                          for q in all_improved)
    axes[1].pie(list(mod_counts.values()), labels=list(mod_counts.keys()),
                colors=["#2a9d8f","#e0e0e0"],
                autopct='%1.0f%%', startangle=90, pctdistance=0.75,
                wedgeprops=dict(width=0.4, edgecolor='white'))
    axes[1].set_title("Source of Improvement")

    plt.tight_layout()
    plt.savefig(f"{FIGS_DIR}/fig8_improvement_breakdown.png")
    plt.savefig(f"{FIGS_DIR}/fig8_improvement_breakdown.pdf")
    plt.show()
    print("Figure 8 saved")


# ── Figure 10: Caption Length Comparison ──
fig, ax = plt.subplots(figsize=(9, 4.5))
length_data   = []
length_labels = []

for name, orig_dict, enr_data in [
    ("LLaVA-1.5",       llava_captions, llava_enriched),
    ("Qwen3-VL-8B",     scout_captions, scout_enriched),
]:
    if orig_dict:
        orig_lens = [len(c.split()) for c in orig_dict.values() if c]
        corr_lens = [len(r["corrected"].split()) for r in enr_data.values() if r.get("corrected")]
        if orig_lens:
            length_data.append(orig_lens)
            length_labels.append(f"{name}\n(Original)")
        if corr_lens:
            length_data.append(corr_lens)
            length_labels.append(f"{name}\n(+ RelCheck)")

if length_data:
    bp = ax.boxplot(length_data, labels=length_labels, patch_artist=True, widths=0.5)
    box_colors = ["#aec6cf","#2a9d8f","#ffab91","#e76f51"]
    for patch, color in zip(bp['boxes'], box_colors[:len(bp['boxes'])]):
        patch.set_facecolor(color)
    ax.set_ylabel("Caption Length (words)")
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{FIGS_DIR}/fig10_caption_lengths.png")
    plt.savefig(f"{FIGS_DIR}/fig10_caption_lengths.pdf")
    plt.show()
    print("Figure 10 saved")


# ── Summary ──
print(f"\n{'='*65}")
print(f"ALL FIGURES SAVED to {FIGS_DIR}/")
print(f"{'='*65}")
for name, orig_m, rc_m in [("LLaVA-1.5","llava_orig","llava_relcheck"),
                              ("Qwen3-VL-8B","qwen_orig","qwen_relcheck")]:
    if method_results.get(orig_m):
        delta = method_acc(rc_m) - method_acc(orig_m)
        print(f"  {name}: {method_acc(orig_m):.1%} → {method_acc(rc_m):.1%} ({delta:+.1%})")


In [ ]:
# ============================================================
# Cell 13 -- R-CHAIR (Table 6)
# ============================================================
# A) Extract triples from B1 + RelCheck captions (50 images)
# B) Generate interactive HTML annotation tool
# C) Automated VLM R-CHAIR via Maverick
# D) Read back manual annotations -> compute R-CHAIR + agreement

import csv, re

RCHAIR_PATH = f"{SAVE_DIR}/rchair_results.json"
ANNOTATION_CSV = f"{SAVE_DIR}/rchair_annotation.csv"
ANNOTATION_DONE = f"{SAVE_DIR}/rchair_annotation_done.csv"
ANNOTATION_HTML = f"{SAVE_DIR}/rchair_annotator.html"

EXTRACT_TRIPLES_PROMPT = """Extract all relational claims from this caption as (subject, relation, object) triples.
Only include claims about relationships between entities (spatial, action, attribute).

Caption: "{caption}"

Output a JSON list: [{{"subject": "...", "relation": "...", "object": "..."}}]
If no relational claims, output: []"""

VERIFY_TRIPLE_PROMPT = """Look at this image carefully. Is the following statement true?

Statement: "{subject} {relation} {object}"

Answer ONLY "yes" or "no"."""


def extract_triples(caption):
    raw = llm_call([{"role": "user", "content":
        EXTRACT_TRIPLES_PROMPT.format(caption=caption)}], max_tokens=500)
    if not raw:
        return []
    raw = re.sub(r'^```json\s*', '', raw)
    raw = re.sub(r'```\s*$', '', raw)
    try:
        triples = json.loads(raw)
        return triples if isinstance(triples, list) else []
    except:
        return []


def verify_triple_vlm(image, subject, relation, obj):
    b64 = encode_b64(image)
    raw = llm_call(
        messages=[{"role": "user", "content": [
            {"type": "text", "text": VERIFY_TRIPLE_PROMPT.format(
                subject=subject, relation=relation, object=obj)},
            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
        ]}],
        model=VLM_MODEL, max_tokens=10,
    )
    if raw and "yes" in raw.lower() and "no" not in raw.lower():
        return True
    return False


# -- Sample images for R-CHAIR --
N_RCHAIR = 100  # increase from 50 for stronger hallucination rate estimates
rchair_sample = random.sample(list(images.keys()), min(N_RCHAIR, len(images)))
# Build methods_rchair: LLaVA and Qwen only (no BLIP-2)
def _safe_var(name, fallback=None):
    g = globals()
    return g[name] if name in g else fallback

_llava_caps_  = _safe_var("llava_captions", {})
_scout_caps_  = _safe_var("scout_captions", {})
_llava_enr_   = _safe_var("llava_enriched", {})
_scout_enr_   = _safe_var("scout_enriched", {})

methods_rchair = {}
if _llava_caps_:
    methods_rchair["llava_orig"]      = dict(_llava_caps_)
    methods_rchair["llava_corrected"] = {
        img_id: (_llava_enr_.get(img_id, {}).get("corrected") or _llava_caps_[img_id])
        for img_id in _llava_caps_
    }
if _scout_caps_:
    methods_rchair["qwen_orig"]      = dict(_scout_caps_)
    methods_rchair["qwen_corrected"] = {
        img_id: (_scout_enr_.get(img_id, {}).get("corrected") or _scout_caps_[img_id])
        for img_id in _scout_caps_
    }

if not methods_rchair:
    print("  WARNING: No LLaVA/Qwen captions found — run Cell 5 first.")

# === PART A: Extract triples (cached) ===
rchair_raw = load_checkpoint(RCHAIR_PATH) or {}

for method, caps in methods_rchair.items():
    key = f"triples_{method}"
    if key in rchair_raw and len(rchair_raw[key]) >= len(rchair_sample):
        print(f"  Triples {method}: cached ({len(rchair_raw[key])})")
        continue
    if key not in rchair_raw:
        rchair_raw[key] = {}
    print(f"  Extracting triples for {method}...")
    for idx, img_id in enumerate(rchair_sample):
        if img_id in rchair_raw[key]:
            continue
        cap = caps.get(img_id, captions[img_id])
        rchair_raw[key][img_id] = extract_triples(cap)[:8]
        if (idx + 1) % 10 == 0:
            print(f"    [{idx+1}/{len(rchair_sample)}]")
            save_checkpoint(rchair_raw, RCHAIR_PATH)
    save_checkpoint(rchair_raw, RCHAIR_PATH)

# CSV backup
csv_rows = []
for img_id in rchair_sample:
    for method, caps_dict in methods_rchair.items():
        key = f"triples_{method}"
        cap = caps_dict.get(img_id, "")
        for i, t in enumerate(rchair_raw.get(key, {}).get(img_id, [])):
            s, r_rel, o = t.get("subject", ""), t.get("relation", ""), t.get("object", "")
            if s and r_rel and o:
                csv_rows.append({
                    "image_id": img_id, "method": method, "caption": cap,
                    "triple_idx": i, "subject": s, "relation": r_rel, "object": o,
                    "triple_text": f"{s} {r_rel} {o}", "hallucinated": "",
                })
with open(ANNOTATION_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=[
        "image_id", "method", "caption", "triple_idx",
        "subject", "relation", "object", "triple_text", "hallucinated"])
    w.writeheader()
    w.writerows(csv_rows)
print(f"  CSV backup: {ANNOTATION_CSV} ({len(csv_rows)} triples)")


# === PART B: HTML Annotation Tool ===
print("\nGenerating annotation tool...")

annotation_data = []
for img_id in rchair_sample:
    b64 = encode_b64(images[img_id])
    entry = {"id": img_id, "b64": b64, "methods": {}}
    for m_key, caps_dict in methods_rchair.items():
        cap = caps_dict.get(img_id, "")
        triples_for_method = []
        for t in rchair_raw.get(f"triples_{m_key}", {}).get(img_id, []):
            s, r, o = t.get("subject", ""), t.get("relation", ""), t.get("object", "")
            if s and r and o:
                triples_for_method.append({"s": s, "r": r, "o": o})
        entry["methods"][m_key] = {"caption": cap, "triples": triples_for_method}
    annotation_data.append(entry)

js_data = [{"id": e["id"], "methods": e["methods"]} for e in annotation_data]

# Build HTML
CSS = """
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:-apple-system,BlinkMacSystemFont,sans-serif;background:#f5f5f5;color:#333}
.header{background:#1a1a2e;color:white;padding:16px 24px;display:flex;justify-content:space-between;align-items:center;position:sticky;top:0;z-index:100}
.header h1{font-size:18px;font-weight:600}
.progress-bar{width:200px;height:8px;background:#333;border-radius:4px;overflow:hidden}
.progress-fill{height:100%;background:#4ade80;transition:width 0.3s}
.progress-text{font-size:13px;color:#aaa;margin-top:4px;text-align:right}
.container{max-width:1100px;margin:24px auto;padding:0 16px}
.nav-bar{display:flex;justify-content:space-between;align-items:center;margin-bottom:16px}
.nav-bar button{padding:10px 24px;border:none;border-radius:6px;font-size:14px;font-weight:600;cursor:pointer}
.nav-bar button:disabled{opacity:0.3;cursor:not-allowed}
.btn-prev{background:#e2e8f0;color:#333}
.btn-next{background:#3b82f6;color:white}
.btn-download{background:#10b981;color:white;padding:10px 24px;border:none;border-radius:6px;font-size:14px;font-weight:600;cursor:pointer}
.image-card{background:white;border-radius:12px;box-shadow:0 2px 8px rgba(0,0,0,0.08);overflow:hidden;margin-bottom:20px}
.image-card img{width:100%;max-height:420px;object-fit:contain;background:#000;display:block}
.card-body{padding:20px}
.method-section{margin-bottom:24px;padding:16px;border-radius:8px}
.method-b1{background:#fef3c7;border:1px solid #fbbf24}
.method-rc{background:#d1fae5;border:1px solid #34d399}
.method-label{font-weight:700;font-size:14px;margin-bottom:6px;text-transform:uppercase;letter-spacing:0.5px}
.method-b1 .method-label{color:#92400e}
.method-rc .method-label{color:#065f46}
.caption-text{font-size:13px;color:#555;margin-bottom:12px;font-style:italic}
.triple-row{display:flex;align-items:center;justify-content:space-between;padding:10px 12px;margin-bottom:6px;background:white;border-radius:6px;border:1px solid #e5e7eb}
.triple-text{font-size:14px;flex:1}
.triple-text em{color:#6366f1;font-style:normal;font-weight:600}
.btn-group{display:flex;gap:6px;margin-left:12px}
.btn-yes,.btn-no{padding:6px 16px;border:2px solid #ddd;border-radius:6px;font-size:13px;font-weight:600;cursor:pointer;background:white}
.btn-yes.selected{background:#22c55e;color:white;border-color:#22c55e}
.btn-no.selected{background:#ef4444;color:white;border-color:#ef4444}
.img-counter{font-size:15px;font-weight:600;color:#555}
.jump-input{width:60px;padding:6px;border:1px solid #ccc;border-radius:4px;text-align:center;font-size:14px;margin:0 6px}
.stats-bar{display:flex;gap:16px;font-size:13px;color:#666;margin-top:8px}
"""

JS_LOGIC = r"""
const annotations={};
const METHOD_KEYS=DATA.length>0?Object.keys(DATA[0].methods):[];
DATA.forEach(d=>{
  annotations[d.id]={};
  METHOD_KEYS.forEach(m=>{
    annotations[d.id][m]={};
    (d.methods[m]&&d.methods[m].triples||[]).forEach((_,i)=>annotations[d.id][m][i]='');
  });
});
let currentIdx=0;

function render(){
  const d=DATA[currentIdx];
  document.getElementById('mainImg').src='data:image/jpeg;base64,'+IMG_B64[d.id];
  document.getElementById('jumpInput').value=currentIdx+1;
  document.getElementById('totalCount').textContent=DATA.length;
  document.getElementById('btnPrev').disabled=currentIdx===0;
  document.getElementById('btnNext').disabled=currentIdx===DATA.length-1;
  const methodsDiv=document.getElementById('methodsContainer');
  methodsDiv.innerHTML='';
  METHOD_KEYS.forEach(m=>{
    const mdata=d.methods[m]||{caption:'',triples:[]};
    const isRC=m.includes('relcheck')||m.includes('corrected');
    const bgColor=isRC?'#d1fae5':'#fef3c7';
    const bdColor=isRC?'#34d399':'#fbbf24';
    const lblColor=isRC?'#065f46':'#92400e';
    const div=document.createElement('div');
    div.style.cssText='margin-bottom:24px;padding:16px;border-radius:8px;background:'+bgColor+';border:1px solid '+bdColor;
    div.innerHTML='<div style="font-weight:700;font-size:14px;margin-bottom:6px;text-transform:uppercase;color:'+lblColor+'">'+m.replace(/_/g,' ')+'</div>'
      +'<div style="font-size:13px;color:#555;margin-bottom:12px;font-style:italic">'+mdata.caption+'</div>'
      +'<div id="triples_'+m+'_'+d.id+'"></div>';
    methodsDiv.appendChild(div);
    renderTriples('triples_'+m+'_'+d.id, mdata.triples||[], d.id, m);
  });
  updateStats();
}

function renderTriples(cid,triples,imgId,method){
  const el=document.getElementById(cid);
  if(!el)return;
  if(!triples.length){el.innerHTML='<div style="color:#999;font-size:13px">No triples extracted.</div>';return}
  el.innerHTML=triples.map((t,i)=>{
    const ann=(annotations[imgId]&&annotations[imgId][method])?annotations[imgId][method][i]:'';
    return '<div style="display:flex;align-items:center;justify-content:space-between;padding:10px 12px;margin-bottom:6px;background:white;border-radius:6px;border:1px solid #e5e7eb">'
      +'<div style="font-size:14px;flex:1"><em style="color:#6366f1;font-style:normal;font-weight:600">'+t.s+'</em> '+t.r+' <em style="color:#6366f1;font-style:normal;font-weight:600">'+t.o+'</em></div>'
      +'<div style="display:flex;gap:6px;margin-left:12px">'
      +'<button style="padding:6px 16px;border:2px solid '+(ann==="no"?"#22c55e":"#ddd")+';border-radius:6px;font-size:13px;font-weight:600;cursor:pointer;background:'+(ann==="no"?"#22c55e":"white")+';color:'+(ann==="no"?"white":"#555")+'" onclick="annotate(''+imgId+'',''+method+'','+i+','no')">Correct</button>'
      +'<button style="padding:6px 16px;border:2px solid '+(ann==="yes"?"#ef4444":"#ddd")+';border-radius:6px;font-size:13px;font-weight:600;cursor:pointer;background:'+(ann==="yes"?"#ef4444":"white")+';color:'+(ann==="yes"?"white":"#555")+'" onclick="annotate(''+imgId+'',''+method+'','+i+','yes')">Hallucinated</button>'
      +'</div></div>'
  }).join('');
}

function annotate(imgId,method,idx,value){
  if(!annotations[imgId])annotations[imgId]={};
  if(!annotations[imgId][method])annotations[imgId][method]={};
  annotations[imgId][method][idx]=value;
  render();
}
function go(delta){currentIdx=Math.max(0,Math.min(DATA.length-1,currentIdx+delta));render();window.scrollTo(0,0)}
function jumpTo(val){const n=parseInt(val);if(n>=1&&n<=DATA.length){currentIdx=n-1;render();window.scrollTo(0,0)}}

function updateStats(){
  let c=0,h=0,rem=0;
  Object.keys(annotations).forEach(id=>{
    METHOD_KEYS.forEach(m=>{
      if(annotations[id][m]) Object.values(annotations[id][m]).forEach(v=>{if(v==='no')c++;else if(v==='yes')h++;else rem++});
    });
  });
  document.getElementById('statCorrect').textContent=c;
  document.getElementById('statHall').textContent=h;
  document.getElementById('statRemain').textContent=rem;
  const total=c+h+rem;const done=c+h;
  document.getElementById('progFill').style.width=(total>0?(done/total*100):0)+'%';
  document.getElementById('progText').textContent=done+'/'+total+' done';
}

function downloadCSV(){
  let csv='image_id,method,caption,triple_idx,subject,relation,object,triple_text,hallucinated\n';
  DATA.forEach(d=>{
    METHOD_KEYS.forEach(m=>{
      const mdata=d.methods[m]||{caption:'',triples:[]};
      (mdata.triples||[]).forEach((t,i)=>{
        const a=(annotations[d.id]&&annotations[d.id][m])?annotations[d.id][m][i]||'':'';
        csv+='"'+d.id+'","'+m+'","'+mdata.caption.replace(/"/g,'""')+'",'+i+',"'+t.s+'","'+t.r+'","'+t.o+'","'+t.s+' '+t.r+' '+t.o+'","'+a+'"\n';
      });
    });
  });
  const blob=new Blob([csv],{type:'text/csv'});
  const a=document.createElement('a');a.href=URL.createObjectURL(blob);a.download='rchair_annotation_done.csv';a.click();
}

document.addEventListener('keydown',e=>{if(e.target.tagName==='INPUT')return;if(e.key==='ArrowLeft')go(-1);if(e.key==='ArrowRight')go(1)});
render();
"""

BODY_HTML = """
<div class="header"><h1>R-CHAIR Annotation - RelCheck</h1><div>
<div class="progress-bar"><div class="progress-fill" id="progFill"></div></div>
<div class="progress-text" id="progText">0/0</div></div></div>
<div class="container">
<div class="nav-bar">
<button class="btn-prev" id="btnPrev" onclick="go(-1)">Prev</button>
<div><span class="img-counter">Image <input class="jump-input" id="jumpInput" onchange="jumpTo(this.value)"> of <span id="totalCount"></span></span>
<div class="stats-bar"><span>Correct: <strong id="statCorrect">0</strong></span>
<span>Hallucinated: <strong id="statHall">0</strong></span>
<span>Remaining: <strong id="statRemain">0</strong></span></div></div>
<div style="display:flex;gap:8px">
<button class="btn-download" onclick="downloadCSV()">Download CSV</button>
<button class="btn-next" id="btnNext" onclick="go(1)">Next</button>
</div></div>
<div class="image-card"><img id="mainImg" src="">
<div class="card-body">
<div id="methodsContainer"></div>
</div></div>
<div class="nav-bar" style="margin-top:0">
<button class="btn-prev" onclick="go(-1)">Prev</button><span></span>
<button class="btn-next" onclick="go(1)">Next</button></div></div>
"""

# Assemble HTML
html_parts = []
html_parts.append('<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">')
html_parts.append('<title>R-CHAIR Annotation Tool</title>')
html_parts.append(f'<style>{CSS}</style></head><body>')
html_parts.append(BODY_HTML)
html_parts.append('<script>')
html_parts.append('const DATA=' + json.dumps(js_data) + ';')
html_parts.append('const IMG_B64={};')
for e in annotation_data:
    html_parts.append(f'IMG_B64["{e["id"]}"]="{e["b64"]}";')
html_parts.append(JS_LOGIC)
html_parts.append('</script></body></html>')

html = '\n'.join(html_parts)
with open(ANNOTATION_HTML, "w") as f:
    f.write(html)

n_triples = sum(len(t) for e in annotation_data for m in e["methods"].values() for t in [m["triples"]])
print(f"\nAnnotation tool saved: {ANNOTATION_HTML}")
print(f"   {len(annotation_data)} images, {n_triples} triples")
print("   Open in browser, click Correct/Hallucinated, download CSV when done")
print(f"   Upload CSV to: {ANNOTATION_DONE}")


# === PART C: Automated VLM R-CHAIR ===
print("\n--- Automated VLM R-CHAIR ---")

for method, caps in methods_rchair.items():
    vlm_key = f"vlm_{method}"
    if vlm_key in rchair_raw and len(rchair_raw[vlm_key]) >= len(rchair_sample):
        print(f"  VLM {method}: cached")
        continue
    if vlm_key not in rchair_raw:
        rchair_raw[vlm_key] = {}
    triples_key = f"triples_{method}"
    print(f"  VLM {method}: verifying...")
    for idx, img_id in enumerate(rchair_sample):
        if img_id in rchair_raw[vlm_key]:
            continue
        triples = rchair_raw.get(triples_key, {}).get(img_id, [])
        n_hall = 0
        verified = []
        for t in triples:
            s, r_rel, o = t.get("subject", ""), t.get("relation", ""), t.get("object", "")
            if not (s and r_rel and o):
                continue
            is_real = verify_triple_vlm(images[img_id], s, r_rel, o)
            verified.append({"s": s, "r": r_rel, "o": o, "verified": is_real})
            if not is_real:
                n_hall += 1
            time.sleep(0.2)
        rchair_raw[vlm_key][img_id] = {
            "n_triples": len(verified), "n_hallucinated": n_hall, "triples": verified,
        }
        if (idx + 1) % 10 == 0:
            print(f"    [{idx+1}/{len(rchair_sample)}]")
            save_checkpoint(rchair_raw, RCHAIR_PATH)
    save_checkpoint(rchair_raw, RCHAIR_PATH)


# === PART D: R-CHAIR Tables ===
print(f"\n{'='*65}")
print(f"TABLE 6a: R-CHAIR - Automated VLM ({len(rchair_sample)} images)")
print(f"{'='*65}")
print(f"{'Method':<22} {'R-CHAIR_s':>10} {'R-CHAIR_i':>10} {'N triples':>10}")
print("-" * 55)

vlm_results = {}
_pairs = [("llava_orig", "llava_corrected", "LLaVA-1.5"),
          ("qwen_orig",  "qwen_corrected",  "Qwen3-VL-8B")]
for orig_k, corr_k, cap_label in _pairs:
    for method, m_label in [(orig_k, f"{cap_label} orig"), (corr_k, f"{cap_label} corrected")]:
        if method not in methods_rchair:
            continue
        vlm_key = f"vlm_{method}"
        samp = [i for i in rchair_sample if i in methods_rchair[method]]
        data = rchair_raw.get(vlm_key, {})
        n_caps_hall = total_triples = total_hall = n_imgs = 0
        for img_id in samp:
            if img_id in data:
                d = data[img_id]
                n_imgs += 1
                if d["n_hallucinated"] > 0:
                    n_caps_hall += 1
                total_triples += d["n_triples"]
                total_hall    += d["n_hallucinated"]
        rchair_s = n_caps_hall / n_imgs if n_imgs else 0
        rchair_i = total_hall / total_triples if total_triples else 0
        vlm_results[method] = {"s": rchair_s, "i": rchair_i}
        delta_str = "—"
        if method == corr_k and orig_k in vlm_results:
            d_i = rchair_i - vlm_results[orig_k]["i"]
            delta_str = f"{d_i:+.1%}"
        print(f"{m_label:<26} {rchair_s:>10.1%} {rchair_i:>10.1%} {delta_str:>12} {total_triples:>10}")

# Manual R-CHAIR (if done)
if os.path.exists(ANNOTATION_DONE):
    print(f"\n{'='*65}")
    print(f"TABLE 6b: R-CHAIR - Human Annotation ({len(rchair_sample)} images)")
    print(f"{'='*65}")
    with open(ANNOTATION_DONE, "r") as f:
        manual_rows = list(csv.DictReader(f))
    annotated = [r for r in manual_rows
                 if r.get("hallucinated", "").strip().lower() in ("yes", "no")]
    print(f"  {len(annotated)} triples annotated out of {len(manual_rows)}")
    print(f"\n{'Method':<22} {'R-CHAIR_s':>10} {'R-CHAIR_i':>10} {'N triples':>10}")
    print("-" * 55)
    manual_results = {}
    _seen_methods = list(dict.fromkeys(r["method"] for r in annotated))
    _manual_pairs = [("llava_orig", "llava_corrected", "LLaVA-1.5"),
                     ("qwen_orig",  "qwen_corrected",  "Qwen3-VL-8B")]
    for orig_k, corr_k, cap_label in _manual_pairs:
        for method, m_label in [(orig_k, f"{cap_label} orig"), (corr_k, f"{cap_label} corrected")]:
            if method not in _seen_methods:
                continue
            method_rows = [r for r in annotated if r["method"] == method]
            by_image = {}
            for r in method_rows:
                img_id = r["image_id"]
                if img_id not in by_image:
                    by_image[img_id] = {"n_triples": 0, "n_hallucinated": 0}
                by_image[img_id]["n_triples"] += 1
                if r["hallucinated"].strip().lower() == "yes":
                    by_image[img_id]["n_hallucinated"] += 1
            n_caps_hall = sum(1 for d in by_image.values() if d["n_hallucinated"] > 0)
            total_triples = sum(d["n_triples"] for d in by_image.values())
            total_hall = sum(d["n_hallucinated"] for d in by_image.values())
            n_imgs = len(by_image)
            rchair_s = n_caps_hall / n_imgs if n_imgs else 0
            rchair_i = total_hall / total_triples if total_triples else 0
            manual_results[method] = {"s": rchair_s, "i": rchair_i}
            delta_str = "—"
            if method == corr_k and orig_k in manual_results:
                d_i = rchair_i - manual_results[orig_k]["i"]
                delta_str = f"{d_i:+.1%}"
            print(f"{m_label:<26} {rchair_s:>10.1%} {rchair_i:>10.1%} {delta_str:>12} {total_triples:>10}")
    # Agreement
    agree, disagree = 0, 0
    for r in annotated:
        img_id, method = r["image_id"], r["method"]
        tidx = int(r["triple_idx"])
        human_hall = r["hallucinated"].strip().lower() == "yes"
        vlm_key = f"vlm_{method}"
        vlm_data = rchair_raw.get(vlm_key, {}).get(img_id, {}).get("triples", [])
        if tidx < len(vlm_data):
            vlm_hall = not vlm_data[tidx]["verified"]
            if human_hall == vlm_hall:
                agree += 1
            else:
                disagree += 1
    total = agree + disagree
    if total > 0:
        pct = agree / total
        kappa = (pct - 0.5) / (1 - 0.5) if pct > 0.5 else 0
        print(f"\n--- Human vs VLM Agreement ---")
        print(f"  Agreement: {agree}/{total} ({pct:.1%})")
        print(f"  Cohen kappa (approx): {kappa:.3f}")
else:
    print(f"\n  Manual annotation not yet done.")
    print(f"  1. Open: {ANNOTATION_HTML}")
    print(f"  2. Annotate all triples (arrow keys to navigate)")
    print(f"  3. Download CSV -> upload to: {ANNOTATION_DONE}")
    print(f"  4. Re-run this cell for Table 6b")



In [ ]:
# ============================================================
# Cell 14 — Multi-Captioner: InstructBLIP (Table 11)
# ============================================================
# Run RelCheck on InstructBLIP captions to prove model-agnostic improvement.
# Uses the SAME KB (GDINO + Maverick) — only the captioner changes.
# Re-uses enrichment + evaluation logic from earlier cells.

IBLEND_PATH = f"{SAVE_DIR}/instructblip_results.json"

# ── Load InstructBLIP ──
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration

print("Loading InstructBLIP...")
ib_processor = InstructBlipProcessor.from_pretrained("Salesforce/instructblip-flan-t5-xl")
ib_model = InstructBlipForConditionalGeneration.from_pretrained(
    "Salesforce/instructblip-flan-t5-xl", torch_dtype=torch.float16, device_map="auto"
)
print("InstructBLIP loaded.")


def caption_with_instructblip(image):
    prompt = "Describe this image in detail. Include all objects, their spatial positions relative to each other, any actions or interactions taking place, and notable attributes like colors and sizes."
    inputs = ib_processor(images=image, text=prompt, return_tensors="pt").to(
        ib_model.device, torch.float16)
    with torch.no_grad():
        out = ib_model.generate(**inputs, max_new_tokens=80)
    return ib_processor.decode(out[0], skip_special_tokens=True).strip()


# ── Load or compute InstructBLIP results ──
ib_raw = load_checkpoint(IBLEND_PATH) or {}

# Use first 100 images (or all if < 100)
ib_sample = list(images.keys())[:min(100, len(images))]

# Stage 1: InstructBLIP captions
if "captions" not in ib_raw:
    ib_raw["captions"] = {}

ib_todo = [i for i in ib_sample if i not in ib_raw["captions"]]
if ib_todo:
    print(f"InstructBLIP captioning {len(ib_todo)} images...")
    for idx, img_id in enumerate(ib_todo):
        ib_raw["captions"][img_id] = caption_with_instructblip(images[img_id])
        if (idx + 1) % 25 == 0:
            print(f"  [{idx+1}/{len(ib_todo)}]")
            save_checkpoint(ib_raw, IBLEND_PATH)
    save_checkpoint(ib_raw, IBLEND_PATH)
    print("InstructBLIP captions done.")
else:
    print(f"Loaded {len(ib_raw['captions'])} InstructBLIP captions from cache.")

# Stage 2: Enrich InstructBLIP captions (reuse same KB)
if "enriched" not in ib_raw:
    ib_raw["enriched"] = {}

ib_enrich_todo = [i for i in ib_sample if i not in ib_raw.get("enriched", {})]
if ib_enrich_todo:
    print(f"Enriching {len(ib_enrich_todo)} InstructBLIP captions...")
    for idx, img_id in enumerate(ib_enrich_todo):
        ib_cap = ib_raw["captions"][img_id]
        kb = knowledge_bases[img_id]  # reuse same KB
        result = enrich_caption_v3(img_id, ib_cap, kb, pil_image=images.get(img_id))
        ib_raw["enriched"][img_id] = result
        if (idx + 1) % 25 == 0:
            print(f"  [{idx+1}/{len(ib_enrich_todo)}]")
            save_checkpoint(ib_raw, IBLEND_PATH)
    save_checkpoint(ib_raw, IBLEND_PATH)
    print("InstructBLIP enrichment done.")
else:
    print(f"Loaded {len(ib_raw['enriched'])} enriched InstructBLIP captions.")

# Stage 3: R-POPE eval for InstructBLIP (original + enriched)
# Stage 2b: B3 (blind) and KB dump ablations for InstructBLIP
if "b3" not in ib_raw:
    ib_raw["b3"] = {}
ib_b3_todo = [i for i in ib_sample if i not in ib_raw["b3"]]
if ib_b3_todo:
    print(f"InstructBLIP B3 (blind correction) for {len(ib_b3_todo)} images...")
    for idx, img_id in enumerate(ib_b3_todo):
        ib_cap = ib_raw["captions"][img_id]
        raw = llm_call([{"role": "user", "content": B3_PROMPT.format(caption=ib_cap)}], max_tokens=300)
        ib_raw["b3"][img_id] = raw.strip().strip('"').strip("'") if raw and len(raw.strip()) >= 10 else ib_cap
        if (idx + 1) % 50 == 0:
            save_checkpoint(ib_raw, IBLEND_PATH)
        time.sleep(0.25)
    save_checkpoint(ib_raw, IBLEND_PATH)
    print("InstructBLIP B3 done.")

if "kb_dump" not in ib_raw:
    ib_raw["kb_dump"] = {}
ib_kd_todo = [i for i in ib_sample if i not in ib_raw["kb_dump"]]
if ib_kd_todo:
    print(f"InstructBLIP KB dump for {len(ib_kd_todo)} images...")
    for idx, img_id in enumerate(ib_kd_todo):
        ib_cap = ib_raw["captions"][img_id]
        kb = knowledge_bases[img_id]
        prompt = KB_DUMP_PROMPT.format(
            caption=ib_cap,
            obj_facts="\n".join(f"- {f}" for f in kb["hard_facts"]) or "- None",
            spatial_facts="\n".join(f"- {f}" for f in kb["spatial_facts"]) or "- None",
            visual_desc=kb["visual_description"][:300] or "- None",
        )
        raw = llm_call([{"role": "user", "content": prompt}], max_tokens=300)
        ib_raw["kb_dump"][img_id] = raw.strip().strip('"').strip("'") if raw and len(raw.strip()) >= 10 else ib_cap
        if (idx + 1) % 50 == 0:
            save_checkpoint(ib_raw, IBLEND_PATH)
        time.sleep(0.25)
    save_checkpoint(ib_raw, IBLEND_PATH)
    print("InstructBLIP KB dump done.")

if "rpope" not in ib_raw:
    ib_raw["rpope"] = {}

print("R-POPE evaluation for InstructBLIP...")
ib_b1_correct, ib_rc_correct, ib_total = 0, 0, 0

for idx, img_id in enumerate(ib_sample):
    if img_id not in img_to_questions:
        continue
    questions = img_to_questions[img_id]

    for entry in questions[:10]:
        q = entry["question"]
        gt = entry["answer"]
        cache_key = f"{img_id}|||{q}"

        if cache_key not in ib_raw["rpope"]:
            ib_orig_cap = ib_raw["captions"].get(img_id, "")
            ib_enr_cap = ib_raw["enriched"].get(img_id, {}).get("corrected", ib_orig_cap)
            ib_b3_cap = ib_raw.get("b3", {}).get(img_id, ib_orig_cap)
            ib_kd_cap = ib_raw.get("kb_dump", {}).get(img_id, ib_orig_cap)
            orig_ans = rpope_judge(ib_orig_cap, q)
            enr_ans = rpope_judge(ib_enr_cap, q)
            b3_ans = rpope_judge(ib_b3_cap, q)
            kd_ans = rpope_judge(ib_kd_cap, q)
            ib_raw["rpope"][cache_key] = {
                "orig": orig_ans, "enr": enr_ans,
                "b3": b3_ans, "kb_dump": kd_ans, "gt": gt
            }
            time.sleep(0.1)

        r = ib_raw["rpope"][cache_key]
        if r["orig"] == gt:
            ib_b1_correct += 1
        if r["enr"] == gt:
            ib_rc_correct += 1
        ib_total += 1

    if (idx + 1) % 25 == 0:
        save_checkpoint(ib_raw, IBLEND_PATH)

save_checkpoint(ib_raw, IBLEND_PATH)

# ── Table 11: Multi-Captioner ──
print(f"\n{'='*75}")
print(f"TABLE 11: Multi-Captioner Generalizability ({len(ib_sample)} images)")
print(f"{'='*75}")

# LLaVA / Qwen results on same subset (fair comparison)
def subset_acc(method, img_ids):
    pairs = [
        (entry["pred"], entry["gt"])
        for img_id in img_ids
        for key, entry in rpope_raw.get(img_id, {}).items()
        if "|||" in key and key.split("|||")[0] == method and entry.get("pred") != "unknown"
    ]
    return compute_metrics(pairs)["accuracy"] if pairs else 0, len(pairs)

ib_b1_acc = ib_b1_correct / ib_total if ib_total else 0
ib_rc_acc = ib_rc_correct / ib_total if ib_total else 0

# InstructBLIP B3 accuracy
ib_b3_correct = 0
for img_id in ib_sample:
    if img_id not in img_to_questions:
        continue
    for entry in img_to_questions[img_id][:10]:
        q = entry["question"]
        cache_key = f"{img_id}|||{q}"
        r = ib_raw["rpope"].get(cache_key, {})
        if r.get("b3") == r.get("gt"):
            ib_b3_correct += 1
ib_b3_acc = ib_b3_correct / ib_total if ib_total else 0

print(f"{'Captioner':<22} {'Original':>10} {'Blind(B3)':>10} {'RelCheck':>10} {'Delta':>8} {'N':>6}")
print("-" * 68)
# LLaVA
la_orig, la_n  = subset_acc("llava_orig", ib_sample)
la_rc, _       = subset_acc("llava_relcheck", ib_sample)
print(f"{'LLaVA-1.5':<22} {la_orig:>10.1%} {'—':>10} {la_rc:>10.1%} {la_rc-la_orig:>+8.1%} {la_n:>6}")
# Qwen
qw_orig, qw_n  = subset_acc("qwen_orig", ib_sample)
qw_rc, _       = subset_acc("qwen_relcheck", ib_sample)
print(f"{'Qwen3-VL-8B':<22} {qw_orig:>10.1%} {'—':>10} {qw_rc:>10.1%} {qw_rc-qw_orig:>+8.1%} {qw_n:>6}")
# InstructBLIP
print(f"{'InstructBLIP':<22} {ib_b1_acc:>10.1%} {ib_b3_acc:>10.1%} {ib_rc_acc:>10.1%} {ib_rc_acc-ib_b1_acc:>+8.1%} {ib_total:>6}")

n_ib_mod = sum(1 for r in ib_raw.get("enriched", {}).values() if r.get("status") == "modified")
print(f"\n  InstructBLIP images modified: {n_ib_mod}/{len(ib_sample)}")

# Cleanup: free InstructBLIP from GPU to reclaim VRAM
del ib_model, ib_processor
torch.cuda.empty_cache()
print("InstructBLIP model unloaded to free VRAM.")


In [ ]:
# ============================================================
# Cell 15 — Architecture Diagram (Figure 1)
# ============================================================
# Paper-ready pipeline diagram showing RelCheck's model-agnostic design.

fig, ax = plt.subplots(figsize=(15, 7))
ax.set_xlim(0, 15)
ax.set_ylim(0, 7)
ax.axis("off")

# ── Styling ──
def draw_box(x, y, w, h, text, color, fontsize=9, bold=False, alpha=0.9, edge="#333"):
    rect = plt.Rectangle((x, y), w, h, facecolor=color, edgecolor=edge,
                          linewidth=1.5, zorder=2, alpha=alpha, joinstyle="round")
    ax.add_patch(rect)
    weight = "bold" if bold else "normal"
    ax.text(x + w/2, y + h/2, text, ha="center", va="center",
            fontsize=fontsize, fontweight=weight, zorder=3)

def draw_arrow(x1, y1, x2, y2, text="", color="#555", lw=1.5):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="-|>", color=color, lw=lw))
    if text:
        mx, my = (x1+x2)/2, (y1+y2)/2 + 0.18
        ax.text(mx, my, text, ha="center", va="bottom", fontsize=7, color="#555",
                style="italic")

# ── Title ──
ax.text(7.5, 6.6, "RelCheck: Training-Free Relational Hallucination Correction",
        ha="center", va="center", fontsize=14, fontweight="bold", color="#1a237e")

# ═══════════════════════════════════════════════════
# LEFT: Input
# ═══════════════════════════════════════════════════
draw_box(0.2, 3.0, 1.4, 1.3, "Input\nImage", "#e0e0e0", fontsize=11, bold=True)

# ═══════════════════════════════════════════════════
# Stage 1: Any MLLM captioner (model-agnostic)
# ═══════════════════════════════════════════════════
draw_box(2.2, 4.5, 2.2, 1.6, "Any MLLM\nCaptioner", "#bbdefb", fontsize=10, bold=True, edge="#1565c0")
# Show supported models as sub-labels
ax.text(3.3, 4.25, "BLIP-2 / LLaVA-1.5 /\nInstructBLIP / Qwen3-VL-8B",
        ha="center", fontsize=7, color="#1565c0", style="italic")

draw_arrow(1.6, 3.8, 2.2, 5.3, "image")
draw_arrow(4.4, 5.3, 5.1, 5.3, "caption")

# ═══════════════════════════════════════════════════
# Stage 2: Visual KB Construction (parallel paths)
# ═══════════════════════════════════════════════════
# KB header
ax.text(6.45, 6.2, "Visual Knowledge Base Construction", fontsize=10,
        fontweight="bold", ha="center", color="#1b5e20")

# HARD layer: GroundingDINO
draw_box(5.1, 5.0, 2.7, 1.1, "GroundingDINO\nObject Detection +\nBbox Positions", "#fff9c4",
         fontsize=8, edge="#f57f17")

# GEOM layer: Spatial geometry
draw_box(5.1, 3.5, 2.7, 1.1, "Spatial Geometry\nDeterministic bbox\nrelation rules", "#ffe0b2",
         fontsize=8, edge="#e65100")

# SOFT layer: VLM description
draw_box(5.1, 2.0, 2.7, 1.1, "Llama-4-Maverick\nVLM Visual\nDescription", "#ffccbc",
         fontsize=8, edge="#bf360c")

# Arrows into KB layers
draw_arrow(1.6, 3.5, 5.1, 5.5, "image")
draw_arrow(1.6, 3.2, 5.1, 2.5, "image")

# Internal KB flow
draw_arrow(6.45, 5.0, 6.45, 4.6, "objects +\nbboxes", color="#888")

# KB aggregate box
draw_box(5.1, 0.6, 2.7, 1.0, "Visual KB\n(HARD + GEOM + SOFT)", "#c8e6c9",
         fontsize=9, bold=True, edge="#2e7d32")

# Arrows down to KB aggregate
draw_arrow(6.0, 2.0, 5.8, 1.6, color="#888")
draw_arrow(6.45, 3.5, 6.45, 1.6, color="#888")
draw_arrow(6.9, 5.0, 7.1, 1.6, color="#888")

# ═══════════════════════════════════════════════════
# Stage 3: Structured Analysis + Enriched Rewrite
# ═══════════════════════════════════════════════════
draw_box(8.5, 3.3, 2.5, 1.8, "Llama-3.3-70B\n\n1. Find errors\n   (KB contradictions)\n2. Find omissions\n3. Rewrite caption",
         "#e1bee7", fontsize=8, edge="#6a1b9a")

draw_arrow(7.8, 1.1, 8.5, 3.5, "KB evidence")
draw_arrow(4.4, 5.0, 8.5, 4.8, "caption")

# ═══════════════════════════════════════════════════
# Stage 4: Verification Gate
# ═══════════════════════════════════════════════════
draw_box(11.6, 3.5, 2.0, 1.4, "LLM Verification\nGate\n\nKB-grounded\nfaithfulness", "#ffcdd2",
         fontsize=8, edge="#c62828")

draw_arrow(11.0, 4.2, 11.6, 4.2, "rewrite")

# ═══════════════════════════════════════════════════
# Output
# ═══════════════════════════════════════════════════
draw_box(11.8, 1.5, 1.6, 1.2, "Corrected\nCaption", "#a5d6a7", fontsize=10, bold=True, edge="#2e7d32")

# Pass/fail arrows
draw_arrow(12.6, 3.5, 12.6, 2.7, "PASS", color="#2e7d32", lw=2)

# Fail arrow — keeps original
ax.annotate("", xy=(9.75, 3.3), xytext=(12.1, 3.5),
            arrowprops=dict(arrowstyle="-|>", color="#c62828", lw=1.2,
                           connectionstyle="arc3,rad=0.5"))
ax.text(10.7, 2.7, "FAIL: keep\noriginal", fontsize=7, color="#c62828", ha="center")

# ═══════════════════════════════════════════════════
# Legend
# ═══════════════════════════════════════════════════
legend_y = 0.15
ax.text(0.3, legend_y, "Key:", fontsize=8, fontweight="bold")
for i, (color, label) in enumerate([
    ("#bbdefb", "Model-agnostic (any MLLM)"),
    ("#fff9c4", "Deterministic detection"),
    ("#e1bee7", "LLM reasoning"),
    ("#ffcdd2", "Verification gate"),
    ("#c8e6c9", "Novel: 3-layer Visual KB"),
]):
    x = 1.5 + i * 2.7
    rect = plt.Rectangle((x, legend_y - 0.05), 0.25, 0.25,
                          facecolor=color, edgecolor="#666", linewidth=0.8)
    ax.add_patch(rect)
    ax.text(x + 0.35, legend_y + 0.08, label, fontsize=7, va="center")

plt.tight_layout()
fig1_path = f"{FIGS_DIR}/fig1_architecture.png"
plt.savefig(fig1_path, dpi=300, bbox_inches="tight", facecolor="white")
plt.savefig(f"{FIGS_DIR}/fig1_architecture.pdf", bbox_inches="tight", facecolor="white")
plt.show()
print(f"Figure 1 saved: {fig1_path} (PNG + PDF)")
